# Bay of Bengal Oxygen Depletion Research

**Author:** Md. Mahfujur Rahman
**License:** MIT (see `LICENSE`)

This notebook is the complete, consolidated code for my Bay of Bengal dissolved-oxygen /
oxygen-minimum-zone (OMZ) research: building the 2005-2022 ensemble oxygen product from two
independent gridded sources, the 14-analysis deoxygenation study built on top of it, the
yearly-map and oxygen-zone-volume outputs, and the Argo-driven driver-attribution modelling. It
brings together four notebooks and one local Python package that were originally developed and
run separately, into one place.

## How this notebook is organised

1. **Part 0 — Project setup**: writes out the `bob_local_pipeline` package (config, boundary
   clipping, regridding, ensembling, the pipeline driver) so the rest of this notebook can import
   it directly.
2. **Part A — Grid + Ensemble pipeline** (sections 1-13): builds the 216-month ensemble oxygen
   product from the G4D-DOC and GEOXYGEN source grids, then the monthly/yearly climatologies,
   kriging interpolation, and the layer-average spatial maps.
3. **Part B — Yearly maps & oxygen-zone volumes** (sections 14-17): continues the numbering from
   Part A — yearly kriged maps, oxic/hypoxic/suboxic/anoxic water-volume time series, and seasonal
   vertical oxygen profiles.
4. **Part C — 14-analysis deoxygenation study**: the main scientific analysis, run against the
   ensemble product built in Part A — long-term trends, Hovmöller diagrams, EOF and tensor
   decomposition, wavelet/FFT analysis, Mann-Kendall trend testing, change-point detection, OMZ
   volume/oxycline tracking, spatial autocorrelation, ENSO/IOD composites, and cross-product QC.
   Keeps its own internal section numbering (0-14), since it was developed as a self-contained
   study.
5. **Part D — Driver attribution**: identifies the dominant physical/climate drivers of dissolved
   oxygen using the Argo-derived driver dataset, across seven independent importance measures
   (CatBoost/LightGBM/XGBoost + TreeSHAP, Random Forest + permutation importance, GAM).

## Running this notebook

This notebook needs the raw G4D-DOC and GEOXYGEN monthly files (Parts A/B) and the Argo driver
parquet table (Part D) to actually execute end to end — see `data/README.md` for where those come
from; they're too large to check into this repository. Every output these sections produce is
already saved under `results/`, with `results/README.md` mapping each figure/table back to the
section that made it, so the analysis is fully reviewable without re-running anything.

`docs/BoB_Oxygen_Analysis_Manual.md` walks through every one of the 14 core analyses in plain
language — what it shows, why it matters, and which parameters are safe to change — for anyone
who wants more context than the in-notebook markdown gives.


## Part 0 — Project setup

The cells below write out the `bob_local_pipeline` package used throughout Parts A and B. If
you already have `bob_local_pipeline/` on disk next to this notebook (it ships in this
repository), running these cells just re-writes the same files — safe either way.

In [ ]:
import os
os.makedirs("bob_local_pipeline", exist_ok=True)
print("bob_local_pipeline/ ready")

**`bob_local_pipeline/__init__.py`** — Package marker + version string.

In [ ]:
%%writefile bob_local_pipeline/__init__.py
"""
bob_local_pipeline — Bay of Bengal G4D-DOC + GEOXYGEN grid/ensemble pipeline.

Local (VS Code / plain Python) companion to the earlier Colab-based project.
See README.md in this folder for how to run it.
"""
__version__ = "0.1.0"

from . import config  # noqa: F401
from . import boundary  # noqa: F401
from . import standardize  # noqa: F401
from . import grid  # noqa: F401
from . import ensemble  # noqa: F401


**`bob_local_pipeline/config.py`** — All paths, thresholds, and tunable parameters for the grid/ensemble pipeline.

In [ ]:
%%writefile bob_local_pipeline/config.py
"""
config.py — all the settings for the Bay of Bengal grid/ensemble pipeline.

Edit the paths under PROJECT_ROOT if your folder layout differs; everything
else downstream (boundary.py, standardize.py, grid.py, ensemble.py,
run_pipeline.py) imports its settings from here, so this is the one file
you should need to touch to point the pipeline at your own data.
"""
from __future__ import annotations

from pathlib import Path

# ---------------------------------------------------------------------------
# Paths — matches the C:\All_data layout described when this was built.
# All raw .nc files (both products) are expected directly in RAW_DATA_DIR;
# the pipeline tells them apart by filename prefix (see FILENAME PATTERNS).
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path(r"C:\All_data")
RAW_DATA_DIR = PROJECT_ROOT
BOUNDARY_SHAPEFILE = PROJECT_ROOT / "Bay_of_Bengal_shapefile" / "Bay_of_Bengal.shp"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
GRIDDED_G4D_DOC_DIR = OUTPUT_DIR / "gridded_G4D_DOC"
GRIDDED_GEOXYGEN_DIR = OUTPUT_DIR / "gridded_GEOXYGEN"
ENSEMBLE_DIR = OUTPUT_DIR / "ensemble"
LOG_DIR = OUTPUT_DIR / "logs"

# ---------------------------------------------------------------------------
# Filename patterns — how run_pipeline.py finds each month's pair of files
# and parses the (year, month) out of the filename. G4D-DOC encodes
# year/month directly; GEOXYGEN encodes a full date (always the 15th).
# ---------------------------------------------------------------------------
G4D_DOC_FILENAME_TEMPLATE = "G4D_DOC_{year:04d}_{month:02d}.nc"
GEOXYGEN_FILENAME_TEMPLATE = "GLOBAL_DO_{year:04d}{month:02d}15_0p5deg_v1.nc"

# ---------------------------------------------------------------------------
# Source variable/dimension names, as found by directly inspecting one file
# from each product (2026-08-19). If a future data drop uses different
# names, update these two dicts rather than the processing code.
# ---------------------------------------------------------------------------
G4D_DOC_SCHEMA = dict(
    variable="DO",
    dim_rename={},  # already lat/lon/depth/time — no renaming needed
    units="umol/kg",
)
GEOXYGEN_SCHEMA = dict(
    variable="OXY",
    dim_rename={"latitude": "lat", "longitude": "lon"},
    units="umol/kg",
)

OUTPUT_VARIABLE_NAME = "o2_umol_kg"
OUTPUT_UNITS = "umol kg-1"

# ---------------------------------------------------------------------------
# Study area and target grid — bounds taken directly from
# Bay_of_Bengal.shp's total_bounds (verified 2026-08-19), 1x1 deg cells
# centered at the .5 degree mark (matches G4D-DOC's native grid exactly,
# so no interpolation error is introduced for that product).
# ---------------------------------------------------------------------------
STUDY_AREA_BBOX = dict(lon_min=78.90, lon_max=95.05, lat_min=5.73, lat_max=24.38)
GRID_RESOLUTION_DEG = 1.0
GRID_CRS = "EPSG:4326"

# A small margin (degrees) used only when pre-filtering GEOXYGEN's global
# grid before interpolation, so the interpolator always has real
# neighboring data on every side of the study area (avoids edge NaNs).
BBOX_MARGIN_DEG = 2.0

# ---------------------------------------------------------------------------
# Vertical (depth) handling.
# Per project decision (2026-08-19): use G4D-DOC's own 26 native depth
# levels (10-1995 m) as the single shared target — G4D-DOC needs no
# vertical interpolation at all, and GEOXYGEN (187 levels, 1-5500 m) is
# interpolated onto these same 26 levels. Both products have real data
# everywhere in this range, so no extrapolation is needed. GEOXYGEN levels
# beyond 1995 m are simply dropped (not used).
# These exact values are read from a real G4D-DOC file at build time; if
# G4D-DOC's depth axis ever changes, STANDARD_DEPTHS below should be
# regenerated the same way (see README "Changing the depth levels").
# ---------------------------------------------------------------------------
STANDARD_DEPTHS_M = [
    10.0, 20.0, 30.0, 40.0, 50.0, 75.0, 100.0, 125.0, 150.0, 200.0, 250.0,
    300.0, 400.0, 500.0, 600.0, 700.0, 800.0, 900.0, 1000.0, 1100.0,
    1200.0, 1300.0, 1400.0, 1500.0, 1750.0, 1995.0,
]

# ---------------------------------------------------------------------------
# Ensemble method.
# Per project decision (2026-08-19): SIMPLE AVERAGE of the two gridded
# products, not reliability weighting — reliability weighting needs
# independent validation data (GLODAPv3/BGC-Argo) to score each product,
# which this pipeline deliberately does not use. If validation data becomes
# available later, ensemble.py has a second function
# (reliability_weighted_mean) ready to use instead — see its docstring.
# ---------------------------------------------------------------------------
ENSEMBLE_METHOD = "simple_mean"

# ---------------------------------------------------------------------------
# Analysis period — inferred from the 216 files actually present
# (2005-01 through 2022-12). Change if your file set covers a different
# range; run_pipeline.py will only process months where BOTH files exist.
# ---------------------------------------------------------------------------
START_YEAR, START_MONTH = 2005, 1
END_YEAR, END_MONTH = 2022, 12


**`bob_local_pipeline/boundary.py`** — Loads the Bay of Bengal boundary polygon and builds the target-grid mask.

In [ ]:
%%writefile bob_local_pipeline/boundary.py
"""
boundary.py — load the Bay of Bengal shapefile and build a reusable
polygon + grid-mask, so every month uses the exact same official study-area
polygon (not just its bounding box).
"""
from __future__ import annotations

import geopandas as gpd
import numpy as np
import xarray as xr
from shapely.vectorized import contains

from . import config


def load_boundary_polygon():
    """Load Bay_of_Bengal.shp and return a single (possibly multi-part)
    shapely geometry in EPSG:4326, verifying the CRS along the way."""
    gdf = gpd.read_file(config.BOUNDARY_SHAPEFILE)
    if gdf.crs is None:
        raise ValueError(
            f"{config.BOUNDARY_SHAPEFILE} has no CRS defined — cannot safely "
            "clip against it. Check the .prj sidecar file is present."
        )
    if gdf.crs.to_string().upper() != config.GRID_CRS.upper():
        gdf = gdf.to_crs(config.GRID_CRS)
    if not gdf.geometry.is_valid.all():
        gdf["geometry"] = gdf.geometry.buffer(0)  # standard fix for minor topology issues
    bounds = gdf.total_bounds
    print(f"[boundary] loaded {config.BOUNDARY_SHAPEFILE.name}: "
          f"{len(gdf)} feature(s), bounds={bounds.round(3).tolist()}, CRS={gdf.crs}")
    return gdf.geometry.union_all()


def build_target_grid():
    """Return (target_lat, target_lon): the 1x1 deg grid-cell-center
    coordinate arrays covering the study bbox, aligned to the .5-degree
    convention (matches G4D-DOC's native grid exactly)."""
    lat = np.arange(
        np.floor(config.STUDY_AREA_BBOX["lat_min"]) + 0.5,
        np.ceil(config.STUDY_AREA_BBOX["lat_max"]) + 0.5,
        config.GRID_RESOLUTION_DEG,
    )
    lon = np.arange(
        np.floor(config.STUDY_AREA_BBOX["lon_min"]) + 0.5,
        np.ceil(config.STUDY_AREA_BBOX["lon_max"]) + 0.5,
        config.GRID_RESOLUTION_DEG,
    )
    return lat, lon


def build_grid_mask(polygon, target_lat, target_lon) -> xr.DataArray:
    """Boolean (lat, lon) DataArray: True where a grid cell's CENTER falls
    inside the official boundary polygon. This is the mask every gridded
    product and the ensemble get multiplied through, so 'clip to the Bay of
    Bengal boundary' means the same exact set of cells everywhere."""
    lon2d, lat2d = np.meshgrid(target_lon, target_lat)
    mask_values = contains(polygon, lon2d, lat2d)
    mask = xr.DataArray(
        mask_values, dims=("lat", "lon"),
        coords={"lat": target_lat, "lon": target_lon},
        name="bay_of_bengal_mask",
    )
    n_in = int(mask.sum())
    print(f"[boundary] grid mask: {n_in}/{mask.size} of the {len(target_lat)}x{len(target_lon)} "
          "target cells have centers inside the Bay of Bengal polygon.")
    if n_in == 0:
        raise ValueError(
            "Boundary mask is empty — no target grid cell centers fall inside "
            "the polygon. Check STUDY_AREA_BBOX / the shapefile's CRS in config.py."
        )
    return mask


**`bob_local_pipeline/standardize.py`** — Per-file loaders for the G4D-DOC and GEOXYGEN source products.

In [ ]:
%%writefile bob_local_pipeline/standardize.py
"""
standardize.py — load one raw monthly file (either product) and reduce it
to a single, common-schema (depth, lat, lon) DataArray of dissolved oxygen
in umol/kg, with the source product's original dimension/variable names
already normalized away. This is the "standardize coordinates, units,
metadata" step; depth-level standardization and horizontal regridding
happen in grid.py (they need the target grid/depths from boundary.py /
config.py, so are kept separate from this simpler per-file step).
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import xarray as xr

from . import config


class MissingFileError(FileNotFoundError):
    pass


class SchemaMismatchError(ValueError):
    """Raised when a raw file doesn't have the variable/dims we expect —
    surfaces a clear message instead of a confusing downstream KeyError."""


def _check_units(da: xr.DataArray, expected_units_substring: str, source_name: str, path: Path):
    units = str(da.attrs.get("units", "")).lower().replace("-1", "").replace(" ", "")
    expected = expected_units_substring.lower().replace(" ", "").replace("/", "")
    if expected not in units and units not in expected:
        print(f"[standardize] WARNING: {source_name} file {path.name} has units "
              f"'{da.attrs.get('units')}', expected something matching "
              f"'{expected_units_substring}'. Proceeding, but verify no unit "
              "conversion is actually needed.")


def load_g4d_doc_month(path: Path) -> xr.DataArray:
    """Load one G4D_DOC_YYYY_MM.nc file -> (depth, lat, lon) DataArray,
    umol/kg, dims already named lat/lon/depth (no renaming needed)."""
    if not path.exists():
        raise MissingFileError(f"G4D-DOC file not found: {path}")
    ds = xr.open_dataset(path)
    schema = config.G4D_DOC_SCHEMA
    if schema["variable"] not in ds.data_vars:
        raise SchemaMismatchError(
            f"{path.name}: expected variable '{schema['variable']}', found "
            f"{list(ds.data_vars)}. Update config.G4D_DOC_SCHEMA if the file "
            "format has changed."
        )
    for dim in ("lat", "lon", "depth"):
        if dim not in ds.dims:
            raise SchemaMismatchError(
                f"{path.name}: expected dimension '{dim}', found {list(ds.dims)}."
            )
    da = ds[schema["variable"]]
    _check_units(da, schema["units"], "G4D-DOC", path)
    if "time" in da.dims:
        da = da.isel(time=0, drop=True)
    da = da.transpose("depth", "lat", "lon")
    da.attrs["source_product"] = "G4D-DOC"
    da.attrs["source_file"] = path.name
    return da


def load_geoxygen_month(path: Path) -> xr.DataArray:
    """Load one GLOBAL_DO_YYYYMMDD_0p5deg_v1.nc file -> (depth, lat, lon)
    DataArray, umol/kg, dims renamed latitude/longitude -> lat/lon."""
    if not path.exists():
        raise MissingFileError(f"GEOXYGEN file not found: {path}")
    ds = xr.open_dataset(path)
    schema = config.GEOXYGEN_SCHEMA
    if schema["variable"] not in ds.data_vars:
        raise SchemaMismatchError(
            f"{path.name}: expected variable '{schema['variable']}', found "
            f"{list(ds.data_vars)}. Update config.GEOXYGEN_SCHEMA if the file "
            "format has changed."
        )
    ds = ds.rename({k: v for k, v in schema["dim_rename"].items() if k in ds.dims})
    for dim in ("lat", "lon", "depth"):
        if dim not in ds.dims:
            raise SchemaMismatchError(
                f"{path.name}: expected dimension '{dim}' (after renaming), "
                f"found {list(ds.dims)}."
            )
    da = ds[schema["variable"]]
    _check_units(da, schema["units"], "GEOXYGEN", path)
    if "time" in da.dims:
        da = da.isel(time=0, drop=True)
    da = da.transpose("depth", "lat", "lon")
    da.attrs["source_product"] = "GEOXYGEN"
    da.attrs["source_file"] = path.name
    return da


def parse_filenames_for_month(year: int, month: int) -> tuple[Path, Path]:
    """Return (g4d_doc_path, geoxygen_path) for a given (year, month),
    using the filename templates in config.py."""
    g4d_path = config.RAW_DATA_DIR / config.G4D_DOC_FILENAME_TEMPLATE.format(year=year, month=month)
    geox_path = config.RAW_DATA_DIR / config.GEOXYGEN_FILENAME_TEMPLATE.format(year=year, month=month)
    return g4d_path, geox_path


def iter_available_months():
    """Yield (year, month, g4d_path, geoxygen_path) for every month in
    config.START_YEAR/MONTH..END_YEAR/MONTH where BOTH files exist on disk.
    Months missing either file are reported and skipped, not silently
    dropped."""
    y, m = config.START_YEAR, config.START_MONTH
    while (y, m) <= (config.END_YEAR, config.END_MONTH):
        g4d_path, geox_path = parse_filenames_for_month(y, m)
        g4d_ok, geox_ok = g4d_path.exists(), geox_path.exists()
        if g4d_ok and geox_ok:
            yield y, m, g4d_path, geox_path
        else:
            missing = []
            if not g4d_ok:
                missing.append(g4d_path.name)
            if not geox_ok:
                missing.append(geox_path.name)
            print(f"[standardize] SKIPPING {y:04d}-{m:02d}: missing file(s) {missing}")
        m += 1
        if m > 12:
            m = 1
            y += 1


**`bob_local_pipeline/grid.py`** — Regridding both source products onto the common 1°×1°, 26-depth-level grid.

In [ ]:
%%writefile bob_local_pipeline/grid.py
"""
grid.py — regrid a standardized (depth, lat, lon) DataArray from
standardize.py onto the common Bay of Bengal 1x1 degree target grid and the
common standard depth levels, then apply the boundary mask.

Horizontal regridding method differs by product, because they start from
genuinely different native grids:
  - G4D-DOC is already on a global 1x1 grid with cell centers at the same
    .5-degree convention as our target grid, so this is an exact reindex
    (no interpolation error at all) — verified against a real file.
  - GEOXYGEN is natively ~0.5deg in longitude but IRREGULARLY spaced in
    latitude (its own metadata says geospatial_lat_resolution=~0.28deg, and
    the actual latitude values are not evenly spaced) despite the "0p5deg"
    filename. It genuinely needs interpolation onto the target grid; this
    pipeline uses bilinear (xarray/scipy linear) interpolation, not
    xesmf conservative regridding, because xesmf requires the ESMF C
    library which isn't guaranteed to be installed. If you have xesmf
    working locally, see the note at the bottom of this file for how to
    swap it in.

Vertical regridding: G4D-DOC's own 26 depth levels (config.STANDARD_DEPTHS_M)
are used as the common target for both products, so G4D-DOC needs no
vertical interpolation either — only GEOXYGEN (187 native levels) is
interpolated vertically, via linear interpolation (cubic requires evenly
spaced monotonic input in scipy for reliable behavior here; linear is the
safer default across an unevenly spaced 187-level axis. See README for how
to switch to cubic if you want it).
"""
from __future__ import annotations

import numpy as np
import xarray as xr

from . import config


def regrid_g4d_doc(da: xr.DataArray, target_lat, target_lon) -> xr.DataArray:
    """Exact reindex onto the target grid (G4D-DOC is already on this grid
    within floating-point tolerance)."""
    # Sort ascending first (G4D-DOC's native lat axis is descending; reindex
    # needs a consistent, matchable set of labels regardless of order).
    da = da.sortby("lat").sortby("lon")
    out = da.reindex(lat=target_lat, lon=target_lon, method="nearest", tolerance=1e-2)
    missing = int(out.isel(depth=0).isnull().sum()) - int(da.sel(
        lat=slice(target_lat.min(), target_lat.max()),
        lon=slice(target_lon.min(), target_lon.max()),
    ).isel(depth=0).isnull().sum())
    if missing > 0:
        print(f"[grid] NOTE: {missing} target grid cell(s) had no exact G4D-DOC "
              "match within tolerance and came back NaN — check grid alignment "
              "if this number looks large.")
    return out


def regrid_geoxygen(da: xr.DataArray, target_lat, target_lon) -> xr.DataArray:
    """Bilinear interpolation onto the target grid, using GEOXYGEN's real
    (irregularly spaced) lat/lon coordinate values — see module docstring."""
    margin = config.BBOX_MARGIN_DEG
    da_pref = da.sel(
        lat=slice(target_lat.min() - margin, target_lat.max() + margin),
        lon=slice(target_lon.min() - margin, target_lon.max() + margin),
    )
    if da_pref.sizes.get("lat", 0) == 0 or da_pref.sizes.get("lon", 0) == 0:
        raise ValueError(
            "GEOXYGEN pre-filtered bbox came back empty — check that its "
            "latitude/longitude coordinates actually cover the Bay of Bengal "
            "(config.STUDY_AREA_BBOX)."
        )
    return da_pref.interp(lat=target_lat, lon=target_lon, method="linear")


def regrid_depth(da: xr.DataArray, target_depths) -> xr.DataArray:
    """Interpolate onto config.STANDARD_DEPTHS_M. A no-op (just a reindex)
    when da's own depth axis already equals target_depths (G4D-DOC's case)."""
    if len(da["depth"]) == len(target_depths) and np.allclose(da["depth"].values, target_depths):
        return da
    max_native_depth = float(da["depth"].max())
    if max_native_depth < max(target_depths):
        print(f"[grid] WARNING: source depth axis only reaches {max_native_depth} m, "
              f"but target depths go to {max(target_depths)} m — deeper target "
              "levels will be NaN (extrapolation is deliberately not performed).")
    return da.interp(depth=list(target_depths), method="linear")


def build_gridded_product(da: xr.DataArray, target_lat, target_lon, target_depths,
                           mask: xr.DataArray) -> xr.Dataset:
    """Full per-product pipeline: horizontal regrid -> vertical regrid ->
    boundary mask -> standardized output Dataset with CF-ish metadata."""
    source = da.attrs.get("source_product")
    if source == "G4D-DOC":
        h = regrid_g4d_doc(da, target_lat, target_lon)
    elif source == "GEOXYGEN":
        h = regrid_geoxygen(da, target_lat, target_lon)
    else:
        raise ValueError(f"Unknown source_product attr: {source!r}")

    hv = regrid_depth(h, target_depths)
    masked = hv.where(mask)

    out = masked.to_dataset(name=config.OUTPUT_VARIABLE_NAME)
    out[config.OUTPUT_VARIABLE_NAME].attrs.update({
        "long_name": "Dissolved oxygen concentration",
        "units": config.OUTPUT_UNITS,
        "standard_name": "mole_concentration_of_dissolved_molecular_oxygen_in_sea_water",
    })
    out.attrs.update({
        "title": f"Bay of Bengal gridded {source} dissolved oxygen",
        "source_product": source,
        "source_file": da.attrs.get("source_file", ""),
        "grid_resolution_deg": config.GRID_RESOLUTION_DEG,
        "crs": config.GRID_CRS,
        "depth_levels_m": ",".join(str(d) for d in target_depths),
        "boundary_shapefile": config.BOUNDARY_SHAPEFILE.name,
        "processing": "clipped to Bay of Bengal polygon; horizontally and vertically "
                       "regridded to the common 1x1 deg / standard-depth grid",
    })
    return out


# ---------------------------------------------------------------------------
# Optional: swap in true conservative (area-weighted) regridding for
# GEOXYGEN if you have xesmf + ESMF installed locally (conda install -c
# conda-forge xesmf esmpy). Bilinear interpolation (used above) is a
# documented, defensible fallback but does not exactly conserve area-mean
# oxygen the way conservative regridding does. To switch:
#
#   import xesmf as xe
#   regridder = xe.Regridder(da_pref, target_grid_ds, method="conservative")
#   regridded = regridder(da_pref)
#
# and use that in place of regrid_geoxygen()'s .interp() call. Left as a
# manual step rather than auto-detected, so a run's regridding method is
# always explicit and reproducible rather than silently depending on what
# happens to be installed.
# ---------------------------------------------------------------------------


**`bob_local_pipeline/ensemble.py`** — NaN-aware simple-average ensembling of the two regridded products.

In [ ]:
%%writefile bob_local_pipeline/ensemble.py
"""
ensemble.py — combine one month's gridded G4D-DOC and GEOXYGEN products
into a single ensembled oxygen field.

Per project decision (2026-08-19), the active method is a SIMPLE AVERAGE —
deliberately not reliability weighting, since that needs independent
validation data (GLODAPv3/BGC-Argo) that this pipeline does not use.
reliability_weighted_mean() below is kept ready to use if that changes
later, but is not called by run_pipeline.py.
"""
from __future__ import annotations

import xarray as xr

from . import config


def simple_mean_ensemble(g4d_ds: xr.Dataset, geox_ds: xr.Dataset) -> xr.Dataset:
    """Unweighted mean of the two gridded products, per grid cell/depth.
    NaN-aware: if one product is missing at a cell (e.g. GEOXYGEN's depth
    range fell short), the other product's value is used alone rather than
    the cell going to NaN."""
    var = config.OUTPUT_VARIABLE_NAME
    stacked = xr.concat(
        [g4d_ds[var], geox_ds[var]],
        dim=xr.DataArray(["G4D-DOC", "GEOXYGEN"], dims="product", name="product"),
    )
    mean = stacked.mean(dim="product", skipna=True)
    n_products = stacked.notnull().sum(dim="product")

    out = mean.to_dataset(name=var)
    out[var].attrs.update({
        "long_name": "Dissolved oxygen concentration (ensemble mean)",
        "units": config.OUTPUT_UNITS,
        "standard_name": "mole_concentration_of_dissolved_molecular_oxygen_in_sea_water",
    })
    out["n_products_averaged"] = n_products
    out["n_products_averaged"].attrs["long_name"] = (
        "Number of source products (of 2: G4D-DOC, GEOXYGEN) contributing "
        "to the ensemble mean at this cell"
    )
    out.attrs.update({
        "title": "Bay of Bengal ensemble dissolved oxygen (G4D-DOC + GEOXYGEN, simple mean)",
        "ensemble_method": "simple_mean",
        "source_products": "G4D-DOC, GEOXYGEN",
        "grid_resolution_deg": config.GRID_RESOLUTION_DEG,
        "crs": config.GRID_CRS,
        "boundary_shapefile": config.BOUNDARY_SHAPEFILE.name,
        "note": "GOBAI-O2 and reliability weighting are deliberately not used "
                "in this ensemble — see config.py ENSEMBLE_METHOD comment.",
    })
    return out


def reliability_weighted_mean(g4d_ds: xr.Dataset, geox_ds: xr.Dataset,
                               weight_g4d_doc: float, weight_geoxygen: float) -> xr.Dataset:
    """NOT currently used by run_pipeline.py (see module docstring). Kept
    here ready to use if you later derive per-product reliability weights
    from independent validation data (e.g. GLODAPv3/BGC-Argo cross-
    validation RMSE) — pass fixed scalar weights that sum to 1."""
    if abs((weight_g4d_doc + weight_geoxygen) - 1.0) > 1e-6:
        raise ValueError("weight_g4d_doc + weight_geoxygen must sum to 1.0")
    var = config.OUTPUT_VARIABLE_NAME
    weighted = g4d_ds[var] * weight_g4d_doc + geox_ds[var] * weight_geoxygen
    out = weighted.to_dataset(name=var)
    out.attrs.update({
        "title": "Bay of Bengal ensemble dissolved oxygen (reliability-weighted)",
        "ensemble_method": "reliability_weighted",
        "weight_G4D-DOC": weight_g4d_doc,
        "weight_GEOXYGEN": weight_geoxygen,
    })
    return out


**`bob_local_pipeline/run_pipeline.py`** — The driver that ties the above together for all 216 months, with logging and resumability.

In [ ]:
%%writefile bob_local_pipeline/run_pipeline.py
"""
run_pipeline.py — the main driver. Run this with:

    python -m bob_local_pipeline.run_pipeline

from the folder that CONTAINS bob_local_pipeline (i.e. C:\\All_data, if you
kept the layout this was delivered with), or open
Bay_of_Bengal_Grid_Ensemble.ipynb in VS Code for the same steps cell-by-cell.

What it does, per month (2005-01 .. 2022-12, wherever both source files
exist):
  1. Load + standardize that month's G4D-DOC and GEOXYGEN files.
  2. Regrid each to the common 1x1 deg / standard-depth grid and clip to
     the Bay of Bengal polygon -> two "gridded" NetCDF files.
  3. Simple-average the two gridded products -> one "ensemble" NetCDF file.

Safe to interrupt and re-run: any month whose 3 output files already exist
is skipped (not recomputed), so a stopped run just picks up where it left
off. Delete a specific month's output files to force it to be redone.
"""
from __future__ import annotations

import sys
import time
import traceback
from pathlib import Path

import pandas as pd

from . import config
from . import boundary
from . import standardize
from . import grid
from . import ensemble


def _output_paths(year: int, month: int):
    g4d_out = config.GRIDDED_G4D_DOC_DIR / f"G4D_DOC_gridded_{year:04d}_{month:02d}.nc"
    geox_out = config.GRIDDED_GEOXYGEN_DIR / f"GEOXYGEN_gridded_{year:04d}_{month:02d}.nc"
    ens_out = config.ENSEMBLE_DIR / f"BoB_Oxygen_Ensemble_{year:04d}_{month:02d}.nc"
    return g4d_out, geox_out, ens_out


def process_month(year: int, month: int, g4d_path: Path, geox_path: Path,
                   target_lat, target_lon, mask) -> dict:
    """Process one month end-to-end. Returns a status dict for the run log.
    Raises nothing — all errors are caught and recorded, so one bad month
    doesn't stop the other 215."""
    g4d_out, geox_out, ens_out = _output_paths(year, month)
    t0 = time.time()

    if g4d_out.exists() and geox_out.exists() and ens_out.exists():
        return dict(year=year, month=month, status="SKIPPED_EXISTS",
                     message="all 3 output files already exist", seconds=0.0)

    try:
        g4d_raw = standardize.load_g4d_doc_month(g4d_path)
        geox_raw = standardize.load_geoxygen_month(geox_path)

        g4d_gridded = grid.build_gridded_product(
            g4d_raw, target_lat, target_lon, config.STANDARD_DEPTHS_M, mask)
        geox_gridded = grid.build_gridded_product(
            geox_raw, target_lat, target_lon, config.STANDARD_DEPTHS_M, mask)

        g4d_out.parent.mkdir(parents=True, exist_ok=True)
        geox_out.parent.mkdir(parents=True, exist_ok=True)
        ens_out.parent.mkdir(parents=True, exist_ok=True)

        g4d_gridded.to_netcdf(g4d_out)
        geox_gridded.to_netcdf(geox_out)

        ens = ensemble.simple_mean_ensemble(g4d_gridded, geox_gridded)
        ens.to_netcdf(ens_out)

        return dict(year=year, month=month, status="OK", message="",
                     seconds=round(time.time() - t0, 1))
    except Exception as e:
        return dict(year=year, month=month, status="FAILED",
                     message=f"{type(e).__name__}: {e}",
                     seconds=round(time.time() - t0, 1),
                     traceback=traceback.format_exc())


def main():
    print("=" * 70)
    print("BAY OF BENGAL G4D-DOC + GEOXYGEN GRID/ENSEMBLE PIPELINE")
    print("=" * 70)
    print(f"Raw data dir : {config.RAW_DATA_DIR}")
    print(f"Output dir   : {config.OUTPUT_DIR}")
    print(f"Period       : {config.START_YEAR}-{config.START_MONTH:02d} to "
          f"{config.END_YEAR}-{config.END_MONTH:02d}")
    print(f"Ensemble     : {config.ENSEMBLE_METHOD}")
    print()

    if not config.BOUNDARY_SHAPEFILE.exists():
        print(f"FATAL: boundary shapefile not found at {config.BOUNDARY_SHAPEFILE}")
        print("Check config.py PROJECT_ROOT / BOUNDARY_SHAPEFILE.")
        sys.exit(1)

    polygon = boundary.load_boundary_polygon()
    target_lat, target_lon = boundary.build_target_grid()
    mask = boundary.build_grid_mask(polygon, target_lat, target_lon)

    for d in (config.GRIDDED_G4D_DOC_DIR, config.GRIDDED_GEOXYGEN_DIR,
              config.ENSEMBLE_DIR, config.LOG_DIR):
        d.mkdir(parents=True, exist_ok=True)

    months = list(standardize.iter_available_months())
    print(f"\nFound {len(months)} month(s) with both source files present.\n")

    results = []
    t_start = time.time()
    for i, (year, month, g4d_path, geox_path) in enumerate(months, start=1):
        r = process_month(year, month, g4d_path, geox_path, target_lat, target_lon, mask)
        results.append(r)
        status_tag = f"[{r['status']:15s}]"
        print(f"{status_tag} {year:04d}-{month:02d}  ({i}/{len(months)}, "
              f"{r['seconds']}s) {r['message']}")

    log_df = pd.DataFrame(results)
    log_path = config.LOG_DIR / "run_log.csv"
    log_df.to_csv(log_path, index=False)

    n_ok = (log_df["status"] == "OK").sum() if len(log_df) else 0
    n_skipped = (log_df["status"] == "SKIPPED_EXISTS").sum() if len(log_df) else 0
    n_failed = (log_df["status"] == "FAILED").sum() if len(log_df) else 0

    print("\n" + "=" * 70)
    print("RUN SUMMARY")
    print("=" * 70)
    print(f"Processed OK      : {n_ok}")
    print(f"Already done      : {n_skipped}")
    print(f"Failed            : {n_failed}")
    print(f"Total elapsed     : {round(time.time() - t_start, 1)}s")
    print(f"Full log written  : {log_path}")
    if n_failed:
        print("\nFailed months:")
        for _, row in log_df[log_df["status"] == "FAILED"].iterrows():
            print(f"  {row['year']:04d}-{row['month']:02d}: {row['message']}")
        print("\nRe-run this script after fixing the issue — completed months "
              "are skipped automatically, only the failed ones will retry.")


if __name__ == "__main__":
    main()


Quick import check before moving on:

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from bob_local_pipeline import config, boundary, standardize, grid, ensemble, run_pipeline

print("Setup OK. Package version:", __import__("bob_local_pipeline").__version__)

---

## Part A — Grid + Ensemble Pipeline (G4D-DOC + GEOXYGEN)

For each of the 216 months (2005-01 to 2022-12): load that month's G4D-DOC and GEOXYGEN files,
clip both to the real Bay of Bengal boundary polygon, regrid both onto the same 1°×1° grid and 26
standard depth levels (10-1995 m), and average the two gridded products together into one ensemble
file. Sections 8-13 below build the monthly/yearly climatologies, depth-layer collapses, kriging
interpolation, and spatial-map figures on top of that ensemble.

## 1. Setup

Run this once per session. If any import fails, see `requirements.txt` / `README.md` for how to install it.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))  # run this notebook from C:\All_data (or wherever bob_local_pipeline/ lives)

from bob_local_pipeline import config, boundary, standardize, grid, ensemble, run_pipeline

import xarray as xr
import matplotlib.pyplot as plt

print("Setup OK. Package version:", __import__("bob_local_pipeline").__version__)


## 2. Configuration

Everything here comes from `bob_local_pipeline/config.py`. Change values there (not here) if you need to — this cell just shows you what's currently active so you can sanity-check it before a full run.


In [ ]:
print("Raw data dir       :", config.RAW_DATA_DIR)
print("Boundary shapefile :", config.BOUNDARY_SHAPEFILE, "  exists:", config.BOUNDARY_SHAPEFILE.exists())
print("Output dir         :", config.OUTPUT_DIR)
print("Period             :", f"{config.START_YEAR}-{config.START_MONTH:02d}", "to", f"{config.END_YEAR}-{config.END_MONTH:02d}")
print("Grid resolution    :", config.GRID_RESOLUTION_DEG, "deg")
print("Standard depths (m):", config.STANDARD_DEPTHS_M)
print("Ensemble method    :", config.ENSEMBLE_METHOD)


## 3. Boundary + target grid

Loads the real polygon (not a bounding box) and builds the 1x1 grid mask every month gets clipped against. Run once — the plot below is a quick visual check that the mask looks like the Bay of Bengal.

In [ ]:
polygon = boundary.load_boundary_polygon()
target_lat, target_lon = boundary.build_target_grid()
mask = boundary.build_grid_mask(polygon, target_lat, target_lon)

fig, ax = plt.subplots(figsize=(6, 5))
mask.plot(ax=ax, cmap="Blues", add_colorbar=False)
ax.set_title(f"Target grid mask ({int(mask.sum())} of {mask.size} cells inside the boundary)")
plt.tight_layout()
plt.show()


## 4. Test on a single month

Processes just one month and shows the result, so you can sanity-check everything (units, magnitudes, spatial pattern) before committing to the full 216-month run below. Change `TEST_YEAR`/`TEST_MONTH` to any month you have both files for.

In [ ]:
TEST_YEAR, TEST_MONTH = 2005, 1

g4d_path, geox_path = standardize.parse_filenames_for_month(TEST_YEAR, TEST_MONTH)
print("G4D-DOC file :", g4d_path, g4d_path.exists())
print("GEOXYGEN file:", geox_path, geox_path.exists())

result = run_pipeline.process_month(TEST_YEAR, TEST_MONTH, g4d_path, geox_path, target_lat, target_lon, mask)
print("\nResult:", {k: v for k, v in result.items() if k != "traceback"})
if result["status"] == "FAILED":
    print(result.get("traceback", ""))


In [ ]:
# Visualize the test month's ensemble at the shallowest depth level (10 m)
g4d_out, geox_out, ens_out = run_pipeline._output_paths(TEST_YEAR, TEST_MONTH)
ens = xr.open_dataset(ens_out)

fig, ax = plt.subplots(figsize=(6, 5))
ens["o2_umol_kg"].isel(depth=0).plot(ax=ax, cmap="viridis")
ax.set_title(f"Ensemble dissolved oxygen, {TEST_YEAR}-{TEST_MONTH:02d}, {float(ens.depth[0]):.0f} m")
plt.tight_layout()
plt.show()


## 5. Full run — all 216 months

This is the real run. It will take a while (expect a few minutes to tens of minutes depending on your machine — GEOXYGEN's interpolation step is the slow part). Progress prints one line per month; it's safe to interrupt (Kernel -> Interrupt) and re-run this cell later — finished months are skipped automatically.


In [ ]:
run_pipeline.main()


## 6. Check the results

Reads `outputs/logs/run_log.csv` and summarizes what happened.

In [ ]:
import pandas as pd

log_df = pd.read_csv(config.LOG_DIR / "run_log.csv")
print(log_df["status"].value_counts())

failed = log_df[log_df["status"] == "FAILED"]
if len(failed):
    print("\nFailed months:")
    display(failed[["year", "month", "message"]])
else:
    print("\nNo failures.")


## 7. Spot-check a few output months

Plots the ensemble field for a handful of months across the record, as a final visual sanity check that the whole 216-month run looks physically sensible (not just that it ran without errors).

In [ ]:
sample_months = [(2005, 1), (2010, 7), (2016, 1), (2022, 12)]

fig, axes = plt.subplots(1, len(sample_months), figsize=(5 * len(sample_months), 4), sharey=True)
for ax, (y, m) in zip(axes, sample_months):
    _, _, ens_path = run_pipeline._output_paths(y, m)
    if not ens_path.exists():
        ax.set_title(f"{y}-{m:02d} (not yet run)")
        continue
    ds = xr.open_dataset(ens_path)
    ds["o2_umol_kg"].isel(depth=0).plot(ax=ax, cmap="viridis", add_colorbar=(ax is axes[-1]))
    ax.set_title(f"{y}-{m:02d}")
plt.tight_layout()
plt.show()


## 8. Monthly climatology (2005-2022 average, per calendar month)

Averages the 216 ensemble files down to 12 — one per calendar month — by taking the mean across all 18 years for each calendar month (e.g. all 18 Januaries -> one January file). Full 26-level depth profile is kept. The mean at each depth/lat/lon cell is NaN-aware (`skipna=True`): if a cell is missing in some years, it's simply averaged over whichever years do have data there, with no minimum-years requirement. Each output file also records `n_years_averaged`, so you can see exactly how many of the (up to) 18 years actually contributed to each cell.

Outputs: `outputs/monthly_climatology/BoB_Oxygen_ClimMonth_01.nc` ... `_12.nc`.


In [ ]:
import xarray as xr
import pandas as pd

CLIM_DIR = config.OUTPUT_DIR / "monthly_climatology"
CLIM_DIR.mkdir(parents=True, exist_ok=True)

var = config.OUTPUT_VARIABLE_NAME

clim_summary = []
for month in range(1, 13):
    # Find every year's ensemble file for this calendar month.
    year_files = []
    for year in range(config.START_YEAR, config.END_YEAR + 1):
        p = config.ENSEMBLE_DIR / f"BoB_Oxygen_Ensemble_{year:04d}_{month:02d}.nc"
        if p.exists():
            year_files.append((year, p))
    if not year_files:
        print(f"[climatology] month {month:02d}: no ensemble files found, skipping")
        continue

    datasets = [xr.open_dataset(p)[var] for _, p in year_files]
    years_used = [y for y, _ in year_files]
    stacked = xr.concat(
        datasets,
        dim=xr.DataArray(years_used, dims="year", name="year"),
    )
    clim_mean = stacked.mean(dim="year", skipna=True)
    n_years = stacked.notnull().sum(dim="year")

    out = clim_mean.to_dataset(name=var)
    out[var].attrs.update({
        "long_name": "Dissolved oxygen concentration (calendar-month climatology, 2005-2022)",
        "units": config.OUTPUT_UNITS,
        "standard_name": "mole_concentration_of_dissolved_molecular_oxygen_in_sea_water",
    })
    out["n_years_averaged"] = n_years
    out["n_years_averaged"].attrs["long_name"] = (
        f"Number of years (of {len(years_used)} available: "
        f"{years_used[0]}-{years_used[-1]}) contributing to the mean at this cell"
    )
    out.attrs.update({
        "title": f"Bay of Bengal dissolved oxygen climatology, calendar month {month:02d}",
        "climatology_method": "simple_mean_across_years, skipna",
        "years_averaged": ",".join(str(y) for y in years_used),
        "source": "mean of BoB_Oxygen_Ensemble_YYYY_MM.nc ensemble files",
        "grid_resolution_deg": config.GRID_RESOLUTION_DEG,
        "crs": config.GRID_CRS,
        "boundary_shapefile": config.BOUNDARY_SHAPEFILE.name,
    })

    out_path = CLIM_DIR / f"BoB_Oxygen_ClimMonth_{month:02d}.nc"
    out.to_netcdf(out_path)

    n_finite = int(clim_mean.notnull().sum())
    clim_summary.append(dict(month=month, years_used=len(years_used),
                              n_finite_cells=n_finite, output=out_path.name))
    print(f"[climatology] month {month:02d}: averaged {len(years_used)} years "
          f"-> {out_path.name} ({n_finite} finite cells)")

pd.DataFrame(clim_summary)


## 9. Depth-layer climatology (6 layers, )

Collapses each 26-level monthly climatology file down to 6 depth layers, by taking a simple (unweighted) mean of whichever original depth levels fall inside each layer's range. This runs **before** kriging, and kriging (section 10) now interpolates these 6 layers instead of the original 26 individual depths.

| # | Layer | Depth range (m) | Original levels averaged |
|---|---|---|---|
| 1 | Surface Layer | 0-10 | 10 |
| 2 | Mixed Layer | 11-100 | 20, 30, 40, 50, 75, 100 |
| 3 | 100-400 m Layer | 101-400 | 125, 150, 200, 250, 300, 400 |
| 4 | 400-700 m Layer | 401-700 | 500, 600, 700 |
| 5 | 700-1000 m Layer | 701-1000 | 800, 900, 1000 |
| 6 | 1000-1995 m Layer | 1001-1995 | 1100, 1200, 1300, 1400, 1500, 1750, 1995 |

Every original depth level is used exactly once (no double-counting, no gaps) — this table is also printed and checked in code below. The Surface Layer is just the 10 m level as-is, since no shallower data exists in this project.

Outputs: `outputs/monthly_climatology_layers/BoB_Oxygen_ClimMonth_MM_Layers.nc` (12 files), each with a `depth_layer` dimension of size 6 (instead of the 26-level `depth` dimension), plus a `layer_name` coordinate so each layer is labeled, not just numbered.


In [ ]:
import numpy as np
import xarray as xr
import pandas as pd

LAYER_DEFS = [
    dict(name="Surface Layer",       depth_min=0,    depth_max=10),
    dict(name="Mixed Layer",         depth_min=11,   depth_max=100),
    dict(name="100-400 m Layer",     depth_min=101,  depth_max=400),
    dict(name="400-700 m Layer",     depth_min=401,  depth_max=700),
    dict(name="700-1000 m Layer",    depth_min=701,  depth_max=1000),
    dict(name="1000-1995 m Layer",   depth_min=1001, depth_max=1995),
]

LAYERS_DIR = config.OUTPUT_DIR / "monthly_climatology_layers"
LAYERS_DIR.mkdir(parents=True, exist_ok=True)

clim_files = sorted((config.OUTPUT_DIR / "monthly_climatology").glob("BoB_Oxygen_ClimMonth_*.nc"))
print(f"Found {len(clim_files)} monthly climatology file(s) to convert into {len(LAYER_DEFS)} depth layers each.\n")

layer_build_summary = []
levels_seen_overall = set()

for fi, f in enumerate(clim_files):
    month = int(f.stem.split("_")[-1])
    src = xr.open_dataset(f)
    all_depths = src["depth"].values

    layer_means, layer_names, layer_dmin, layer_dmax, layer_n = [], [], [], [], []
    for layer in LAYER_DEFS:
        layer_label, dmin, dmax = layer["name"], layer["depth_min"], layer["depth_max"]
        idx = np.where((all_depths >= dmin) & (all_depths <= dmax))[0]
        if len(idx) == 0:
            raise ValueError(f"Layer {layer_label} ({dmin}-{dmax} m) matched no depth levels "
                              "-- check LAYER_DEFS against config.STANDARD_DEPTHS_M.")
        if fi == 0:  # print the level assignment once, so it's easy to verify by eye
            levels_here = [float(all_depths[i]) for i in idx]
            print(f"  {layer_label:20s} ({dmin}-{dmax} m): {levels_here}")
            levels_seen_overall.update(all_depths[idx].tolist())
        layer_means.append(src["o2_umol_kg"].isel(depth=idx).mean(dim="depth", skipna=True))
        layer_names.append(layer_label)
        layer_dmin.append(dmin)
        layer_dmax.append(dmax)
        layer_n.append(len(idx))

    if fi == 0:
        missing = set(all_depths.tolist()) - levels_seen_overall
        extra = levels_seen_overall - set(all_depths.tolist())
        assert not missing and not extra, (
            f"Layer definitions do not exactly cover all 26 levels! "
            f"missing={missing} extra_or_duplicated={extra}"
        )
        print(f"  -> all {len(all_depths)} original depth levels accounted for exactly once.\n")

    stacked = xr.concat(layer_means, dim=pd.Index(range(1, len(LAYER_DEFS) + 1), name="depth_layer"))
    out = stacked.to_dataset(name="o2_umol_kg")
    out["o2_umol_kg"].attrs.update({
        "long_name": "Dissolved oxygen concentration (depth-layer mean of the monthly climatology)",
        "units": config.OUTPUT_UNITS,
        "standard_name": "mole_concentration_of_dissolved_molecular_oxygen_in_sea_water",
    })
    out["layer_name"] = ("depth_layer", layer_names)
    out["depth_min_m"] = ("depth_layer", layer_dmin)
    out["depth_max_m"] = ("depth_layer", layer_dmax)
    out["n_levels_averaged"] = ("depth_layer", layer_n)
    out.attrs.update({
        "title": f"Bay of Bengal 6-layer dissolved oxygen climatology, calendar month {month:02d}",
        "layer_method": "simple_unweighted_mean_of_original_depth_levels_within_each_layer",
        "source": f"depth-layer mean of {f.name}",
        "grid_resolution_deg": config.GRID_RESOLUTION_DEG,
        "crs": config.GRID_CRS,
        "boundary_shapefile": config.BOUNDARY_SHAPEFILE.name,
    })

    out_path = LAYERS_DIR / f"BoB_Oxygen_ClimMonth_{month:02d}_Layers.nc"
    out.to_netcdf(out_path)

    n_finite = int(stacked.notnull().sum())
    layer_build_summary.append(dict(month=month, n_finite_cells=n_finite, output=out_path.name))
    print(f"[layers] month {month:02d}: {len(LAYER_DEFS)} layers -> {out_path.name} ({n_finite} finite cells)")

pd.DataFrame(layer_build_summary)


## 10. Kriging interpolation (open-source equivalent of ArcGIS Pro's workflow)

Reproduces, in plain Python, the original ArcGIS Pro workflow: **Raster to Point** -> **Ordinary Kriging, spherical variogram** -> clip/mask to the Bay of Bengal boundary -> a new interpolated `.nc`. This runs on the **6 depth layers** from section 9 rather than all 26 individual depth levels — done separately for each of the 12 monthly files.

- **Raster to Point**: for each of the 6 depth layers, every finite grid cell in the layer-averaged file becomes a (lon, lat, value) point — this is exactly what ArcGIS's tool does, just done directly with the array instead of a raster layer.
- **Kriging (Ordinary & Spherical)**: [`pykrige`](https://github.com/GeostatsGuy/pykrige)'s `OrdinaryKriging` with `variogram_model="spherical"` — the open-source equivalent of `arcpy.sa.Kriging` with the same kriging type and variogram model as the original ArcGIS Pro workflow. Uses a 12-nearest-neighbor local search (`n_closest_points=12`), matching ArcGIS's own default local search neighborhood for Kriging, so it stays fast on a fine grid.
- **Processing Extent / Mask environments**: the output grid is built from `config.STUDY_AREA_BBOX` (the real shapefile bounds already used everywhere else in this notebook) — that's the "extent" — and every output cell is set to NaN unless its center falls inside the real `Bay_of_Bengal.shp` polygon, via the same `boundary.build_grid_mask()` used for the 1 deg grid — that's the "mask". Together these match the Processing Extent / Raster Analysis: Mask environment settings from the original ArcGIS Pro workflow.
- **Symbology / color ramp**: not applicable here — a "Stretched, Green to Blue" color ramp is a *display* setting, not something stored inside a NetCDF file, so it isn't part of these outputs. Apply it yourself when you load a result into ArcGIS Pro or plot it.
- Kriging is done **per depth layer** (6 per month, not 26) since kriging interpolates one 2D surface at a time — each output `.nc` keeps a `(depth_layer, lat, lon)` structure, interpolated at each of the 6 layers.

**Runtime is now much shorter** than a full 26-level run would have been — roughly a quarter of it (6/26), since there's 6 surfaces to krige per month instead of 26. Run the test cell first (10a) to confirm everything works and see real timing on your own machine before committing to the full run (10b).

**One-time setup**: this needs the `pykrige` package, which wasn't in the original `requirements.txt`. Run `pip install pykrige` in the same terminal/environment you used for the rest of this project, then restart the kernel if it was already running.


### 10a. Test — one month, one layer, coarse resolution

Sanity-checks the kriging logic and shows you a plot in well under a minute, at a deliberately coarse 0.2 deg test resolution (not the final 0.01 deg). Confirms the code runs correctly on your machine before you commit to the full job.

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from pykrige.ok import OrdinaryKriging

TEST_MONTH = 1
TEST_LAYER_IDX = 0  # Surface Layer (0-10 m)
TEST_RESOLUTION_DEG = 0.2  # coarse, for a fast sanity check only

polygon = boundary.load_boundary_polygon()

test_src = xr.open_dataset(config.OUTPUT_DIR / "monthly_climatology_layers" / f"BoB_Oxygen_ClimMonth_{TEST_MONTH:02d}_Layers.nc")
test_slice = test_src["o2_umol_kg"].isel(depth_layer=TEST_LAYER_IDX)
layer_label = str(test_src["layer_name"].isel(depth_layer=TEST_LAYER_IDX).values)

# "Raster to Point": flatten the finite grid cells of this one layer into points
lon2d, lat2d = np.meshgrid(test_slice.lon.values, test_slice.lat.values)
valid = test_slice.notnull().values
pts_lon, pts_lat, pts_val = lon2d[valid], lat2d[valid], test_slice.values[valid]
print(f"Month {TEST_MONTH:02d}, {layer_label}: {len(pts_val)} source points for kriging")

# "Kriging (Ordinary & Spherical)"
t0 = time.time()
ok_model = OrdinaryKriging(pts_lon, pts_lat, pts_val, variogram_model="spherical", verbose=False, enable_plotting=False)
test_lon = np.arange(config.STUDY_AREA_BBOX["lon_min"], config.STUDY_AREA_BBOX["lon_max"] + TEST_RESOLUTION_DEG, TEST_RESOLUTION_DEG)
test_lat = np.arange(config.STUDY_AREA_BBOX["lat_min"], config.STUDY_AREA_BBOX["lat_max"] + TEST_RESOLUTION_DEG, TEST_RESOLUTION_DEG)
z, ss = ok_model.execute("grid", test_lon, test_lat, backend="C", n_closest_points=12)
elapsed = time.time() - t0
n_points = len(test_lon) * len(test_lat)
print(f"Kriged {n_points} points in {elapsed:.2f}s ({elapsed / n_points * 1e6:.2f} microseconds/point on this machine)")

# "Mask = Bay_of_Bengal shapefile"
test_mask = boundary.build_grid_mask(polygon, test_lat, test_lon)
z_masked = np.where(test_mask.values, z, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
test_slice.plot(ax=axes[0], cmap="viridis")
axes[0].set_title(f"Original 1x1 deg layer, month {TEST_MONTH:02d}, {layer_label}")
axes[1].pcolormesh(test_lon, test_lat, z_masked, cmap="viridis")
axes[1].set_title(f"Kriged @ {TEST_RESOLUTION_DEG} deg (test), {layer_label}")
plt.tight_layout()
plt.show()


### 10b. Full run — all 12 months x 6 depth layers, 0.01 deg

**Runtime note**: based on the timing cell 10a just measured on your machine, this cell prints an estimated total time before it starts. It's safe to interrupt and re-run: any month whose output file already exists is skipped, so progress isn't lost — but if you interrupt *mid-month*, that month's partial file is not written, so it will simply restart from the beginning of that month next time.

Outputs: `outputs/interpolated_kriging/BoB_Oxygen_ClimMonth_MM_kriged_0p01deg.nc` (12 files), each with a `depth_layer` dimension of size 6.


In [ ]:
import numpy as np
import time
import pandas as pd
from pykrige.ok import OrdinaryKriging

KRIGE_RESOLUTION_DEG = 0.01  # change here (e.g. 0.05 or 0.1) for a faster, coarser run
N_CLOSEST_POINTS = 12         # local search neighborhood size, matches ArcGIS Kriging's default

KRIGE_DIR = config.OUTPUT_DIR / "interpolated_kriging"
KRIGE_DIR.mkdir(parents=True, exist_ok=True)

polygon = boundary.load_boundary_polygon()
fine_lon = np.arange(config.STUDY_AREA_BBOX["lon_min"], config.STUDY_AREA_BBOX["lon_max"] + KRIGE_RESOLUTION_DEG, KRIGE_RESOLUTION_DEG)
fine_lat = np.arange(config.STUDY_AREA_BBOX["lat_min"], config.STUDY_AREA_BBOX["lat_max"] + KRIGE_RESOLUTION_DEG, KRIGE_RESOLUTION_DEG)
fine_mask = boundary.build_grid_mask(polygon, fine_lat, fine_lon)
n_points_per_layer = len(fine_lat) * len(fine_lon)
print(f"Fine grid: {len(fine_lat)} x {len(fine_lon)} = {n_points_per_layer:,} points per depth layer\n")

layer_files = sorted((config.OUTPUT_DIR / "monthly_climatology_layers").glob("BoB_Oxygen_ClimMonth_*_Layers.nc"))
print(f"Found {len(layer_files)} monthly depth-layer file(s) to interpolate.\n")

krige_summary = []
t_run_start = time.time()
first_layer_timed = False

for f in layer_files:
    month = int(f.stem.split("_")[3])  # "BoB_Oxygen_ClimMonth_MM_Layers" -> MM
    out_path = KRIGE_DIR / f"BoB_Oxygen_ClimMonth_{month:02d}_kriged_0p01deg.nc"
    if out_path.exists():
        print(f"[krige] month {month:02d}: output already exists, skipping")
        krige_summary.append(dict(month=month, status="SKIPPED_EXISTS", seconds=0.0))
        continue

    src = xr.open_dataset(f)
    layer_ids = src["depth_layer"].values
    layer_names = src["layer_name"].values
    z_all = np.full((len(layer_ids), len(fine_lat), len(fine_lon)), np.nan)
    t_month_start = time.time()

    for li in range(len(layer_ids)):
        da = src["o2_umol_kg"].isel(depth_layer=li)
        layer_label = str(layer_names[li])
        lon2d, lat2d = np.meshgrid(da.lon.values, da.lat.values)
        valid = da.notnull().values
        pts_lon, pts_lat, pts_val = lon2d[valid], lat2d[valid], da.values[valid]

        if len(pts_val) < 4:
            print(f"  [krige] month {month:02d}, {layer_label}: only {len(pts_val)} "
                  "source points, too few to krige — leaving this layer as NaN")
            continue

        t0 = time.time()
        ok_model = OrdinaryKriging(pts_lon, pts_lat, pts_val, variogram_model="spherical",
                                     verbose=False, enable_plotting=False)
        z, ss = ok_model.execute("grid", fine_lon, fine_lat, backend="C",
                                   n_closest_points=N_CLOSEST_POINTS)
        z_all[li] = np.where(fine_mask.values, z, np.nan)

        if not first_layer_timed:
            first_layer_timed = True
            per_layer_s = time.time() - t0
            est_total_s = per_layer_s * len(layer_ids) * len(layer_files)
            print(f"[krige] first depth layer took {per_layer_s:.1f}s on this machine -> "
                  f"rough total estimate for the full run: {est_total_s / 60:.0f} minutes "
                  f"({est_total_s / 3600:.1f} hours)\n")

    out = xr.Dataset(
        {"o2_umol_kg": (("depth_layer", "lat", "lon"), z_all)},
        coords={"depth_layer": layer_ids, "lat": fine_lat, "lon": fine_lon},
    )
    out["layer_name"] = ("depth_layer", layer_names)
    out["o2_umol_kg"].attrs.update({
        "long_name": "Dissolved oxygen concentration (kriging-interpolated depth-layer climatology)",
        "units": config.OUTPUT_UNITS,
        "standard_name": "mole_concentration_of_dissolved_molecular_oxygen_in_sea_water",
    })
    out.attrs.update({
        "title": f"Bay of Bengal kriged dissolved oxygen 6-layer climatology, calendar month {month:02d}",
        "interpolation_method": "ordinary_kriging_spherical_variogram",
        "n_closest_points": N_CLOSEST_POINTS,
        "resolution_deg": KRIGE_RESOLUTION_DEG,
        "source": f"kriged from {f.name} (Raster to Point equivalent, per depth layer)",
        "boundary_shapefile": config.BOUNDARY_SHAPEFILE.name,
        "crs": config.GRID_CRS,
        "note": "Equivalent to ArcGIS Pro: Raster to Point -> Kriging (Ordinary, Spherical), "
                "Processing Extent/Mask = Bay_of_Bengal shapefile, applied to 6 depth layers "
                "instead of 26 individual depths. Color ramp/symbology is a display setting, "
                "not stored in this file.",
    })
    out.to_netcdf(out_path)

    month_elapsed = time.time() - t_month_start
    n_finite = int(np.isfinite(z_all).sum())
    krige_summary.append(dict(month=month, status="OK", seconds=round(month_elapsed, 1)))
    print(f"[krige] month {month:02d}: done in {month_elapsed / 60:.1f} min "
          f"-> {out_path.name} ({n_finite:,} finite depth_layer/lat/lon cells)")

total_elapsed = time.time() - t_run_start
print(f"\nTotal elapsed: {total_elapsed / 60:.1f} minutes")
pd.DataFrame(krige_summary)


## 11. Spatial maps (high-resolution PNGs)

Six PNG maps built from `outputs/interpolated_kriging/`, using the 6 kriged depth layers from section 10:

1. **Surface Layer (0-10 m)** — 12-panel grid (3 columns x 4 rows), one panel per calendar month.
2. **Mixed Layer (11-100 m)** — 12-panel grid, one panel per calendar month.
3. **Upper Twilight Layer (101-400 m)** — single map, the 12-month average.
4. **Middle Twilight Layer (401-700 m)** — single map, the 12-month average.
5. **Lower Twilight Layer (701-1000 m)** — single map, the 12-month average.
6. **Aphotic Layer (1001-1995 m)** — single map, the 12-month average.

Design choices: the color scale is shared across all 12 panels within maps 1 and 2 (so a given color means the same oxygen value in every month), and independent for each of maps 3-6; each box uses a true 1:1 geographic aspect ratio (1 degree of latitude = 1 degree of longitude, physically); output is 300 DPI on a large figure. The color ramp is a custom 5-stop gradient -- Red -> Purple/Violet -> Blue -> Teal -> Green -- where red = lowest oxygen and green = highest oxygen. The color scale (vmin/vmax) is tightened to each map's actual data range (with a small padding), so the full color range is used even when a layer's real variation is narrow. Black isooxygen contour lines are labeled and drawn at a round step (20, 10, 5, 2, or 1 umol/kg) chosen per map so every map gets 3-5 lines, regardless of how wide or narrow that map's data range is. Font is Times New Roman if it's installed on this machine (it falls back to the closest available serif font otherwise, with a printed note).

Outputs: `outputs/figures/Map1_Surface_Layer_Monthly.png` ... `Map6_Aphotic_Layer_Average.png` (6 files). This reads all 12 large (~140 MB) kriged files per map and renders at 300 DPI, so it will take a few minutes and several GB of memory at peak — the code below is written to keep memory use as low as practical (loading one month at a time, float32 arrays, closing files immediately) rather than holding everything at full precision at once.


In [ ]:
import gc
import numpy as np
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MultipleLocator, FuncFormatter

FIG_DIR = config.OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# --- Font: Times New Roman, with a visible fallback if it's not installed ---
available_fonts = set(f.name for f in fm.fontManager.ttflist)
if "Times New Roman" in available_fonts:
    plt.rcParams["font.family"] = "Times New Roman"
else:
    plt.rcParams["font.family"] = "serif"
    plt.rcParams["font.serif"] = ["Times New Roman", "Liberation Serif", "Times", "DejaVu Serif"]
    print("[fonts] 'Times New Roman' was not found on this machine -- using the closest "
          "available serif font instead. On a normal Windows install Times New Roman "
          "should be found automatically; if these maps don't look right, check your system fonts.")

# --- Color ramp: 5-stop gradient ---
# Red (lowest) -> Purple/Violet -> Blue -> Teal -> Green (highest).
# The Teal stop blends your two given hex codes (#0080FF and #00FF80) into
# one midpoint color (#00C0C0) between Blue and Green.
RED_TO_GREEN = LinearSegmentedColormap.from_list(
    "RedToGreen", ["#FF0000", "#8000FF", "#0000FF", "#00C0C0", "#00FF00"]
)

# --- Degree-only tick formatters (no minutes/seconds, no decimals) ---
def lat_formatter(x, pos):
    return f"{abs(round(x)):.0f}°N" if x >= 0 else f"{abs(round(x)):.0f}°S"

def lon_formatter(x, pos):
    return f"{abs(round(x)):.0f}°E" if x >= 0 else f"{abs(round(x)):.0f}°W"

MONTH_ABBR = ["JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"]

LAYER_TITLES = {
    1: "Monthly Climatological Dissolved Oxygen in the Surface Layer (0 m -10 m) of BoB",
    2: "Monthly Climatological Dissolved Oxygen in the Mixed Layer (11 m -100 m) of BoB",
    3: "Mean Dissolved Oxygen in the Upper Twilight Layer (101 m -400 m) of BoB",
    4: "Mean Dissolved Oxygen in the Middle Twilight Layer (401 m -700 m) of BoB",
    5: "Mean Dissolved Oxygen in the Lower Twilight Layer (701 m -1000 m) of BoB",
    6: "Mean Dissolved Oxygen in the Aphotic Layer (1001 m -1995 m) of BoB",
}

# Isolines are smooth, low-frequency features -- computing them on every single
# native 0.01deg cell (millions of points) costs a lot of memory/time for no
# visible gain. This stride only thins the CONTOUR LINE computation; the color
# fill below still uses the full native-resolution grid.
CONTOUR_STRIDE = 3

# Round contour-line spacings to try, largest (coarsest) first. Picks the
# coarsest step that still yields 3-5 contour lines for a given map's data
# range, so narrow-range layers (e.g. ~20 umol/kg span) still get enough
# isolines while wide-range layers don't get an overcrowded 20-unit grid.
CONTOUR_STEP_CANDIDATES = [50, 40, 30, 20, 10, 5, 2, 1, 0.5, 0.2, 0.1]

def pick_contour_levels(data_min, data_max, min_lines=3, max_lines=5):
    # A level at or beyond data_min/data_max is never actually crossed by the
    # field (nothing in the data reaches past its own min/max), so it never
    # renders as a visible line -- only levels STRICTLY INSIDE the data range
    # count towards min_lines/max_lines. Filtering to those interior levels
    # here (rather than counting the raw floor/ceil-bracketed level list) is
    # what makes this actually guarantee 3-5 visible isolines per map.
    fallback = None
    for step in CONTOUR_STEP_CANDIDATES:
        lo = np.floor(data_min / step) * step
        hi = np.ceil(data_max / step) * step
        levels = np.arange(lo, hi + step, step)
        visible = levels[(levels > data_min) & (levels < data_max)]
        if min_lines <= len(visible) <= max_lines:
            return visible, step
        if len(visible) >= min_lines and fallback is None:
            fallback = (visible, step)  # closest overshoot if no step lands in range
    if fallback is not None:
        return fallback
    step = CONTOUR_STEP_CANDIDATES[-1]  # data range is ~0 even at the finest step
    lo = np.floor(data_min / step) * step
    hi = np.ceil(data_max / step) * step
    levels = np.arange(lo, hi + step, step)
    return levels[(levels > data_min) & (levels < data_max)], step

def style_box(ax, lon, lat):
    # Shared per-box styling: degree-only ticks on left/bottom, true geographic
    # aspect ratio, no internal gridlines, box border kept.
    ax.set_xlim(lon.min(), lon.max())
    ax.set_ylim(lat.min(), lat.max())
    ax.set_aspect("equal", adjustable="box")  # 1 deg lat == 1 deg lon, physically
    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_major_locator(MultipleLocator(5))
    ax.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
    ax.yaxis.set_major_formatter(FuncFormatter(lat_formatter))
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)

def draw_field(ax, lon, lat, data, vmin, vmax, levels):
    # Stretched color fill (full native resolution) + labeled isooxygen
    # contour lines (computed on a thinned grid -- see CONTOUR_STRIDE).
    mesh = ax.pcolormesh(lon, lat, data, cmap=RED_TO_GREEN, vmin=vmin, vmax=vmax,
                          shading="auto", rasterized=True)
    s = CONTOUR_STRIDE
    cs = ax.contour(lon[::s], lat[::s], data[::s, ::s], levels=levels,
                     colors="black", linewidths=0.5)
    ax.clabel(cs, inline=True, fontsize=6, fmt="%d")
    return mesh

print("Section 11 setup complete. Font in use:", plt.rcParams["font.family"])


In [ ]:
PANEL_LAYERS = [1, 2]  # depth_layer index: 1 = Surface, 2 = Mixed

for layer_id in PANEL_LAYERS:
    # Load one month at a time as float32 (halves memory vs the source float64)
    # into a preallocated (12, ny, nx) stack -- avoids ever holding 12 separate
    # full xarray Datasets/DataArrays open at once.
    lon = lat = None
    stack = None
    for i, month in enumerate(range(1, 13)):
        p = config.OUTPUT_DIR / "interpolated_kriging" / f"BoB_Oxygen_ClimMonth_{month:02d}_kriged_0p01deg.nc"
        with xr.open_dataset(p) as ds:
            da = ds["o2_umol_kg"].sel(depth_layer=layer_id)
            vals = da.values.astype(np.float32)
            if lon is None:
                lon = da.lon.values.copy()
                lat = da.lat.values.copy()
                stack = np.empty((12,) + vals.shape, dtype=np.float32)
        stack[i] = vals
        del vals, da

    # Color scale (vmin/vmax) is tightened to the ACTUAL shared data range across
    # all 12 months (plus a small padding), so the full ramp is used even when
    # a layer's real variation is narrow. Contour levels are kept at round
    # multiples of 20 umol/kg, independent of the tightened color scale.
    # Color scale (vmin/vmax) stays SHARED across all 12 months, so the same
    # color always means the same oxygen value in every panel. Contour levels
    # are instead picked PER PANEL below, from that month's own data range --
    # a level chosen only from the pooled 12-month range can fall entirely
    # outside a narrower individual month, leaving that panel with zero
    # visible isolines even though the level list looked fine on paper.
    finite = stack[np.isfinite(stack)]
    data_min, data_max = float(finite.min()), float(finite.max())
    pad = max((data_max - data_min) * 0.05, 0.5)
    vmin, vmax = data_min - pad, data_max + pad
    del finite
    print(f"[map] layer {layer_id}: data range {data_min:.1f}-{data_max:.1f} umol/kg, "
          f"color scale {vmin:.1f}-{vmax:.1f} (shared across all 12 months)")

    fig = plt.figure(figsize=(16, 20), layout="constrained")
    gs = GridSpec(4, 4, width_ratios=[1, 1, 1, 0.06], figure=fig)

    mesh = None
    for i, month in enumerate(range(1, 13)):
        row, col = divmod(month - 1, 3)
        ax = fig.add_subplot(gs[row, col])
        month_finite = stack[i][np.isfinite(stack[i])]
        month_levels, month_step = pick_contour_levels(float(month_finite.min()), float(month_finite.max()))
        mesh = draw_field(ax, lon, lat, stack[i], vmin, vmax, month_levels)
        style_box(ax, lon, lat)
        ax.text(0.03, 0.97, MONTH_ABBR[month - 1], transform=ax.transAxes,
                 fontsize=14, fontweight="bold", va="top", ha="left")

    cax = fig.add_subplot(gs[:, 3])
    cbar = fig.colorbar(mesh, cax=cax)
    cbar.set_label("Dissolved Oxygen (µmol/kg)", fontsize=13)

    fig.suptitle(LAYER_TITLES[layer_id], fontsize=18, fontweight="bold")

    layer_tag = "Surface_Layer" if layer_id == 1 else "Mixed_Layer"
    out_path = FIG_DIR / f"Map{layer_id}_{layer_tag}_Monthly.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    del stack
    gc.collect()
    print(f"[map] saved {out_path.name}")


In [ ]:
SINGLE_LAYERS = [3, 4, 5, 6]  # 100-400m, 400-700m, 700-1000m, 1000-1995m

for map_num, layer_id in enumerate(SINGLE_LAYERS, start=3):
    # Running-sum accumulation instead of stacking all 12 months -- only ever
    # holds the accumulator plus one month's data in memory at a time.
    lon = lat = None
    running_sum = running_count = None
    for month in range(1, 13):
        p = config.OUTPUT_DIR / "interpolated_kriging" / f"BoB_Oxygen_ClimMonth_{month:02d}_kriged_0p01deg.nc"
        with xr.open_dataset(p) as ds:
            da = ds["o2_umol_kg"].sel(depth_layer=layer_id)
            vals = da.values
            if lon is None:
                lon = da.lon.values.copy()
                lat = da.lat.values.copy()
                running_sum = np.zeros_like(vals, dtype=np.float64)
                running_count = np.zeros_like(vals, dtype=np.int32)
            finite = np.isfinite(vals)
            running_sum[finite] += vals[finite]
            running_count[finite] += 1
        del vals, da

    with np.errstate(invalid="ignore", divide="ignore"):
        avg = np.where(running_count > 0, running_sum / running_count, np.nan).astype(np.float32)
    del running_sum, running_count

    # Same tightened-to-actual-range approach as the panel maps (see above).
    finite_vals = avg[np.isfinite(avg)]
    data_min, data_max = float(finite_vals.min()), float(finite_vals.max())
    pad = max((data_max - data_min) * 0.05, 0.5)
    vmin, vmax = data_min - pad, data_max + pad
    levels, contour_step = pick_contour_levels(data_min, data_max)
    print(f"[map] layer {layer_id} (12-month average): data range {data_min:.1f}-{data_max:.1f} umol/kg, "
          f"color scale {vmin:.1f}-{vmax:.1f}, {len(levels)} contour levels "
          f"(step {contour_step})")

    fig, ax = plt.subplots(figsize=(9, 9))
    mesh = draw_field(ax, lon, lat, avg, vmin, vmax, levels)
    style_box(ax, lon, lat)

    cbar = fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Dissolved Oxygen (µmol/kg)", fontsize=13)

    ax.set_title(LAYER_TITLES[layer_id], fontsize=15, fontweight="bold", wrap=True)

    out_path = FIG_DIR / f"Map{map_num}_Layer{layer_id}_Average.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    del avg
    gc.collect()
    print(f"[map] saved {out_path.name}")


## 12. Yearly average (2005-2022, one file per year, full 26-level depth)

Averages the 216 ensemble files down to 18 -- one per calendar year -- by taking the mean across that year's (up to) 12 calendar months (e.g. all 12 months of 2005 -> one 2005 file). Full 26-level depth profile is kept. The mean at each depth/lat/lon cell is NaN-aware (`skipna=True`): if a cell is missing in some months, it's simply averaged over whichever months do have data there, with no minimum-months requirement -- same rule as the monthly climatology in section 8. Each output file also records `n_months_averaged`, so you can see exactly how many of the (up to) 12 months actually contributed to each cell.

Outputs: `outputs/yearly_average/BoB_Oxygen_YearAvg_2005.nc` ... `_2022.nc` (18 files).


In [ ]:
import xarray as xr
import pandas as pd

YEARLY_DIR = config.OUTPUT_DIR / "yearly_average"
YEARLY_DIR.mkdir(parents=True, exist_ok=True)

var = config.OUTPUT_VARIABLE_NAME

yearly_summary = []
for year in range(config.START_YEAR, config.END_YEAR + 1):
    # Find every calendar month's ensemble file for this year.
    month_files = []
    for month in range(1, 13):
        p = config.ENSEMBLE_DIR / f"BoB_Oxygen_Ensemble_{year:04d}_{month:02d}.nc"
        if p.exists():
            month_files.append((month, p))
    if not month_files:
        print(f"[yearly] year {year}: no ensemble files found, skipping")
        continue

    datasets = [xr.open_dataset(p)[var] for _, p in month_files]
    months_used = [m for m, _ in month_files]
    stacked = xr.concat(
        datasets,
        dim=xr.DataArray(months_used, dims="month", name="month"),
    )
    year_mean = stacked.mean(dim="month", skipna=True)
    n_months = stacked.notnull().sum(dim="month")

    out = year_mean.to_dataset(name=var)
    out[var].attrs.update({
        "long_name": "Dissolved oxygen concentration (yearly average)",
        "units": config.OUTPUT_UNITS,
        "standard_name": "mole_concentration_of_dissolved_molecular_oxygen_in_sea_water",
    })
    out["n_months_averaged"] = n_months
    n_months_used = len(months_used)
    out["n_months_averaged"].attrs["long_name"] = (
        f"Number of months (of {n_months_used} available this year) "
        "contributing to the mean at this cell"
    )
    months_joined = ",".join(f"{m:02d}" for m in months_used)
    out.attrs.update({
        "title": f"Bay of Bengal dissolved oxygen yearly average, {year}",
        "yearly_average_method": "simple_mean_across_months, skipna",
        "months_averaged": months_joined,
        "source": f"mean of BoB_Oxygen_Ensemble_{year:04d}_MM.nc ensemble files",
        "grid_resolution_deg": config.GRID_RESOLUTION_DEG,
        "crs": config.GRID_CRS,
        "boundary_shapefile": config.BOUNDARY_SHAPEFILE.name,
    })

    out_path = YEARLY_DIR / f"BoB_Oxygen_YearAvg_{year:04d}.nc"
    out.to_netcdf(out_path)

    n_finite = int(year_mean.notnull().sum())
    yearly_summary.append(dict(year=year, months_used=n_months_used,
                                n_finite_cells=n_finite, output=out_path.name))
    print(f"[yearly] year {year}: averaged {n_months_used} months "
          f"-> {out_path.name} ({n_finite} finite cells)")

pd.DataFrame(yearly_summary)


## 13. Yearly depth-layer average (6 layers, same definition as section 9)

Collapses each 26-level yearly-average file from section 12 down to the same 6 depth layers used throughout this project, by taking a simple (unweighted) mean of whichever original depth levels fall inside each layer's range. Uses the exact same `LAYER_DEFS` boundaries as section 9 (0-10 m, 11-100 m, 101-400 m, 401-700 m, 701-1000 m, 1001-1995 m), so a given layer means the same depth range everywhere in this notebook.

Outputs: `outputs/yearly_average_layers/BoB_Oxygen_YearAvg_2005_Layers.nc` ... `_2022_Layers.nc` (18 files), each with a `depth_layer` dimension of size 6 (instead of the 26-level `depth` dimension), plus a `layer_name` coordinate so each layer is labeled, not just numbered.


In [ ]:
import numpy as np
import xarray as xr
import pandas as pd

# Same 6-layer definition as section 9 (monthly climatology depth layers).
LAYER_DEFS = [
    dict(name="Surface Layer",       depth_min=0,    depth_max=10),
    dict(name="Mixed Layer",         depth_min=11,   depth_max=100),
    dict(name="100-400 m Layer",     depth_min=101,  depth_max=400),
    dict(name="400-700 m Layer",     depth_min=401,  depth_max=700),
    dict(name="700-1000 m Layer",    depth_min=701,  depth_max=1000),
    dict(name="1000-1995 m Layer",   depth_min=1001, depth_max=1995),
]

YEARLY_LAYERS_DIR = config.OUTPUT_DIR / "yearly_average_layers"
YEARLY_LAYERS_DIR.mkdir(parents=True, exist_ok=True)

yearly_files = sorted((config.OUTPUT_DIR / "yearly_average").glob("BoB_Oxygen_YearAvg_*.nc"))
n_layer_defs = len(LAYER_DEFS)
print(f"Found {len(yearly_files)} yearly-average file(s) to convert into {n_layer_defs} depth layers each.\n")

yearly_layer_summary = []
levels_seen_overall = set()

for fi, f in enumerate(yearly_files):
    year = int(f.stem.split("_")[-1])
    src = xr.open_dataset(f)
    all_depths = src["depth"].values

    layer_means, layer_names, layer_dmin, layer_dmax, layer_n = [], [], [], [], []
    for layer in LAYER_DEFS:
        layer_label, dmin, dmax = layer["name"], layer["depth_min"], layer["depth_max"]
        idx = np.where((all_depths >= dmin) & (all_depths <= dmax))[0]
        if len(idx) == 0:
            raise ValueError(f"Layer {layer_label} ({dmin}-{dmax} m) matched no depth levels "
                              "-- check LAYER_DEFS against config.STANDARD_DEPTHS_M.")
        if fi == 0:  # print the level assignment once, so it's easy to verify by eye
            levels_here = [float(all_depths[i]) for i in idx]
            print(f"  {layer_label:20s} ({dmin}-{dmax} m): {levels_here}")
            levels_seen_overall.update(all_depths[idx].tolist())
        layer_means.append(src["o2_umol_kg"].isel(depth=idx).mean(dim="depth", skipna=True))
        layer_names.append(layer_label)
        layer_dmin.append(dmin)
        layer_dmax.append(dmax)
        layer_n.append(len(idx))

    if fi == 0:
        missing = set(all_depths.tolist()) - levels_seen_overall
        extra = levels_seen_overall - set(all_depths.tolist())
        assert not missing and not extra, (
            f"Layer definitions do not exactly cover all 26 levels! "
            f"missing={missing} extra_or_duplicated={extra}"
        )
        print(f"  -> all {len(all_depths)} original depth levels accounted for exactly once.\n")

    stacked = xr.concat(layer_means, dim=pd.Index(range(1, n_layer_defs + 1), name="depth_layer"))
    out = stacked.to_dataset(name="o2_umol_kg")
    out["o2_umol_kg"].attrs.update({
        "long_name": "Dissolved oxygen concentration (depth-layer mean of the yearly average)",
        "units": config.OUTPUT_UNITS,
        "standard_name": "mole_concentration_of_dissolved_molecular_oxygen_in_sea_water",
    })
    out["layer_name"] = ("depth_layer", layer_names)
    out["depth_min_m"] = ("depth_layer", layer_dmin)
    out["depth_max_m"] = ("depth_layer", layer_dmax)
    out["n_levels_averaged"] = ("depth_layer", layer_n)
    out.attrs.update({
        "title": f"Bay of Bengal 6-layer dissolved oxygen yearly average, {year}",
        "layer_method": "simple_unweighted_mean_of_original_depth_levels_within_each_layer",
        "source": f"depth-layer mean of {f.name}",
        "grid_resolution_deg": config.GRID_RESOLUTION_DEG,
        "crs": config.GRID_CRS,
        "boundary_shapefile": config.BOUNDARY_SHAPEFILE.name,
    })

    out_path = YEARLY_LAYERS_DIR / f"BoB_Oxygen_YearAvg_{year:04d}_Layers.nc"
    out.to_netcdf(out_path)

    n_finite = int(stacked.notnull().sum())
    yearly_layer_summary.append(dict(year=year, n_finite_cells=n_finite, output=out_path.name))
    print(f"[yearly-layers] year {year}: {n_layer_defs} layers -> {out_path.name} ({n_finite} finite cells)")

pd.DataFrame(yearly_layer_summary)


---

## Part B — Yearly Maps, Oxygen-Zone Volumes & Seasonal Profiles

Continues the section numbering from Part A (sections 14-17). Requires `outputs/yearly_average/`
and `outputs/yearly_average_layers/` to already exist (built by sections 12/13 above).

## 14. Krige the yearly depth-layer files (2005, 2010, 2015, 2020 only)

This kriges only the 4 years the section 15 maps actually need -- not all 18 -- since that's the minimum required here and cuts runtime roughly 4x versus kriging every year. Same method as section 10: `pykrige`'s `OrdinaryKriging`, spherical variogram, 12-nearest-neighbor local search, output clipped to the real `Bay_of_Bengal.shp` polygon, run separately per depth layer (6 per year).

Outputs: `outputs/yearly_interpolated_kriging/BoB_Oxygen_YearAvg_2005_kriged_0p01deg.nc`, `_2010_...`, `_2015_...`, `_2020_...` (4 files), each with a `depth_layer` dimension of size 6, at 0.01 deg resolution.

Resumable: if you interrupt this cell, already-written year files are skipped on re-run (but a year interrupted mid-run restarts from that year's beginning, same as section 10).


In [ ]:
import numpy as np
import time
import pandas as pd
from pykrige.ok import OrdinaryKriging

KRIGE_RESOLUTION_DEG = 0.01
N_CLOSEST_POINTS = 12
MAP_YEARS = [2005, 2010, 2015, 2020]  # the only years section 15's maps need

KRIGE_YEARLY_DIR = config.OUTPUT_DIR / "yearly_interpolated_kriging"
KRIGE_YEARLY_DIR.mkdir(parents=True, exist_ok=True)

polygon = boundary.load_boundary_polygon()
fine_lon = np.arange(config.STUDY_AREA_BBOX["lon_min"], config.STUDY_AREA_BBOX["lon_max"] + KRIGE_RESOLUTION_DEG, KRIGE_RESOLUTION_DEG)
fine_lat = np.arange(config.STUDY_AREA_BBOX["lat_min"], config.STUDY_AREA_BBOX["lat_max"] + KRIGE_RESOLUTION_DEG, KRIGE_RESOLUTION_DEG)
fine_mask = boundary.build_grid_mask(polygon, fine_lat, fine_lon)
n_points_per_layer = len(fine_lat) * len(fine_lon)
print(f"Fine grid: {len(fine_lat)} x {len(fine_lon)} = {n_points_per_layer:,} points per depth layer")
print(f"Kriging years: {MAP_YEARS}\n")

yearly_krige_summary = []
t_run_start = time.time()
first_layer_timed = False

for year in MAP_YEARS:
    f = config.OUTPUT_DIR / "yearly_average_layers" / f"BoB_Oxygen_YearAvg_{year}_Layers.nc"
    out_path = KRIGE_YEARLY_DIR / f"BoB_Oxygen_YearAvg_{year}_kriged_0p01deg.nc"
    if out_path.exists():
        print(f"[krige-yearly] year {year}: output already exists, skipping")
        yearly_krige_summary.append(dict(year=year, status="SKIPPED_EXISTS", seconds=0.0))
        continue
    if not f.exists():
        print(f"[krige-yearly] year {year}: source file {f.name} not found, skipping "
              "-- run section 13 first.")
        yearly_krige_summary.append(dict(year=year, status="MISSING_SOURCE", seconds=0.0))
        continue

    src = xr.open_dataset(f)
    layer_ids = src["depth_layer"].values
    layer_names = src["layer_name"].values
    z_all = np.full((len(layer_ids), len(fine_lat), len(fine_lon)), np.nan)
    t_year_start = time.time()

    for li in range(len(layer_ids)):
        da = src["o2_umol_kg"].isel(depth_layer=li)
        layer_label = str(layer_names[li])
        lon2d, lat2d = np.meshgrid(da.lon.values, da.lat.values)
        valid = da.notnull().values
        pts_lon, pts_lat, pts_val = lon2d[valid], lat2d[valid], da.values[valid]

        if len(pts_val) < 4:
            print(f"  [krige-yearly] year {year}, {layer_label}: only {len(pts_val)} "
                  "source points, too few to krige -- leaving this layer as NaN")
            continue

        t0 = time.time()
        ok_model = OrdinaryKriging(pts_lon, pts_lat, pts_val, variogram_model="spherical",
                                     verbose=False, enable_plotting=False)
        z, ss = ok_model.execute("grid", fine_lon, fine_lat, backend="C",
                                   n_closest_points=N_CLOSEST_POINTS)
        z_all[li] = np.where(fine_mask.values, z, np.nan)

        if not first_layer_timed:
            first_layer_timed = True
            per_layer_s = time.time() - t0
            est_total_s = per_layer_s * len(layer_ids) * len(MAP_YEARS)
            print(f"[krige-yearly] first depth layer took {per_layer_s:.1f}s on this machine -> "
                  f"rough total estimate: {est_total_s / 60:.0f} minutes\n")

    out = xr.Dataset(
        {"o2_umol_kg": (("depth_layer", "lat", "lon"), z_all)},
        coords={"depth_layer": layer_ids, "lat": fine_lat, "lon": fine_lon},
    )
    out["layer_name"] = ("depth_layer", layer_names)
    out["o2_umol_kg"].attrs.update({
        "long_name": "Dissolved oxygen concentration (kriging-interpolated yearly depth-layer average)",
        "units": config.OUTPUT_UNITS,
        "standard_name": "mole_concentration_of_dissolved_molecular_oxygen_in_sea_water",
    })
    out.attrs.update({
        "title": f"Bay of Bengal kriged dissolved oxygen 6-layer yearly average, {year}",
        "interpolation_method": "ordinary_kriging_spherical_variogram",
        "n_closest_points": N_CLOSEST_POINTS,
        "resolution_deg": KRIGE_RESOLUTION_DEG,
        "source": f"kriged from {f.name} (Raster to Point equivalent, per depth layer)",
        "boundary_shapefile": config.BOUNDARY_SHAPEFILE.name,
        "crs": config.GRID_CRS,
    })
    out.to_netcdf(out_path)

    year_elapsed = time.time() - t_year_start
    n_finite = int(np.isfinite(z_all).sum())
    yearly_krige_summary.append(dict(year=year, status="OK", seconds=round(year_elapsed, 1)))
    print(f"[krige-yearly] year {year}: done in {year_elapsed / 60:.1f} min "
          f"-> {out_path.name} ({n_finite:,} finite depth_layer/lat/lon cells)")

total_elapsed = time.time() - t_run_start
print(f"\nTotal elapsed: {total_elapsed / 60:.1f} minutes")
pd.DataFrame(yearly_krige_summary)


## 15. Yearly spatial maps (Upper Three Layers / Lower Three Layers PNGs)

Two 12-panel PNGs built from `outputs/yearly_interpolated_kriging/` (section 14):

1. **Upper Three Layer** -- columns: Surface Layer, Mixed Layer, Upper Twilight Layer; rows: 2005, 2010, 2015, 2020.
2. **Lower Three Layer** -- columns: Middle Twilight Layer, Lower Twilight Layer, Aphotic Layer; rows: 2005, 2010, 2015, 2020.

Design choices: each PNG uses **one single color scale shared across all 12 panels** (so colors are directly comparable across both layer and year within that PNG -- note this means the deeper, lower-oxygen layers will look closer to flat/uniform next to the much higher surface-layer range, since they share one scale by design). Isooxygen contour levels are still picked **per panel** (3-5 round-numbered lines each, same fix as section 11) so every panel gets legible isolines regardless of the shared color scale. Each panel is labeled with the layer name and year, bold, upper-left, on two lines. Same Red-Purple-Blue-Teal-Green ramp, degree-only axis ticks, true geographic aspect ratio, and Times New Roman font as section 11. Output at 300 DPI on a large figure.

Outputs: `outputs/figures/Map7_UpperThreeLayers_Yearly.png`, `outputs/figures/Map8_LowerThreeLayers_Yearly.png`.

This section is self-contained -- it redefines its own color ramp/formatters/helpers rather than relying on section 11 having been run first, so it works whether or not you ran section 11 in this kernel session.


In [ ]:
import gc
import numpy as np
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MultipleLocator, FuncFormatter

FIG_DIR = config.OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

available_fonts = set(f.name for f in fm.fontManager.ttflist)
if "Times New Roman" in available_fonts:
    plt.rcParams["font.family"] = "Times New Roman"
else:
    plt.rcParams["font.family"] = "serif"
    plt.rcParams["font.serif"] = ["Times New Roman", "Liberation Serif", "Times", "DejaVu Serif"]
    print("[fonts] 'Times New Roman' was not found on this machine -- using the closest "
          "available serif font instead.")

# Same 5-stop Red -> Purple/Violet -> Blue -> Teal -> Green ramp as section 11.
RED_TO_GREEN = LinearSegmentedColormap.from_list(
    "RedToGreen", ["#FF0000", "#8000FF", "#0000FF", "#00C0C0", "#00FF00"]
)

def lat_formatter(x, pos):
    return f"{abs(round(x)):.0f}\u00b0N" if x >= 0 else f"{abs(round(x)):.0f}\u00b0S"

def lon_formatter(x, pos):
    return f"{abs(round(x)):.0f}\u00b0E" if x >= 0 else f"{abs(round(x)):.0f}\u00b0W"

LAYER_SHORT_NAMES = {
    1: "Surface Layer",
    2: "Mixed Layer",
    3: "Upper Twilight Layer",
    4: "Middle Twilight Layer",
    5: "Lower Twilight Layer",
    6: "Aphotic Layer",
}

CONTOUR_STRIDE = 3
CONTOUR_STEP_CANDIDATES = [50, 40, 30, 20, 10, 5, 2, 1, 0.5, 0.2, 0.1]

def pick_contour_levels(data_min, data_max, min_lines=3, max_lines=5):
    fallback = None
    for step in CONTOUR_STEP_CANDIDATES:
        lo = np.floor(data_min / step) * step
        hi = np.ceil(data_max / step) * step
        levels = np.arange(lo, hi + step, step)
        visible = levels[(levels > data_min) & (levels < data_max)]
        if min_lines <= len(visible) <= max_lines:
            return visible, step
        if len(visible) >= min_lines and fallback is None:
            fallback = (visible, step)
    if fallback is not None:
        return fallback
    step = CONTOUR_STEP_CANDIDATES[-1]
    lo = np.floor(data_min / step) * step
    hi = np.ceil(data_max / step) * step
    levels = np.arange(lo, hi + step, step)
    return levels[(levels > data_min) & (levels < data_max)], step

def style_box(ax, lon, lat):
    ax.set_xlim(lon.min(), lon.max())
    ax.set_ylim(lat.min(), lat.max())
    ax.set_aspect("equal", adjustable="box")
    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_major_locator(MultipleLocator(5))
    ax.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
    ax.yaxis.set_major_formatter(FuncFormatter(lat_formatter))
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)

def draw_field(ax, lon, lat, data, vmin, vmax, levels):
    mesh = ax.pcolormesh(lon, lat, data, cmap=RED_TO_GREEN, vmin=vmin, vmax=vmax,
                          shading="auto", rasterized=True)
    s = CONTOUR_STRIDE
    cs = ax.contour(lon[::s], lat[::s], data[::s, ::s], levels=levels,
                     colors="black", linewidths=0.5)
    ax.clabel(cs, inline=True, fontsize=6, fmt="%d")
    return mesh

print("Section 15 setup complete. Font in use:", plt.rcParams["font.family"])


In [ ]:
KRIGE_YEARLY_DIR = config.OUTPUT_DIR / "yearly_interpolated_kriging"
MAP_YEARS = [2005, 2010, 2015, 2020]

YEARLY_MAPS = [
    dict(
        title="Yearly Average Spatial Map in Upper Three Layer",
        layers=[1, 2, 3],
        filename="Map7_UpperThreeLayers_Yearly.png",
    ),
    dict(
        title="Yearly Average Spatial Map in Lower Three Layer",
        layers=[4, 5, 6],
        filename="Map8_LowerThreeLayers_Yearly.png",
    ),
]

for map_def in YEARLY_MAPS:
    map_title = map_def["title"]
    map_layers = map_def["layers"]
    map_filename = map_def["filename"]

    # One file open per year, reused across that year's 3 relevant layers --
    # avoids opening any file more than once.
    lon = lat = None
    panels = {}
    for year in MAP_YEARS:
        p = KRIGE_YEARLY_DIR / f"BoB_Oxygen_YearAvg_{year}_kriged_0p01deg.nc"
        with xr.open_dataset(p) as ds:
            for layer_id in map_layers:
                da = ds["o2_umol_kg"].sel(depth_layer=layer_id)
                vals = da.values.astype(np.float32)
                if lon is None:
                    lon = da.lon.values.copy()
                    lat = da.lat.values.copy()
                panels[(layer_id, year)] = vals

    # ONE shared color scale across all 12 panels in this PNG.
    all_vals = np.concatenate([v[np.isfinite(v)].ravel() for v in panels.values()])
    data_min, data_max = float(all_vals.min()), float(all_vals.max())
    pad = max((data_max - data_min) * 0.05, 0.5)
    vmin, vmax = data_min - pad, data_max + pad
    del all_vals
    print(f"[yearly-map] {map_title}: shared scale {vmin:.1f}-{vmax:.1f} umol/kg "
          f"(data range {data_min:.1f}-{data_max:.1f})")

    fig = plt.figure(figsize=(16, 20), layout="constrained")
    gs = GridSpec(4, 4, width_ratios=[1, 1, 1, 0.06], figure=fig)

    mesh = None
    for row, year in enumerate(MAP_YEARS):
        for col, layer_id in enumerate(map_layers):
            ax = fig.add_subplot(gs[row, col])
            data = panels[(layer_id, year)]
            panel_finite = data[np.isfinite(data)]
            panel_levels, panel_step = pick_contour_levels(
                float(panel_finite.min()), float(panel_finite.max())
            )
            mesh = draw_field(ax, lon, lat, data, vmin, vmax, panel_levels)
            style_box(ax, lon, lat)
            layer_label = LAYER_SHORT_NAMES[layer_id]
            ax.text(0.03, 0.97, f"{layer_label}\n{year}", transform=ax.transAxes,
                     fontsize=13, fontweight="bold", va="top", ha="left")

    cax = fig.add_subplot(gs[:, 3])
    cbar = fig.colorbar(mesh, cax=cax)
    cbar.set_label("Dissolved Oxygen (\u00b5mol/kg)", fontsize=13)

    fig.suptitle(map_title, fontsize=18, fontweight="bold")

    out_path = FIG_DIR / map_filename
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    del panels
    gc.collect()
    print(f"[yearly-map] saved {out_path.name}")


## 16. Yearly oxygen-zone volumes (Oxic / Hypoxic / Severe Hypoxic / Suboxic-Anoxic)

Computes, for each of the 18 years, the total ocean **volume** (not area) falling into each of 4 dissolved-oxygen zones, from `outputs/yearly_average/` (the 26-depth-level, 1x1 deg yearly-average files -- *not* the kriged or depth-layer-collapsed data).

**Zone thresholds** (`o2_umol_kg`) -- an oxygen value sitting exactly on a boundary counts toward the more hypoxic (lower-oxygen) zone:

| Zone | Condition |
|---|---|
| Oxic Zone | O₂ > 61 |
| Hypoxic Zone (Dead Zone) | 22 < O₂ ≤ 61 |
| Severe Hypoxic Zone (Critical Dead Zone) | 10 < O₂ ≤ 22 |
| Suboxic/Near Anoxic Zone (Core Dead Zone/Desert) | O₂ ≤ 10 |

**Volume method:**
- Each of the 26 depth levels is a single point sample (10, 20, 30, ... 1995 m), not a range, so each is first turned into a layer **thickness**: the boundary between two consecutive levels is their midpoint; the top edge is 0 m (sea surface); the bottom edge is extrapolated below the deepest level (1995 m) by the same half-spacing as the last two levels.
- Each 1x1 deg grid cell's surface **area** is computed with the exact spherical-cap formula (`R^2 * dlon_rad * (sin(lat_hi) - sin(lat_lo))`, `R = 6371 km`), so area shrinks correctly with latitude rather than being treated as constant.
- `cell volume = area(lat) x thickness(depth)`, summed over every finite grid cell whose oxygen value falls in that zone, for that year. A per-year sanity check confirms the 4 zone volumes sum exactly to that year's total finite ocean volume.
- Values are expressed in **million cubic kilometers (10⁶ km³)**.

**Output:** `outputs/oxygen_zone_volumes/BoB_Oxygen_Zone_Volumes_Yearly.xlsx` -- 18 rows (2005-2022), with the 4 zone columns under one genuinely merged header cell titled "Volume in Million Cubic Kilometers (10⁶ Km³)", using a true Excel merge rather than a CSV approximation.

**One-time setup**: this needs the `openpyxl` package for the merged-cell Excel output (`pip install openpyxl` if the next cell errors on import -- it's a common dependency and is very likely already installed alongside pandas).


In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

ZONE_DIR = config.OUTPUT_DIR / "oxygen_zone_volumes"
ZONE_DIR.mkdir(parents=True, exist_ok=True)

var = config.OUTPUT_VARIABLE_NAME
EARTH_RADIUS_KM = 6371.0

yearly_avg_files = sorted((config.OUTPUT_DIR / "yearly_average").glob("BoB_Oxygen_YearAvg_*.nc"))
print(f"Found {len(yearly_avg_files)} yearly-average file(s).\n")
if not yearly_avg_files:
    raise FileNotFoundError(
        "No files found in outputs/yearly_average/ -- run section 12 first."
    )

# --- Depth-bin thicknesses (km), from the first file's depth axis. ---
# Point samples -> layer thicknesses via midpoint boundaries; top edge = 0 m (sea
# surface); bottom edge extrapolated below the deepest level by the same
# half-spacing as the last two levels.
with xr.open_dataset(yearly_avg_files[0]) as first_ds:
    depths_m = first_ds["depth"].values.astype(float)
    lats = first_ds["lat"].values.astype(float)

edges_m = np.zeros(len(depths_m) + 1)
edges_m[0] = 0.0
edges_m[1:-1] = (depths_m[:-1] + depths_m[1:]) / 2.0
half_spacing = (depths_m[-1] - depths_m[-2]) / 2.0
edges_m[-1] = depths_m[-1] + half_spacing
thickness_km = (edges_m[1:] - edges_m[:-1]) / 1000.0

print("Depth bin edges (m):   ", np.round(edges_m, 1))
print("Depth bin thickness (km):", np.round(thickness_km, 4))

# --- Grid-cell surface area (km^2) by latitude band -- exact spherical formula. ---
# 1x1 deg cells; area depends only on latitude (longitude cancels out on a sphere).
dlon_rad = np.radians(config.GRID_RESOLUTION_DEG)
lat_lo = np.radians(lats - config.GRID_RESOLUTION_DEG / 2.0)
lat_hi = np.radians(lats + config.GRID_RESOLUTION_DEG / 2.0)
area_km2 = (EARTH_RADIUS_KM ** 2) * dlon_rad * (np.sin(lat_hi) - np.sin(lat_lo))

area_by_lat = xr.DataArray(area_km2, dims="lat", coords={"lat": lats})
thickness_by_depth = xr.DataArray(thickness_km, dims="depth", coords={"depth": depths_m})

# --- Zone columns ---
ZONE_COLUMNS = [
    "Oxic Zone",
    "Hypoxic Zone (Dead Zone)",
    "Severe Hypoxic Zone (Critical Dead Zone)",
    "Suboxic/Near Anoxic Zone (Core Dead Zone/Desert)",
]

def classify_volumes(da, cell_volume_km3):
    """dict of zone name -> total volume (km^3) for one year's oxygen field.
    An exact boundary value (61, 22, or 10) counts toward the lower/more
    hypoxic zone."""
    oxic = cell_volume_km3.where(da > 61)
    hypoxic = cell_volume_km3.where((da <= 61) & (da > 22))
    severe = cell_volume_km3.where((da <= 22) & (da > 10))
    suboxic = cell_volume_km3.where(da <= 10)
    return {
        ZONE_COLUMNS[0]: float(oxic.sum(skipna=True)),
        ZONE_COLUMNS[1]: float(hypoxic.sum(skipna=True)),
        ZONE_COLUMNS[2]: float(severe.sum(skipna=True)),
        ZONE_COLUMNS[3]: float(suboxic.sum(skipna=True)),
    }

rows = []
for f in yearly_avg_files:
    year = int(f.stem.split("_")[-1])
    with xr.open_dataset(f) as ds:
        da = ds[var]  # dims: depth, lat, lon

        cell_volume_km3 = (thickness_by_depth * area_by_lat).broadcast_like(da).where(da.notnull())
        zone_vol_km3 = classify_volumes(da, cell_volume_km3)

        total_km3 = float(cell_volume_km3.sum(skipna=True))
        zone_sum_km3 = sum(zone_vol_km3.values())
        rel_mismatch = abs(total_km3 - zone_sum_km3) / total_km3 if total_km3 else 0.0
        assert rel_mismatch < 1e-9, (
            f"Year {year}: zone volumes don't sum to the total ocean volume "
            f"(relative mismatch {rel_mismatch:.2e})"
        )

    row = {"Year": year}
    row.update({name: vol_km3 / 1e6 for name, vol_km3 in zone_vol_km3.items()})  # -> million km^3
    rows.append(row)
    print(f"[zone-volume] year {year}: total {total_km3 / 1e6:.4f} million km^3 "
          f"across all 4 zones (sanity check passed)")

zone_df = pd.DataFrame(rows).sort_values("Year").reset_index(drop=True)
print(f"\n{len(zone_df)} year(s) computed.")
zone_df


In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter

xlsx_path = ZONE_DIR / "BoB_Oxygen_Zone_Volumes_Yearly.xlsx"

wb = Workbook()
ws = wb.active
ws.title = "Zone Volumes"

HEADER_TITLE = "Volume in Million Cubic Kilometers (10\u2076 Km\u00b3)"

# Row 1: the 4 zone columns share one genuinely merged header cell.
ws.merge_cells(start_row=1, start_column=2, end_row=1, end_column=5)
title_cell = ws.cell(row=1, column=2, value=HEADER_TITLE)
title_cell.alignment = Alignment(horizontal="center", vertical="center")
title_cell.font = Font(bold=True)

# Row 2: "Year" + the 4 individual zone names.
year_header = ws.cell(row=2, column=1, value="Year")
year_header.font = Font(bold=True)
year_header.alignment = Alignment(horizontal="center", vertical="center")
for j, col_name in enumerate(ZONE_COLUMNS, start=2):
    c = ws.cell(row=2, column=j, value=col_name)
    c.font = Font(bold=True)
    c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

# Data rows (row 3 onward), values rounded to 4 decimal places for readability.
for i, data_row in zone_df.iterrows():
    r = i + 3
    ws.cell(row=r, column=1, value=int(data_row["Year"])).alignment = Alignment(horizontal="center")
    for j, col_name in enumerate(ZONE_COLUMNS, start=2):
        cell = ws.cell(row=r, column=j, value=round(float(data_row[col_name]), 4))
        cell.number_format = "0.0000"
        cell.alignment = Alignment(horizontal="center")

ws.column_dimensions["A"].width = 10
for j in range(2, 6):
    ws.column_dimensions[get_column_letter(j)].width = 26
ws.row_dimensions[2].height = 30
ws.freeze_panes = "A3"

wb.save(xlsx_path)
print(f"[zone-volume] wrote {xlsx_path}")
zone_df


## 17. Seasonal vertical oxygen profiles (4 seasons x 10 points)

One 2x2 master figure, one panel per season, each panel showing 10 vertical dissolved-oxygen profiles at fixed locations, built from `outputs/monthly_climatology/` (the 26-level, 1x1 deg calendar-month climatology -- *not* the depth-layer-collapsed data).

**Seasons** (equal-weight, NaN-aware mean across that season's calendar-month climatology files):

| Season | Months averaged |
|---|---|
| Southwest Monsoon | June, July, August |
| Post Monsoon | September, October |
| Northeast Monsoon | November, December, January, February |
| Pre-Monsoon | March, April, May |

**The 10 profile points** (85°E 8°N, 85°E 12°N, 85°E 16°N, 88°E 8°N, 88°E 12°N, 88°E 16°N, 91°E 8°N, 91°E 12°N, 91°E 16°N, 91°E 20°N) don't sit exactly on the 1x1 deg grid (which is centered at the .5° mark), so each profile uses its **nearest available grid cell**, the cell actually used for each point is printed once, so the offset is visible. The legend lists each point's label (e.g. "85°E 8°N") vertically, in the same order as the point list above, inside each panel.

**Design choices:** depth on the y-axis increasing downward (0 m at top, ~1995 m at bottom -- standard oceanographic convention), oxygen on the x-axis, 10 distinguishable categorical colors (matplotlib's `tab10`), Times New Roman font matching the other maps in this project. Any point with no finite data at all in a given season (e.g. a coastal point over shallow/land-adjacent grid cells) is skipped for that panel with a printed warning, rather than plotting an empty or broken line.

Output: `outputs/figures/Fig_Seasonal_Vertical_O2_Profiles.png` (300 DPI).


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

FIG_DIR = config.OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

available_fonts = set(f.name for f in fm.fontManager.ttflist)
if "Times New Roman" in available_fonts:
    plt.rcParams["font.family"] = "Times New Roman"
else:
    plt.rcParams["font.family"] = "serif"
    plt.rcParams["font.serif"] = ["Times New Roman", "Liberation Serif", "Times", "DejaVu Serif"]
    print("[fonts] 'Times New Roman' was not found on this machine -- using the closest "
          "available serif font instead.")

var = config.OUTPUT_VARIABLE_NAME
CLIM_DIR = config.OUTPUT_DIR / "monthly_climatology"

SEASONS = [
    dict(name="Southwest Monsoon (June - August)",       months=[6, 7, 8]),
    dict(name="Post Monsoon (September - October)",      months=[9, 10]),
    dict(name="Northeast Monsoon (November - February)", months=[11, 12, 1, 2]),
    dict(name="Pre-Monsoon (March - May)",                months=[3, 4, 5]),
]

# The 10 requested points, in the exact order the legend must list them.
PROFILE_POINTS = [
    (85, 8), (85, 12), (85, 16),
    (88, 8), (88, 12), (88, 16),
    (91, 8), (91, 12), (91, 16), (91, 20),
]

POINT_COLORS = plt.get_cmap("tab10").colors  # 10 distinguishable categorical colors

def season_mean_field(months):
    """Equal-weight, NaN-aware mean across this season's calendar-month
    climatology files."""
    das = []
    for m in months:
        p = CLIM_DIR / f"BoB_Oxygen_ClimMonth_{m:02d}.nc"
        if not p.exists():
            raise FileNotFoundError(f"{p} not found -- run section 8 first.")
        das.append(xr.open_dataset(p)[var])
    stacked = xr.concat(das, dim=xr.DataArray(months, dims="month", name="month"))
    return stacked.mean(dim="month", skipna=True)

# Report the actual nearest grid cell used for each requested point -- the grid is
# the same for every season, so this only needs doing once, against one reference file.
ref_path = CLIM_DIR / "BoB_Oxygen_ClimMonth_01.nc"
print("Nearest-grid-cell lookup for each requested point (the 1x1 deg grid is "
      "centered at the .5 deg mark, so the actual cell may differ slightly):")
with xr.open_dataset(ref_path) as ref_ds:
    for lon_pt, lat_pt in PROFILE_POINTS:
        nearest = ref_ds[var].sel(lon=lon_pt, lat=lat_pt, method="nearest")
        print(f"  requested {lon_pt}\u00b0E {lat_pt}\u00b0N -> nearest grid cell "
              f"{float(nearest['lon']):.2f}\u00b0E {float(nearest['lat']):.2f}\u00b0N")
print()

fig, axes = plt.subplots(2, 2, figsize=(14, 16), layout="constrained")
axes_flat = axes.flatten()

for ax, season in zip(axes_flat, SEASONS):
    field = season_mean_field(season["months"])  # dims: depth, lat, lon
    depths = field["depth"].values

    for (lon_pt, lat_pt), color in zip(PROFILE_POINTS, POINT_COLORS):
        profile = field.sel(lon=lon_pt, lat=lat_pt, method="nearest")
        values = profile.values
        finite = np.isfinite(values)
        if not finite.any():
            print(f"  [warning] {season['name']}: {lon_pt}\u00b0E {lat_pt}\u00b0N has no "
                  "finite data at any depth -- skipping this line.")
            continue
        ax.plot(values[finite], depths[finite], color=color, linewidth=1.6,
                 marker="o", markersize=3, label=f"{lon_pt}\u00b0E {lat_pt}\u00b0N")

    ax.set_title(season["name"], fontsize=13, fontweight="bold")
    ax.set_xlabel("Dissolved Oxygen (\u00b5mol/kg)", fontsize=11)
    ax.set_ylabel("Depth (m)", fontsize=11)
    ax.invert_yaxis()  # depth increasing downward
    ax.grid(True, linewidth=0.4, alpha=0.5)
    ax.legend(loc="best", fontsize=8, ncol=1, framealpha=0.9)

fig.suptitle("Seasonal Vertical Dissolved Oxygen Profiles at 10 Selected Locations",
             fontsize=17, fontweight="bold")

out_path = FIG_DIR / "Fig_Seasonal_Vertical_O2_Profiles.png"
fig.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"\n[profiles] saved {out_path}")


---

## Part C — 14-Analysis Deoxygenation Study

Runs 14 analyses on the ensemble dissolved-oxygen dataset built in Part A (`outputs/ensemble/`,
216 monthly files). This section keeps its own internal numbering (0 = setup/configuration,
1-14 = the analyses, then a closing summary), since it was originally developed as a
self-contained study with its own configuration cell.

**A note on honesty over polish**: several analyses below (8, 13, 14 especially) are written to
report what the data and methods actually show, including when a method *fails* to catch
something it "should" (e.g. whether change-point detection or the ML QC step actually flags the
known Dec 2019–Jan 2020 G4D-DOC data-quality anomaly), and to flag a real data gap (no
temperature/salinity product) rather than silently working around it.

## 0. Setup

### 0.0 Configuration

All paths, thresholds, and tunable parameters used anywhere in this notebook are defined in the single cell below. Change values here, not in the analysis cells further down, and re-run from this cell forward.

In [ ]:
import importlib
for pkg in ["xarray","netCDF4","numpy","scipy","pandas","matplotlib","pymannkendall","ruptures","tensorly","eofs","pywt","skgstat","esda","libpysal","gsw","sklearn","pycwt"]:
    try:
        importlib.import_module(pkg)
        print(f"OK  {pkg}")
    except ImportError as e:
        print(f"MISSING  {pkg}  ({e})")

In [ ]:
import sys
!{sys.executable} -m pip install pymannkendall ruptures scikit-gstat esda libpysal gsw pycwt

In [ ]:
"""
============================================================================
GLOBAL CONFIGURATION -- every path, threshold, and tunable parameter used
anywhere in this notebook lives here. Change values HERE, not inside the
analysis cells below, and re-run from this cell down.
============================================================================
"""
import warnings
from pathlib import Path
import numpy as np

# ---------------------------------------------------------------------------
# 1. DATA ROOT PATHS -- edit these if your folder layout differs from
#    C:\All_data\... . Everything else in the notebook is built from these.
# ---------------------------------------------------------------------------
PROJECT_ROOT          = Path(r"C:\All_data")
ENSEMBLE_DIR           = PROJECT_ROOT / "outputs" / "ensemble"                 # 216 monthly ensemble .nc files (Analyses 1-12, 14)
GRIDDED_G4D_DOC_DIR    = PROJECT_ROOT / "outputs" / "gridded_G4D_DOC"          # per-product grids, needed only for Analysis 13 (QC)
GRIDDED_GEOXYGEN_DIR   = PROJECT_ROOT / "outputs" / "gridded_GEOXYGEN"         # per-product grids, needed only for Analysis 13 (QC)
BOUNDARY_SHAPEFILE     = PROJECT_ROOT / "Bay_of_Bengal_shapefile" / "Bay_of_Bengal.shp"  # optional, context/plotting only

ONI_CSV_PATH           = PROJECT_ROOT / "external_climate_indices" / "ONI_CPC_1950-2026.csv"     # Analysis 12 (ENSO)
DMI_CSV_PATH           = PROJECT_ROOT / "external_climate_indices" / "DMI_HadISST_1870-2026.csv" # Analysis 12 (IOD)
# Deliberately NOT used (see Analysis 12's markdown cell): no Indian-monsoon
# rainfall index is included -- one could not be sourced in a usable,
# operationally-current form. Analysis 12 uses ONI + DMI only.

# Analysis 14 (AOU) needs temperature & salinity, which this project's pipeline
# does not produce and which was NOT found anywhere under C:\All_data (checked
# recursively before writing this notebook). TS_CLIMATOLOGY_PATH is therefore a
# FALLBACK slot for a World Ocean Atlas 2023 (or similar) monthly T/S
# climatology file, regridded or interpolated onto this project's lat/lon/depth
# grid, that YOU supply -- point this at your own file when you have one.
# Analysis 14 checks for this file's existence and skips its computation with
# a clear warning (rather than fabricating T/S) if it is not found.
TS_CLIMATOLOGY_PATH    = PROJECT_ROOT / "external_TS_climatology" / "woa23_temperature_salinity.nc"

FIGURE_DIR = Path("./figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# 2. ANALYSIS 1 -- the 7 depths its moving-average time series are plotted at.
#    350 m is NOT one of the 26 native depth levels (which jump 300 -> 400 m
#    directly) -- it is produced by LINEAR INTERPOLATION along the depth axis
#    (see Analysis 1's markdown cell), not by snapping to the nearest level.
# ---------------------------------------------------------------------------
ANALYSIS1_DEPTHS_M = [100, 150, 200, 250, 300, 350, 400]

# ---------------------------------------------------------------------------
# 3. MOVING-AVERAGE WINDOW (months) -- used in Analysis 1 and anywhere else a
#    smoothed monthly series is shown.
# ---------------------------------------------------------------------------
MOVING_AVERAGE_MONTHS = 12

# ---------------------------------------------------------------------------
# 4. REGIONAL DECOMPOSITION -- used throughout (Analyses 1, 2, 7, 9, 10, 12).
#    3 latitudinal bands of EQUAL WIDTH across the ACTUAL latitude range found
#    in the data (computed from the files at runtime, not hardcoded degrees --
#    see build_region_masks() below), plus one basin-wide average.
# ---------------------------------------------------------------------------
N_LAT_BANDS = 3                       # Southern / Middle / Northern
REGION_NAMES = ["Southern", "Middle", "Northern"]   # low -> high latitude, matches N_LAT_BANDS
REGION_AVERAGING_METHOD = "cosine_weighted"   # "cosine_weighted" (recommended) or "simple"
# Why cosine-weighted: a 1x1 deg grid cell spans slightly less real east-west
# distance at higher latitude (its area shrinks by a factor of cos(latitude)),
# so a plain unweighted mean over grid cells very slightly over-weights the
# higher-latitude cells relative to their true area. At this basin's narrow
# latitude range (~5.5-24.5N) and 1 deg resolution the effect is small
# (cos(24.5) ~ 0.91 vs cos(5.5) ~ 0.995, an ~9% area difference top-to-bottom
# of the domain) but handling it properly costs nothing and is the physically
# correct thing to do, so it is the recommended default. Set to "simple" to
# revert to a plain unweighted mean instead.

# ---------------------------------------------------------------------------
# 5. OMZ / OXYGEN-ZONE THRESHOLDS (umol/kg) -- used in Analyses 9 and 10.
#    No project-specific thresholds were found anywhere in this pipeline's
#    config, so these are commonly-cited OMZ literature defaults (broadly
#    consistent with the oxic/hypoxic/suboxic/anoxic terminology used e.g. by
#    Paulmier & Ruiz-Pino, 2009, "Oxygen minimum zones (OMZs) in the modern
#    ocean", Prog. Oceanogr.). REPLACE these with your own published/preferred
#    cutoffs if you have them.
#
#    IMPORTANT DISCREPANCY, flagged rather than silently resolved: your
#    existing outputs/oxygen_zone_volumes/ table (from the companion yearly
#    notebook) uses DIFFERENT thresholds -- Hypoxic <=61, Severe Hypoxic <=22,
#    Suboxic/Near Anoxic <=10 umol/kg -- under similar-sounding but not
#    identical category names ("Hypoxic (Dead Zone)", "Severe Hypoxic
#    (Critical Dead Zone)", "Suboxic/Near Anoxic (Core Dead Zone/Desert)").
#    The two threshold sets are NOT the same numbers and will NOT produce the
#    same volumes for a "hypoxic" or "suboxic" category -- do not directly
#    compare OMZ volumes between this notebook and that one without picking
#    one consistent threshold set for both.
# ---------------------------------------------------------------------------
OMZ_ZONE_THRESHOLDS = {
    # name          : (lower_bound_umol_kg, upper_bound_umol_kg]  -- lower exclusive, upper inclusive
    "Oxic":          (90.0,  np.inf),
    "Hypoxic":       (22.0,  90.0),
    "Suboxic":       (2.0,   22.0),
    "Anoxic":        (-np.inf, 2.0),   # "functionally anoxic" in most OMZ literature
}
# The single threshold used to define "the OMZ" for Analyses 9 and 10 (volume/
# area/thickness, and the oxycline/upper-boundary depth) is the Hypoxic/
# Suboxic boundary, i.e. water below this value is counted as "inside the OMZ":
OMZ_BOUNDARY_THRESHOLD_UMOL_KG = OMZ_ZONE_THRESHOLDS["Hypoxic"][0]   # 22.0 umol/kg

# ---------------------------------------------------------------------------
# 6. RANDOM SEED(S) -- for every stochastic method (tensor decomposition
#    initialization in Analysis 4, Isolation Forest in Analysis 13).
# ---------------------------------------------------------------------------
RANDOM_SEED = 42

# ---------------------------------------------------------------------------
# 7. COLOR CONVENTIONS -- used consistently across every figure in this
#    notebook. Do not override these per-figure; change them here if you want
#    a different look everywhere at once.
# ---------------------------------------------------------------------------
CMAP_SEQUENTIAL = "viridis"    # absolute concentration fields (e.g. O2 maps, Hovmoller)
CMAP_DIVERGING  = "RdBu_r"     # trend / anomaly / composite fields, always zero-centered
SIGNIFICANCE_ALPHA = 0.05      # p < this value is "significant" everywhere in this notebook
STIPPLE_KW = dict(marker=".", s=3, color="black", alpha=0.6)   # significance stippling style

# ---------------------------------------------------------------------------
# 8. FIGURE FORMATTING -- 600 DPI PNG, Times New Roman 12pt everywhere, with a
#    graceful fallback chain if the exact font isn't installed on this machine.
# ---------------------------------------------------------------------------
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

FIGURE_DPI = 600
FONT_SIZE = 12
_available_fonts = set(f.name for f in fm.fontManager.ttflist)
if "Times New Roman" in _available_fonts:
    plt.rcParams["font.family"] = "Times New Roman"
    _font_in_use = "Times New Roman"
else:
    _fallback_chain = ["Times New Roman", "Liberation Serif", "DejaVu Serif"]
    plt.rcParams["font.family"] = "serif"
    plt.rcParams["font.serif"] = _fallback_chain
    _font_in_use = next((f for f in _fallback_chain if f in _available_fonts), "DejaVu Serif")
    warnings.warn(
        f"'Times New Roman' was not found on this machine -- using '{_font_in_use}' "
        "instead (font.serif fallback chain: Times New Roman -> Liberation Serif -> "
        "DejaVu Serif). Figures will still be produced, just in a different serif font. "
        "See the manual's Troubleshooting section to install Times New Roman if you "
        "need an exact match."
    )
plt.rcParams["font.size"] = FONT_SIZE

import matplotlib.patches as mpatches

def add_outer_frame(fig, pad_inches=0.02, linewidth=1.3, edgecolor="black"):
    """Single bounding box framing the whole figure, anchored to the figure's
    own (0,0)-(1,1) transFigure coordinates rather than a tight-bbox measured
    at one draw pass. This can never drift out of sync with the final saved
    layout (unlike a stale ax.get_tightbbox() snapshot combined with
    bbox_inches='tight'), because it doesn't depend on measuring content at
    all -- constrained_layout already guarantees padding between every
    subplot (and the figure's suptitle) and the figure edge, so a frame
    placed safely inside that guaranteed margin can't overlap content.

    Replaces the old per-cell pattern of `fig.canvas.draw()` +
    `ax.get_tightbbox(...)` + a Rectangle placed at the measured bbox. That
    pattern was the source of a systemic bug: `savefig()` used to pass
    `bbox_inches="tight"`, which triggers a SEPARATE internal re-render/
    re-layout pass at save time. That combines badly with
    `constrained_layout=True` (used on nearly every figure here) -- the final
    saved layout could end up very slightly different from the one measured
    by the earlier `fig.canvas.draw()` call, so the frame (frozen from that
    stale measurement) ended up misaligned with the final content, slicing
    through titles/labels in the saved PNG. Fixed by (a) this
    content-independent frame and (b) dropping `bbox_inches="tight"` from
    `savefig()`'s defaults -- constrained_layout already manages the
    figure's own padding, so it was redundant as well as the actual cause of
    the drift.

    `pad_inches` (not a bare figure-fraction) is the key design choice: every
    figure in this notebook uses `constrained_layout`'s DEFAULT `h_pad`/
    `w_pad`, which reserves a FIXED 0.04 inch margin between the figure's
    outer edge and its outermost content (including the suptitle) --
    confirmed empirically (measured suptitle-to-edge gaps of ~0.04in across
    figures of very different sizes). That 0.04in margin is constant in
    inches but corresponds to a DIFFERENT figure-fraction on every figure
    here (figsize ranges from ~8 to ~19 inches on a side), so a single
    hardcoded fraction (the first version of this function used pad=0.006)
    is guaranteed to be too tight for at least one figure -- which is
    exactly what happened: it sliced through Analysis 1's suptitle. Using
    `pad_inches` and converting to a fraction per-figure from
    `fig.get_size_inches()` keeps the frame at a constant, comfortably-inside
    distance (here, half of the reserved 0.04in) from the figure edge on
    EVERY figure, regardless of its size."""
    w_in, h_in = fig.get_size_inches()
    pad_x = pad_inches / w_in
    pad_y = pad_inches / h_in
    frame = mpatches.Rectangle(
        (pad_x, pad_y), 1 - 2 * pad_x, 1 - 2 * pad_y, transform=fig.transFigure,
        fill=False, edgecolor=edgecolor, linewidth=linewidth, zorder=1000, clip_on=False,
    )
    fig.add_artist(frame)
    return frame

def savefig(fig, filename, **kwargs):
    """Save a figure the standard way for this notebook: 600 DPI PNG, under
    FIGURE_DIR. `filename` should already include the analysis-number
    prefix, e.g. '01_longterm_moving_average.png'.

    NOTE: does NOT pass bbox_inches="tight" -- every figure in this notebook
    uses constrained_layout=True, which already manages the figure's own
    padding/spacing. bbox_inches="tight" triggers a second, separate
    re-layout pass at save time that can end up very slightly different from
    the constrained_layout pass, which is what caused the outer frame
    (see add_outer_frame() above) to drift out of alignment with the actual
    saved content in earlier versions of this notebook. Pass
    bbox_inches="tight" explicitly via **kwargs for one-off figures that
    genuinely need it (none currently do)."""
    out_path = FIGURE_DIR / filename
    save_kwargs = dict(dpi=FIGURE_DPI)
    save_kwargs.update(kwargs)
    fig.savefig(out_path, **save_kwargs)
    print(f"[figure] saved {out_path}  (font in use: {_font_in_use})")
    return out_path

np.random.seed(RANDOM_SEED)
print("Configuration loaded.")
print(f"  ENSEMBLE_DIR         = {ENSEMBLE_DIR}")
print(f"  Font in use          = {_font_in_use}")
print(f"  Region method        = {REGION_AVERAGING_METHOD} ({N_LAT_BANDS} bands: {REGION_NAMES})")
print(f"  OMZ boundary (umol/kg) = {OMZ_BOUNDARY_THRESHOLD_UMOL_KG} (Hypoxic/Suboxic threshold)")
print(f"  Random seed           = {RANDOM_SEED}")
print(f"  Figures               -> {FIGURE_DIR.resolve()}  @ {FIGURE_DPI} DPI")


## 0.1 Master data loader

Loads all 216 monthly ensemble files **once** into a single in-memory `xarray.Dataset` (`ds_ensemble`, dims `time, depth, lat, lon`) and reuses it for every analysis below — no analysis re-reads from disk.

**A data-format detail worth knowing**: each individual `BoB_Oxygen_Ensemble_YYYY_MM.nc` file has no internal `time` coordinate at all — it's a single month's snapshot with only `(depth, lat, lon)`. The month is encoded only in the *filename*. So the loader below uses `xarray.open_mfdataset` with a small `preprocess` function that reads the year/month back out of each file's path and attaches it as a proper `time` coordinate *before* the files are combined — this is what lets everything downstream (moving averages, FFT, wavelet, Mann-Kendall, etc.) treat the data as a normal monthly time series.

The true date range actually present on disk is auto-detected from the files found (not hardcoded), and printed below — **the original project plan assumed a 2005–2024 record, but the files actually on disk only go through 2022-12; this cell prints the real range so that discrepancy is visible up front rather than silently assumed.**


In [ ]:
import re
import xarray as xr
import pandas as pd

_FNAME_RE = re.compile(r"BoB_Oxygen_Ensemble_(\d{4})_(\d{2})\.nc$")

def _attach_time_from_filename(ds: xr.Dataset) -> xr.Dataset:
    """xr.open_mfdataset preprocess hook: each ensemble file has no internal
    time coordinate, only (depth, lat, lon) -- the year/month live only in the
    filename. This reads them back out and adds a proper 'time' dimension/
    coordinate (first-of-month timestamp) before the files get combined."""
    path = ds.encoding.get("source", "")
    m = _FNAME_RE.search(path)
    if not m:
        raise ValueError(f"Could not parse year/month out of ensemble filename: {path!r}")
    year, month = int(m.group(1)), int(m.group(2))
    t = pd.Timestamp(year=year, month=month, day=1)
    return ds.expand_dims(time=[t])

def load_ensemble_dataset(ensemble_dir=ENSEMBLE_DIR) -> xr.Dataset:
    """Loads every BoB_Oxygen_Ensemble_YYYY_MM.nc file in ensemble_dir into one
    combined (time, depth, lat, lon) xarray.Dataset, sorted chronologically.
    Called once; the result is cached in the module-level ds_ensemble below."""
    files = sorted(ensemble_dir.glob("BoB_Oxygen_Ensemble_*.nc"))
    if not files:
        raise FileNotFoundError(
            f"No BoB_Oxygen_Ensemble_*.nc files found in {ensemble_dir} -- check "
            "PROJECT_ROOT / ENSEMBLE_DIR in the configuration cell above."
        )
    ds = xr.open_mfdataset(
        files, preprocess=_attach_time_from_filename, combine="by_coords",
        engine="netcdf4", chunks=None,
    )
    ds = ds.sortby("time")
    return ds

ds_ensemble = load_ensemble_dataset()
ds_ensemble.load()  # small dataset (216 files x ~170 KB) -- pull fully into memory once

DETECTED_START = pd.Timestamp(ds_ensemble.time.values.min())
DETECTED_END = pd.Timestamp(ds_ensemble.time.values.max())
N_MONTHS_FOUND = ds_ensemble.sizes["time"]
N_MONTHS_EXPECTED = (DETECTED_END.year - DETECTED_START.year) * 12 + (DETECTED_END.month - DETECTED_START.month) + 1

print("=" * 78)
print(f"ENSEMBLE DATE RANGE DETECTED FROM FILES ON DISK: "
      f"{DETECTED_START:%Y-%m} through {DETECTED_END:%Y-%m}  ({N_MONTHS_FOUND} monthly files)")
if DETECTED_END.year < 2024:
    print(f"NOTE: this notebook's original brief assumed the record ran through 2024 -- "
          f"the files actually on disk only go through {DETECTED_END:%Y-%m}. All 14 "
          "analyses below use the range detected here, not any hardcoded assumption.")
if N_MONTHS_FOUND != N_MONTHS_EXPECTED:
    print(f"WARNING: {N_MONTHS_EXPECTED - N_MONTHS_FOUND} month(s) are missing from an "
          f"otherwise-continuous {DETECTED_START:%Y-%m}-{DETECTED_END:%Y-%m} span -- "
          "some analyses below (FFT, wavelet, EOF, Mann-Kendall) assume a regular "
          "monthly series and may need this gap handled explicitly if it's large.")
else:
    print("No missing months -- a complete, gap-free monthly record.")
print("=" * 78)

O2_VAR = "o2_umol_kg"
N_PRODUCTS_VAR = "n_products_averaged"
DEPTH_LEVELS_M = ds_ensemble["depth"].values.astype(float)
LAT_VALUES = ds_ensemble["lat"].values.astype(float)
LON_VALUES = ds_ensemble["lon"].values.astype(float)
print(f"Grid: {len(DEPTH_LEVELS_M)} depth levels ({DEPTH_LEVELS_M.min():.0f}-{DEPTH_LEVELS_M.max():.0f} m), "
      f"{len(LAT_VALUES)} lat x {len(LON_VALUES)} lon cells "
      f"({LAT_VALUES.min():.1f}-{LAT_VALUES.max():.1f} N, {LON_VALUES.min():.1f}-{LON_VALUES.max():.1f} E)")


## 0.2 Regional decomposition (3 equal-latitude bands + basin)

Splits the basin into 3 latitude bands of **equal width across the actual latitude range found in the data** — `lat.min()`/`lat.max()` are read from the loaded dataset, not hardcoded, then divided into 3 equal parts:

- **Southern** — lowest third of the latitude range
- **Middle** — middle third
- **Northern** — highest third

Each region's series is the horizontal (lat/lon) mean of every valid (non-NaN) grid cell inside that band, at every depth/time — using a **cosine-latitude-weighted mean** by default (see `REGION_AVERAGING_METHOD` in the config cell for why, and how to switch to a plain unweighted mean). A **Basin** series (mean over the whole domain, same weighting) is also produced. Change `N_LAT_BANDS`/`REGION_NAMES` in the config cell if you want a different number of bands — everything below reads those variables rather than assuming 3.


In [ ]:
import numpy as np
import xarray as xr

def build_region_masks(lat_values=LAT_VALUES, n_bands=N_LAT_BANDS, names=REGION_NAMES) -> dict:
    """Returns {region_name: boolean_mask_over_lat} for n_bands equal-width
    latitude bands spanning lat_values.min() .. lat_values.max() (computed
    from the actual data, not hardcoded degrees). Band edges are inclusive on
    the lower bound and inclusive on the upper bound for the last band only,
    so every lat value falls in exactly one band."""
    lat_min, lat_max = float(lat_values.min()), float(lat_values.max())
    edges = np.linspace(lat_min, lat_max, n_bands + 1)
    masks = {}
    for i, name in enumerate(names):
        lo, hi = edges[i], edges[i + 1]
        if i == n_bands - 1:
            m = (lat_values >= lo) & (lat_values <= hi)
        else:
            m = (lat_values >= lo) & (lat_values < hi)
        masks[name] = m
        print(f"  {name:10s} band: {lo:.2f} to {hi:.2f} N  ({m.sum()} of {len(lat_values)} lat rows)")
    return masks, edges

print("Latitude band edges (equal-width thirds of the detected range):")
REGION_LAT_MASKS, REGION_LAT_EDGES = build_region_masks()

def regional_mean(da: xr.DataArray, method=REGION_AVERAGING_METHOD) -> xr.Dataset:
    """Collapses a DataArray's lat/lon dims into regional means: one DataArray
    per entry in REGION_NAMES plus 'Basin', combined along a new 'region' dim.
    method: 'cosine_weighted' (area-correct, recommended) or 'simple'
    (unweighted mean over grid cells)."""
    if method == "cosine_weighted":
        weights = np.cos(np.deg2rad(da["lat"]))
    elif method == "simple":
        weights = xr.ones_like(da["lat"], dtype=float)
    else:
        raise ValueError(f"Unknown REGION_AVERAGING_METHOD: {method!r}")

    out = {}
    for name, latmask in REGION_LAT_MASKS.items():
        sub = da.isel(lat=latmask)
        w = weights.isel(lat=latmask)
        out[name] = sub.weighted(w.fillna(0)).mean(dim=("lat", "lon"), skipna=True)
    out["Basin"] = da.weighted(weights.fillna(0)).mean(dim=("lat", "lon"), skipna=True)

    combined = xr.concat(
        [out[n] for n in REGION_NAMES + ["Basin"]],
        dim=xr.DataArray(REGION_NAMES + ["Basin"], dims="region", name="region"),
    )
    return combined

# Quick sanity check: regional + basin monthly-mean series at the shallowest
# depth level, just to confirm the machinery runs before the real analyses use it.
_sanity = regional_mean(ds_ensemble[O2_VAR].isel(depth=0))
print("\nSanity check -- regional_mean() output shape:", dict(_sanity.sizes))


## 1. Long-term temporal variation (12-month moving average)

**What this shows**: monthly-mean dissolved oxygen at 7 fixed depths (100, 150, 200, 250, 300, 350, 400 m — set in `ANALYSIS1_DEPTHS_M` above), for each of the 3 latitude regions plus the whole-basin average, smoothed with a 12-month centered moving average to make the long-term trend and multi-year swings visible underneath the strong seasonal cycle.

**On the 350 m line**: the native depth levels jump directly from 300 m to 400 m (there is no 350 m sample in the data). Rather than silently snapping 350 m to whichever of those two is "closer", the cell below uses `xarray`'s linear interpolation *along the depth axis* (`.interp(depth=...)`) for all 7 requested depths at once — for the 6 depths that already exist natively (100, 150, 200, 250, 300, 400 m) this returns the exact native value (interpolating exactly at a data point is a no-op), and only the 350 m line is a genuine interpolated estimate, linearly between the real 300 m and 400 m values at each grid cell/month.

**Reading the figure**: each of the 4 stacked panels is one region (Southern → Middle → Northern → Basin, top to bottom, sharing the same time axis). Within a panel there are 7 lines, one per depth, colored from light/yellow (shallow, 100 m) to dark/purple (deep, 400 m) using the same `viridis` scale used for the absolute-value maps elsewhere in this notebook. The **faint, thin** version of each line (alpha ≈ 0.3) is the raw monthly value; the **bold** version is the 12-month centered moving average — the thing to actually read for a trend. A single shared legend (depth colors, plus a raw-vs-smoothed key) applies to all 4 panels, and all 4 panels sit inside one outer frame as a single figure.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import numpy as np

# --- 1. Extract the 7 config depths, interpolating linearly along depth ----
# (a no-op at the 6 native depths, a genuine linear interpolation at 350 m
# only -- see markdown above).
da_depths_1 = ds_ensemble[O2_VAR].interp(depth=ANALYSIS1_DEPTHS_M, method="linear")
da_depths_1 = da_depths_1.assign_coords(depth=("depth", ANALYSIS1_DEPTHS_M))

# --- 2. Collapse to regional + basin series, still keeping (region, time, depth)
regional_1 = regional_mean(da_depths_1)          # dims: region, time, depth
raw_1 = regional_1                                 # raw monthly
smoothed_1 = regional_1.rolling(
    time=MOVING_AVERAGE_MONTHS, center=True, min_periods=max(1, MOVING_AVERAGE_MONTHS // 2)
).mean()

# --- 3. Plot: 4 rows x 1 col, shared x-axis, one panel per region + basin ---
panel_order_1 = REGION_NAMES + ["Basin"]           # Southern, Middle, Northern, Basin
depth_colors_1 = plt.get_cmap(CMAP_SEQUENTIAL)(np.linspace(0.05, 0.95, len(ANALYSIS1_DEPTHS_M)))

fig1, axes1 = plt.subplots(
    nrows=len(panel_order_1), ncols=1, figsize=(11, 13), sharex=True, constrained_layout=True
)

for ax, region_name in zip(axes1, panel_order_1):
    for depth_m, color in zip(ANALYSIS1_DEPTHS_M, depth_colors_1):
        raw_line = raw_1.sel(region=region_name, depth=depth_m)
        smooth_line = smoothed_1.sel(region=region_name, depth=depth_m)
        ax.plot(raw_line["time"], raw_line, color=color, linewidth=0.7, alpha=0.30, zorder=2)
        ax.plot(smooth_line["time"], smooth_line, color=color, linewidth=1.8, alpha=1.0, zorder=3,
                 label=f"{depth_m:.0f} m")
    ax.set_ylabel("O$_2$ (\u00b5mol/kg)")
    ax.set_title(region_name, loc="left", fontsize=11, fontweight="bold")
    ax.grid(alpha=0.25, linewidth=0.5)

axes1[-1].set_xlabel("Time")

# --- 4. One shared legend for the whole figure (depth colors + raw/smoothed key)
depth_handles_1 = [
    mlines.Line2D([], [], color=c, linewidth=1.8, label=f"{d:.0f} m")
    for d, c in zip(ANALYSIS1_DEPTHS_M, depth_colors_1)
]
style_handles_1 = [
    mlines.Line2D([], [], color="black", linewidth=1.8, alpha=1.0, label="12-mo moving average"),
    mlines.Line2D([], [], color="black", linewidth=0.7, alpha=0.30, label="raw monthly"),
]
fig1.legend(
    handles=depth_handles_1 + style_handles_1,
    loc="outside right upper", title="Depth", frameon=True, fontsize=9, title_fontsize=10,
)

fig1.suptitle("Bay of Bengal — Long-Term Dissolved Oxygen, 7 Depths, 12-Month Moving Average",
              fontsize=13, fontweight="bold")

# --- 5. Single outer bounding box around all 4 panels -----------------------
add_outer_frame(fig1)

savefig(fig1, "01_longterm_moving_average.png")
plt.show()


## 2. Time-depth Hovmöller plots

**What this shows**: a "Hovmöller diagram" for each region + the basin — time along the x-axis, depth along the y-axis, and color showing the horizontally-averaged oxygen concentration at that depth and month. Unlike Analysis 1, this uses the **full native 26-level depth axis** (10–1995 m), not just the 7 selected depths, so it shows the whole vertical structure at once: the shallow oxycline, the oxygen-minimum core, and the slow recovery at depth.

**Reading the figure**: depth increases downward (0 m at the top, consistent with the usual oceanographic convention), time runs left to right, and color follows the same `viridis` scale (dark = low oxygen, bright yellow = high oxygen) used everywhere in this notebook for absolute concentration fields. A shallow persistent oxygen-minimum "core" appears as a dark horizontal band; if that band widens (top edge moving upward and/or bottom edge moving downward) or darkens over the 2005–2022 record, that is a visual precursor to the trend results quantified later in Analyses 7 and 9. All 4 panels (Southern, Middle, Northern, Basin) share one colorbar so their colors are directly comparable to each other.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# --- 1. Regional + basin mean at EVERY native depth level (not just the 7
# Analysis-1 depths) -- dims: region, time, depth
regional_full_2 = regional_mean(ds_ensemble[O2_VAR])

panel_order_2 = [REGION_NAMES[0], REGION_NAMES[1], REGION_NAMES[2], "Basin"]  # Southern, Middle, Northern, Basin
panel_layout_2 = [(0, 0), (0, 1), (1, 0), (1, 1)]  # 2x2 positions for the order above

vmin_2 = float(regional_full_2.min())
vmax_2 = float(regional_full_2.max())
levels_2 = np.linspace(vmin_2, vmax_2, 21)

fig2, axes2 = plt.subplots(2, 2, figsize=(13, 9), sharex=True, sharey=True, constrained_layout=True)

_cs_2 = None
for region_name, (r, c) in zip(panel_order_2, panel_layout_2):
    ax = axes2[r, c]
    da_rc = regional_full_2.sel(region=region_name).transpose("depth", "time")
    _cs_2 = ax.contourf(
        da_rc["time"].values, da_rc["depth"].values, da_rc.values,
        levels=levels_2, cmap=CMAP_SEQUENTIAL, extend="both",
    )
    ax.set_title(region_name, loc="left", fontsize=11, fontweight="bold")
    if c == 0:
        ax.set_ylabel("Depth (m)")
    if r == 1:
        ax.set_xlabel("Time")

# NOTE: axes2 uses sharey=True, so all 4 panels share ONE underlying y-limits
# object -- invert_yaxis() must be called exactly ONCE (not once per panel
# inside the loop above), otherwise repeated calls toggle the shared limits
# back and forth and an even number of calls (4, one per panel) cancels out
# to no inversion at all.
axes2[0, 0].invert_yaxis()   # 0 m at the top, applies to all 4 shared-y panels

fig2.suptitle("Bay of Bengal — Time-Depth Hovm\u00f6ller: Dissolved Oxygen (\u00b5mol/kg)",
              fontsize=13, fontweight="bold")

# --- 2. One shared colorbar for all 4 panels --------------------------------
cbar_2 = fig2.colorbar(_cs_2, ax=axes2.ravel().tolist(), shrink=0.9, pad=0.02)
cbar_2.set_label("O$_2$ (\u00b5mol/kg)")

# --- 3. Single outer bounding box around all 4 panels -----------------------
add_outer_frame(fig2)

savefig(fig2, "02_hovmoller_time_depth.png")
plt.show()


## 3. Three-dimensional EOF analysis

**What this shows**: Empirical Orthogonal Function (EOF) analysis decomposes the full `(time, depth, lat, lon)` oxygen field into a small number of recurring 3-D spatial-depth "patterns" (the EOFs / spatial modes) and the time series that says how strongly each pattern is expressed each month (the PCs / principal components). It is a way of asking "what are the two or three dominant, independent modes of variability in this whole 4-D dataset, and how much of the total variance does each one explain?" — done here with the `eofs` package's `xarray` interface, which natively handles the multi-dimensional `(depth, lat, lon)` spatial structure (it is flattened internally into one composite spatial axis, weighted, decomposed, and un-flattened back to `(mode, depth, lat, lon)` for plotting, so no manual reshaping is needed).

**Weighting and missing data**: each grid cell is weighted by `sqrt(cos(latitude))` before decomposition (the standard EOF area-weighting, consistent with the cosine-weighted regional averaging used throughout this notebook) so that cells don't get spurious extra influence just because they sit at a particular latitude. `eofs` requires a *time-invariant* missing-value mask (a cell is either always valid or always missing) — the cell below checks this and, if any cell is missing in only *some* months (which would break that assumption), conservatively drops it from the EOF entirely (treats it as always-missing) rather than filling in a fabricated value, and reports how many cells (if any) were affected.

**Reading the figure**: three rows, one per leading mode (Mode 1 = most variance explained, down to Mode 3). Left column: the mode's horizontal spatial pattern at 150 m (a representative OMZ-core depth), in the diverging `RdBu_r` scale, centered at zero — a region colored strongly red vs. strongly blue means those two areas move in *opposite* directions when this mode is active. Right column: the same mode's root-mean-square amplitude by depth (how strongly this mode is expressed at each depth level, regardless of geographic sign), y-axis inverted so 0 m is at the top. The bottom row shows all 3 PCs' time series together (one line each) and a scree plot of the percent variance explained by the first several modes, so you can judge whether the record is dominated by 1-2 big modes or whether variance is spread more evenly.

**What the modes physically represent**: the code below does not just show the patterns — it also *checks* each of the 3 leading PCs against a plain seasonal-cycle proxy (sine/cosine of month-of-year) and a plain linear-trend proxy, reports the correlation with each, and prints a plain-language interpretation (e.g. "Mode 1 correlates strongly with the seasonal cycle and weakly with the long-term trend, so it most likely represents monsoon-driven seasonal variability rather than the deoxygenation trend quantified in Analysis 7") rather than asserting an interpretation without evidence.


In [ ]:
!pip install xarray netCDF4 numpy scipy pandas matplotlib pymannkendall ruptures tensorly eofs pywt scikit-gstat esda libpysal gsw scikit-learn pycwt

In [ ]:
!pip install eofs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
from eofs.xarray import Eof

EOF_N_MODES = 3
EOF_REPRESENTATIVE_DEPTH_M = 150.0   # OMZ-core depth used for the spatial-map panels

# --- 1. Build the field to decompose, enforcing a time-invariant NaN mask ---
da_3 = ds_ensemble[O2_VAR].transpose("time", "depth", "lat", "lon")
_n_missing_sometimes = int(((da_3.isnull().any("time")) & (~da_3.isnull().all("time"))).sum())
if _n_missing_sometimes > 0:
    print(f"NOTE: {_n_missing_sometimes} grid cell(s) are missing in only SOME months -- "
          "eofs requires a time-invariant mask, so these cells are conservatively treated "
          "as always-missing (excluded from the EOF) rather than filled in.")
    always_valid_3 = da_3.notnull().all("time")
    da_3 = da_3.where(always_valid_3)
else:
    print("Missing-value mask is time-invariant (every cell is either always valid or "
          "always missing) -- no cells needed to be dropped for the EOF solver.")

# --- 2. Cosine-latitude area weights, broadcast to the (depth, lat, lon) spatial shape
_coslat_3 = np.sqrt(np.clip(np.cos(np.deg2rad(LAT_VALUES)), 0, None))
weights_3 = np.broadcast_to(_coslat_3[None, :, None], (len(DEPTH_LEVELS_M), len(LAT_VALUES), len(LON_VALUES))).astype(float)

solver_3 = Eof(da_3, weights=weights_3, center=True)
eofs_spatial_3 = solver_3.eofs(neofs=EOF_N_MODES)              # dims: mode, depth, lat, lon
pcs_3 = solver_3.pcs(npcs=EOF_N_MODES, pcscaling=1)             # dims: time, mode (unit variance)
variance_fraction_3 = solver_3.varianceFraction(neigs=10)       # dims: mode (first 10, for the scree plot)

print(f"Leading {EOF_N_MODES} modes explain "
      f"{float(variance_fraction_3.isel(mode=slice(0, EOF_N_MODES)).sum()) * 100:.1f}% of total variance.")
for i in range(EOF_N_MODES):
    print(f"  Mode {i+1}: {float(variance_fraction_3.isel(mode=i)) * 100:.1f}% of variance")

# --- 3. Data-driven check: does each PC look seasonal, trend-like, or neither? ---
_time_idx_3 = pd.DatetimeIndex(da_3["time"].values)
_month_num_3 = _time_idx_3.month.values
_season_sin_3 = np.sin(2 * np.pi * _month_num_3 / 12.0)
_season_cos_3 = np.cos(2 * np.pi * _month_num_3 / 12.0)
_trend_proxy_3 = np.arange(len(_time_idx_3), dtype=float)

_mode_interpretations_3 = []
for i in range(EOF_N_MODES):
    pc_i = pcs_3.isel(mode=i).values
    r_season = float(np.sqrt(
        np.corrcoef(pc_i, _season_sin_3)[0, 1] ** 2 + np.corrcoef(pc_i, _season_cos_3)[0, 1] ** 2
    ) / np.sqrt(2))  # combined seasonal correlation strength, roughly 0-1
    r_trend = float(np.corrcoef(pc_i, _trend_proxy_3)[0, 1])
    if abs(r_season) > 0.5 and abs(r_season) > abs(r_trend) * 1.5:
        verdict = "most likely represents monsoon-driven SEASONAL variability"
    elif abs(r_trend) > 0.4 and abs(r_trend) > r_season * 1.5:
        verdict = "most likely represents the long-term DEOXYGENATION TREND (compare to Analysis 7)"
    else:
        verdict = "does not cleanly separate into a seasonal or trend signal -- likely reflects interannual variability (compare to Analyses 5, 6, 12)"
    _mode_interpretations_3.append((i + 1, r_season, r_trend, verdict))
    print(f"Mode {i+1}: seasonal-correlation-strength={r_season:.2f}, linear-trend-correlation={r_trend:+.2f}  ->  {verdict}")

# --- 4. Composite figure: per-mode spatial map + depth profile, PC series, scree ---
fig3 = plt.figure(figsize=(13.0, 13.0), constrained_layout=True)
gs3 = fig3.add_gridspec(4, 2, height_ratios=[1, 1, 1, 0.9])

_map_vmax_3 = float(np.nanmax(np.abs(eofs_spatial_3.sel(depth=EOF_REPRESENTATIVE_DEPTH_M).values)))
_norm_3 = TwoSlopeNorm(vcenter=0.0, vmin=-_map_vmax_3, vmax=_map_vmax_3)

_map_axes_3 = []
_prof_axes_3 = []
for i in range(EOF_N_MODES):
    ax_map = fig3.add_subplot(gs3[i, 0])
    pattern_i = eofs_spatial_3.isel(mode=i).sel(depth=EOF_REPRESENTATIVE_DEPTH_M)
    cs = ax_map.pcolormesh(LON_VALUES, LAT_VALUES, pattern_i.values, cmap=CMAP_DIVERGING,
                             norm=_norm_3, shading="nearest")
    ax_map.set_title(f"Mode {i+1} spatial pattern @ {EOF_REPRESENTATIVE_DEPTH_M:.0f} m "
                      f"({float(variance_fraction_3.isel(mode=i))*100:.1f}% var.)",
                      loc="left", fontsize=10, fontweight="bold")
    ax_map.set_ylabel("Latitude")
    if i == EOF_N_MODES - 1:
        ax_map.set_xlabel("Longitude")
    _map_axes_3.append(ax_map)

    ax_prof = fig3.add_subplot(gs3[i, 1])
    rms_by_depth = np.sqrt((eofs_spatial_3.isel(mode=i) ** 2).mean(dim=("lat", "lon")))
    ax_prof.plot(rms_by_depth.values, rms_by_depth["depth"].values, color="black", linewidth=1.6)
    ax_prof.invert_yaxis()
    ax_prof.set_title(f"Mode {i+1} vertical (depth) amplitude", loc="left", fontsize=10, fontweight="bold")
    ax_prof.set_ylabel("Depth (m)")
    if i == EOF_N_MODES - 1:
        ax_prof.set_xlabel("RMS EOF loading")
    _prof_axes_3.append(ax_prof)

cbar_3 = fig3.colorbar(cs, ax=_map_axes_3, shrink=0.85, pad=0.02)
cbar_3.set_label("EOF loading (dimensionless, sign-relative)")

ax_pc_3 = fig3.add_subplot(gs3[3, 0])
for i in range(EOF_N_MODES):
    ax_pc_3.plot(pcs_3["time"].values, pcs_3.isel(mode=i).values, linewidth=1.2, label=f"PC{i+1}")
ax_pc_3.axhline(0, color="grey", linewidth=0.6)
ax_pc_3.set_title("Principal component (PC) time series", loc="left", fontsize=10, fontweight="bold")
ax_pc_3.set_xlabel("Time")
ax_pc_3.set_ylabel("PC amplitude (unit variance)")
ax_pc_3.legend(fontsize=9)

ax_scree_3 = fig3.add_subplot(gs3[3, 1])
_n_scree = variance_fraction_3.sizes["mode"]
ax_scree_3.plot(np.arange(1, _n_scree + 1), variance_fraction_3.values * 100, marker="o", color="black")
ax_scree_3.set_title("Scree plot", loc="left", fontsize=10, fontweight="bold")
ax_scree_3.set_xlabel("Mode number")
ax_scree_3.set_ylabel("% variance explained")
ax_scree_3.set_xticks(np.arange(1, _n_scree + 1))

fig3.suptitle("Bay of Bengal — 3-D EOF Analysis (space \u00d7 depth \u00d7 time)",
              fontsize=13, fontweight="bold")

add_outer_frame(fig3)

savefig(fig3, "03_eof_analysis.png")
plt.show()


## 4. Tensor decomposition (CP / PARAFAC)

**What this shows, and how it differs from Analysis 3's EOF**: EOF analysis (Analysis 3) works by flattening `depth`, `lat`, and `lon` together into one long "space" axis and finding patterns in the resulting 2-D `(time, space)` matrix — it never distinguishes *which* of depth, lat, or lon is driving a pattern, only the combined spatial-depth shape. A **tensor (CP / PARAFAC) decomposition** instead keeps all 4 dimensions — `time`, `lat`, `lon`, `depth` — fully separate, and represents the field as a sum of a small number of *rank-1 components*, where each component is one vector per dimension (a time series, a latitude profile, a longitude profile, and a depth profile) multiplied together. This preserves the genuine multi-way structure of the data instead of collapsing it, at the cost of a more restrictive model (each component must be perfectly "separable" across dimensions, which EOF's flattened space does not require). Comparing the two is informative: where they agree, that's a robust signal; where they disagree, it usually means the true pattern doesn't factor cleanly across lat/lon/depth independently.

**Missing data**: the always-invalid (land / outside-boundary) grid cells are mean-centered like everywhere else in this notebook, then filled with `0` — which, after mean-centering, represents "exactly at the long-term mean" and therefore contributes no artificial signal to the decomposition. This is different from EOF's approach (which excludes those cells from the decomposition entirely) and is disclosed here for transparency.

**Choosing the rank**: CP decomposition needs you to pick a rank (number of components) in advance. The cell below fits ranks 1 through 6 and plots the reconstruction error (fraction of the tensor's variance NOT explained) against rank; the rank actually used for the factor plots below is chosen automatically as the smallest rank where adding one more component stops helping much (the "elbow"), capped at a modest maximum of 5 for interpretability — printed explicitly so the choice is not hidden.

**Reading the factor plots**: each row is one CP component; columns are that component's time factor (a line), its spatial factor (the outer product of its latitude and longitude vectors, shown as a map — this is the natural way to visualize a CP component's spatial shape, since latitude and longitude are *separate* factors in this decomposition, unlike EOF's combined spatial pattern), and its depth factor (a profile, 0 m at the top). None of the 4 individual factor vectors are meaningful in isolation or in a fixed sign/scale — only their product is — so read each row as one indivisible pattern.


In [ ]:
!pip install tensorly

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
import tensorly as tl
from tensorly.decomposition import parafac

tl.set_backend("numpy")

CP_RANKS_TO_TRY = [1, 2, 3, 4, 5, 6]
CP_RANK_MAX_FOR_ELBOW = 5   # keep the CP-decomposition rank modest (3-5) for interpretability
CP_ELBOW_IMPROVEMENT_THRESHOLD = 0.03   # stop once one more component improves fit by < 3%

# --- 1. Build the 4-way tensor (time, lat, lon, depth), mean-centered, NaNs -> 0
da_4 = ds_ensemble[O2_VAR].transpose("time", "lat", "lon", "depth")
_field_mean_4 = float(da_4.mean(skipna=True))
da_4_centered = da_4 - _field_mean_4
_n_invalid_4 = int(da_4_centered.isnull().sum())
_always_invalid_mask_4 = da_4.isnull().all("time")
_n_always_invalid_cells_4 = int(_always_invalid_mask_4.sum())          # (lat,lon,depth) triples
_n_always_invalid_entries_4 = _n_always_invalid_cells_4 * da_4.sizes["time"]
_n_sometimes_invalid_entries_4 = _n_invalid_4 - _n_always_invalid_entries_4
tensor_4 = da_4_centered.fillna(0.0).values.astype(float)
print(f"Tensor shape (time, lat, lon, depth) = {tensor_4.shape}")
print(f"  {_n_always_invalid_entries_4} of {tensor_4.size} entries are ALWAYS invalid "
      f"(land / outside the basin boundary, {_n_always_invalid_cells_4} distinct (lat,lon,depth) "
      "cells, invalid at every one of the 216 months).")
if _n_sometimes_invalid_entries_4 > 0:
    print(f"  {_n_sometimes_invalid_entries_4} additional entries are invalid in only SOME "
          "months at otherwise-valid ocean cells (e.g. months where n_products_averaged=0 for "
          "that cell) -- these are real data gaps, not land.")
print("  All of the above are mean-centered-and-zero-filled as documented above (0 after "
      "centering = 'exactly at the long-term mean', i.e. no artificial signal injected).")

_tensor_norm_4 = float(np.linalg.norm(tensor_4))

# --- 2. Reconstruction-error-vs-rank curve, to justify the chosen rank -----
np.random.seed(RANDOM_SEED)
_errors_4 = []
_factors_by_rank_4 = {}
for r in CP_RANKS_TO_TRY:
    weights_r, factors_r = parafac(tensor_4, rank=r, n_iter_max=200, tol=1e-7,
                                     random_state=RANDOM_SEED, normalize_factors=False)
    recon_r = tl.cp_to_tensor((weights_r, factors_r))
    err_r = float(np.linalg.norm(tensor_4 - recon_r) / _tensor_norm_4)
    _errors_4.append(err_r)
    _factors_by_rank_4[r] = (weights_r, factors_r)
    print(f"  rank {r}: relative reconstruction error = {err_r:.4f}")

# Elbow: smallest rank (<= CP_RANK_MAX_FOR_ELBOW) where the NEXT rank's improvement is small.
# Default (if no clear elbow appears within the cap) is the cap itself, CP_RANK_MAX_FOR_ELBOW --
# NEVER the largest rank merely tried for the curve, so the "modest rank" guidance is always honored.
CP_RANK_CHOSEN = CP_RANK_MAX_FOR_ELBOW
_elbow_found_4 = False
for i in range(len(CP_RANKS_TO_TRY) - 1):
    r = CP_RANKS_TO_TRY[i]
    if r > CP_RANK_MAX_FOR_ELBOW:
        break
    improvement = (_errors_4[i] - _errors_4[i + 1]) / _errors_4[i]
    if improvement < CP_ELBOW_IMPROVEMENT_THRESHOLD:
        CP_RANK_CHOSEN = r
        _elbow_found_4 = True
        break
if _elbow_found_4:
    print(f"Chosen CP rank = {CP_RANK_CHOSEN} (clear elbow in the reconstruction-error-vs-rank "
          f"curve: adding one more component improved the fit by less than "
          f"{CP_ELBOW_IMPROVEMENT_THRESHOLD*100:.0f}%).")
else:
    print(f"Chosen CP rank = {CP_RANK_CHOSEN}: no clear elbow appeared within the tested ranks up "
          f"to the cap (reconstruction error kept improving by >= {CP_ELBOW_IMPROVEMENT_THRESHOLD*100:.0f}% "
          f"per added component all the way to rank {CP_RANK_MAX_FOR_ELBOW}) -- using the cap itself "
          "per the 'modest rank (3-5)' guidance rather than an untested, larger rank.")

weights_4, factors_4 = _factors_by_rank_4[CP_RANK_CHOSEN]
factor_time_4, factor_lat_4, factor_lon_4, factor_depth_4 = factors_4   # order matches tensor axes

# --- 3. Composite figure: rank-selection curve + one row of factors per component
fig4 = plt.figure(figsize=(13.0, 2.0 * CP_RANK_CHOSEN + 2.0), constrained_layout=True)
gs4 = fig4.add_gridspec(CP_RANK_CHOSEN + 1, 3, height_ratios=[0.8] + [1] * CP_RANK_CHOSEN)

ax_err_4 = fig4.add_subplot(gs4[0, :])
ax_err_4.plot(CP_RANKS_TO_TRY, _errors_4, marker="o", color="black")
ax_err_4.axvline(CP_RANK_CHOSEN, color="firebrick", linestyle="--", linewidth=1.2,
                   label=f"chosen rank = {CP_RANK_CHOSEN}")
ax_err_4.set_xlabel("CP rank")
ax_err_4.set_ylabel("Relative reconstruction error")
ax_err_4.set_title("Reconstruction error vs. rank (elbow used to choose the rank below)",
                    loc="left", fontsize=10, fontweight="bold")
ax_err_4.legend(fontsize=9)

_factor_axes_4 = [ax_err_4]
for comp in range(CP_RANK_CHOSEN):
    ax_t = fig4.add_subplot(gs4[comp + 1, 0])
    ax_t.plot(da_4["time"].values, factor_time_4[:, comp], color="black", linewidth=1.2)
    ax_t.axhline(0, color="grey", linewidth=0.5)
    ax_t.set_title(f"Component {comp+1}: time factor", loc="left", fontsize=9, fontweight="bold")
    if comp == CP_RANK_CHOSEN - 1:
        ax_t.set_xlabel("Time")

    ax_m = fig4.add_subplot(gs4[comp + 1, 1])
    spatial_pattern = np.outer(factor_lat_4[:, comp], factor_lon_4[:, comp])
    vmax_m = float(np.max(np.abs(spatial_pattern))) or 1.0
    cs4 = ax_m.pcolormesh(LON_VALUES, LAT_VALUES, spatial_pattern, cmap=CMAP_DIVERGING,
                            norm=TwoSlopeNorm(vcenter=0.0, vmin=-vmax_m, vmax=vmax_m), shading="nearest")
    ax_m.set_title(f"Component {comp+1}: lat \u00d7 lon", loc="left", fontsize=9, fontweight="bold")
    fig4.colorbar(cs4, ax=ax_m, shrink=0.85, pad=0.02)
    if comp == CP_RANK_CHOSEN - 1:
        ax_m.set_xlabel("Longitude")

    ax_d = fig4.add_subplot(gs4[comp + 1, 2])
    ax_d.plot(factor_depth_4[:, comp], DEPTH_LEVELS_M, color="black", linewidth=1.2)
    ax_d.axvline(0, color="grey", linewidth=0.5)
    ax_d.invert_yaxis()
    ax_d.set_title(f"Component {comp+1}: depth factor", loc="left", fontsize=9, fontweight="bold")
    if comp == CP_RANK_CHOSEN - 1:
        ax_d.set_xlabel("Depth factor loading")

    _factor_axes_4 += [ax_t, ax_m, ax_d]

fig4.suptitle(f"Bay of Bengal — CP Tensor Decomposition (rank {CP_RANK_CHOSEN}, "
              "time \u00d7 lat \u00d7 lon \u00d7 depth)", fontsize=13, fontweight="bold")

add_outer_frame(fig4)

savefig(fig4, "04_tensor_decomposition.png")
plt.show()


## 5. Temporal wavelet analysis

**What this shows**: a continuous wavelet transform (CWT, Morlet wavelet, following Torrence & Compo 1998) of the basin-mean oxygen anomaly at **four depths spanning the OMZ core — 150 m, 200 m, 250 m, and 300 m** (see `WAVELET_DEPTHS_M` in the code cell), shown as four scalogram panels in one composite figure. Unlike a single FFT (Analysis 6), a wavelet transform shows **when** each periodicity was strong or weak, not just whether it exists somewhere in the 18-year record — this is the tool for asking "was the annual cycle always this strong?" or "did an ENSO-band (2-7 year) oscillation appear only around specific years, and does that differ by depth?"

**How the anomaly is built, exactly** (identical procedure at each of the four depths): (1) start from the basin-mean monthly series at that depth; (2) **deseasonalize** by subtracting the long-term monthly climatology (the mean value for each calendar month, averaged across all 18 years) from every point, so the strong repeating annual cycle doesn't dominate the picture; (3) **detrend** the deseasonalized series with a simple linear best-fit line removed (`scipy.signal.detrend`), so the secular multi-year decline quantified separately in Analysis 7 doesn't leak into the wavelet power at very long periods. What's left is the interannual-and-shorter wiggle the wavelet transform actually analyzes.

**Reading the figure**: 2x2 grid, one scalogram per depth (150 m top-left, 200 m top-right, 250 m bottom-left, 300 m bottom-right), each with its own colorbar since power scales can differ by depth. Within each panel: x-axis is time, y-axis is period (in years, log scale, short periods at the top). Color is wavelet power (brighter/yellower = a stronger oscillation at that period, at that time). The hatched region is the **cone of influence** — periods/times where edge effects make the transform unreliable, conventionally excluded from interpretation. The black contour line marks the boundary of 95% significance against a red-noise (AR(1)) background, computed with the correct Torrence & Compo (1998) formula. Two horizontal reference bands are marked in every panel: the annual cycle (~1 year) and the interannual ENSO/IOD band (2-7 years) — code below also prints, separately for each depth, which specific years show the strongest interannual-band power, so you can cross-check them against the composite/lag-correlation results in Analysis 12 and see whether the ENSO/IOD signature strengthens, weakens, or shifts timing with depth, rather than eyeballing the scalograms alone. That "strongest years" ranking explicitly **excludes** any time/period point inside the cone of influence before ranking, at every depth — without that exclusion, the ranking would just point at the two ends of the 2005-2022 record (where edge effects inflate apparent power at long periods) rather than a genuine oceanographic signal, so the code checks for that rather than reporting it uncritically.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal
import pycwt

WAVELET_DEPTHS_M = [150.0, 200.0, 250.0, 300.0]   # native depth levels, spanning the OMZ core

# --- 1. Per-depth basin-mean series, deseasonalize + detrend, then CWT -----
_dt_years_5 = 1.0 / 12.0   # monthly sampling, expressed in years
_mother_5 = pycwt.Morlet(6.0)

wavelet_results_5 = {}
for _depth_5 in WAVELET_DEPTHS_M:
    basin_series = regional_mean(ds_ensemble[O2_VAR].sel(depth=_depth_5)).sel(region="Basin")
    time_index = pd.DatetimeIndex(basin_series["time"].values)

    climatology = basin_series.groupby("time.month").mean("time")
    deseasonalized = (basin_series.groupby("time.month") - climatology).values
    anomaly = scipy.signal.detrend(deseasonalized, type="linear")
    print(f"Wavelet input @ {_depth_5:.0f} m: basin-mean O2 anomaly, deseasonalized "
          f"(per-calendar-month climatology removed) and linearly detrended. "
          f"std = {np.std(anomaly):.3f} umol/kg.")

    wave, scales, freqs, coi, _fft, _fftfreqs = pycwt.cwt(
        anomaly, _dt_years_5, dj=1/12, s0=2*_dt_years_5, wavelet=_mother_5,
    )
    power = np.abs(wave) ** 2
    period = 1.0 / freqs

    alpha, _, _ = pycwt.ar1(anomaly)
    signif, _fft_theor = pycwt.significance(
        1.0, _dt_years_5, scales, sigma_test=0, alpha=alpha,
        significance_level=0.95, wavelet=_mother_5,
    )
    sig95 = power / (signif[:, None] * np.ones((1, power.shape[1])))
    print(f"  Estimated AR(1) lag-1 autocorrelation (red-noise significance test) = {alpha:.3f}")

    # Data-driven check: strongest 2-7yr (ENSO/IOD-band) years, EXCLUDING the cone of
    # influence (COI) -- without that exclusion this would just report the two ends of
    # the record, where edge effects inflate apparent power at long periods.
    interannual_mask = (period >= 2.0) & (period <= 7.0)
    coi_valid_mask = period[:, None] <= coi[None, :]
    usable_mask = interannual_mask[:, None] & coi_valid_mask
    interannual_power_series = np.full(power.shape[1], np.nan)
    for _ti in range(power.shape[1]):
        _col_mask = usable_mask[:, _ti]
        if _col_mask.any():
            interannual_power_series[_ti] = power[_col_mask, _ti].mean()
    n_excluded = int(np.isnan(interannual_power_series).sum())
    valid_series_mask = ~np.isnan(interannual_power_series)
    if n_excluded > 0:
        print(f"  {n_excluded} of {power.shape[1]} months have NO reliable (outside-COI) "
              "2-7yr-band estimate and are excluded from the ranking below.")
    threshold = np.nanpercentile(interannual_power_series[valid_series_mask], 90)
    strong_years = sorted(set(
        time_index.year[valid_series_mask & (interannual_power_series >= threshold)]
    ))
    print(f"  Years with strongest 10% of RELIABLE (outside-COI) interannual (2-7yr band) "
          f"wavelet power @ {_depth_5:.0f} m: {strong_years}")

    wavelet_results_5[_depth_5] = dict(
        time=time_index, power=power, period=period, coi=coi, sig95=sig95,
        strong_years=strong_years,
    )

print("\n(Cross-check the strongest-years lists above against the El Nino / La Nina / "
      "+IOD / -IOD years identified in Analysis 12's composite analysis.)")

# --- 2. Composite figure: one scalogram panel per depth, 2x2 grid ----------
fig5, axes5 = plt.subplots(2, 2, figsize=(13.0, 12.0), constrained_layout=True)

for _ax5, _depth_5 in zip(axes5.ravel(), WAVELET_DEPTHS_M):
    r = wavelet_results_5[_depth_5]
    _levels_5 = np.linspace(0, np.nanpercentile(r["power"], 99.5), 21)
    cs5 = _ax5.contourf(r["time"], r["period"], r["power"], levels=_levels_5,
                          cmap=CMAP_SEQUENTIAL, extend="max")
    _ax5.contour(r["time"], r["period"], r["sig95"], [1.0], colors="black", linewidths=1.2)

    # Cone of influence: hatch out the unreliable region
    _ax5.fill_between(r["time"], r["coi"], r["period"].max(), facecolor="none", edgecolor="black",
                        hatch="xxx", linewidth=0.0, alpha=0.4, zorder=5)
    _ax5.plot(r["time"], r["coi"], color="black", linewidth=1.0, zorder=6)

    _ax5.axhline(1.0, color="white", linestyle="--", linewidth=1.0, alpha=0.8)
    _ax5.axhspan(2.0, 7.0, color="white", alpha=0.08)
    _ax5.text(r["time"][3], 1.0, " annual", color="white", fontsize=8, va="bottom")
    _ax5.text(r["time"][3], 3.5, " ENSO/IOD band (2-7 yr)", color="white", fontsize=8, va="center")

    _ax5.set_yscale("log", base=2)
    _ax5.set_ylim(r["period"].max(), r["period"].min())   # short periods at the top
    _ax5.set_ylabel("Period (years)")
    _ax5.set_xlabel("Time")
    _ax5.set_title(f"{_depth_5:.0f} m", loc="left", fontsize=11, fontweight="bold")

    cbar5 = fig5.colorbar(cs5, ax=_ax5, pad=0.02)
    cbar5.set_label("Wavelet power ((µmol/kg)$^2$)")

fig5.suptitle("Bay of Bengal — Basin-Mean O$_2$ Anomaly Wavelet Power Spectrum, "
              "150/200/250/300 m (Morlet, Torrence & Compo 1998)",
              fontsize=13, fontweight="bold")

add_outer_frame(fig5)

savefig(fig5, "05_wavelet_analysis.png")
plt.show()


## 6. FFT / harmonic decomposition of the seasonal cycle

**What this shows**: a classic frequency-domain view of the seasonal cycle, at **four depths spanning the OMZ core — 150 m, 200 m, 250 m, and 300 m** (the same depths used in Analyses 5 and 8), one composite figure per depth (4 PNGs total: `06_fft_harmonic_decomposition_150m.png`, `_200m.png`, `_250m.png`, `_300m.png`). Splitting by depth into separate figures (rather than one cramped combined image) keeps each 4-row x 3-column panel grid legible. First a periodogram (power vs. frequency, in cycles/year) identifies exactly how much of the variance sits at the annual (1 cycle/year) and semi-annual (2 cycles/year) frequencies after the long-term linear trend is removed. Then an explicit **harmonic regression** — a least-squares fit of `a1*sin(2πt/12) + b1*cos(2πt/12) + a2*sin(2πt/6) + b2*cos(2πt/6)` (annual + semi-annual terms only) — reconstructs the seasonal cycle as an explicit, reusable mathematical model rather than just a picture. What's left over (observed minus this two-harmonic reconstruction) is used as a simple proxy for interannual variability — the year-to-year wiggle that isn't just "the monsoon doing its usual thing."

**Reading each figure**: 4 rows (Southern, Middle, Northern, Basin), 3 columns, at that figure's one depth. Left: the periodogram, with the annual and semi-annual frequencies marked and their power values printed. Middle: the detrended observed series (faint) against the 2-harmonic reconstructed seasonal cycle (bold) — a good match means the seasonal cycle is well described by just these two harmonics; a poor match means higher harmonics or non-stationarity matter more than a simple annual+semiannual model captures. Right: the residual (observed minus reconstructed) — this is the interannual-variability proxy; compare its magnitude and any drift to the wavelet's interannual (2-7yr) band in Analysis 5 and the ENSO/IOD composites in Analysis 12. Comparing the four depth-specific figures side by side also shows whether the seasonal cycle's strength/shape (and how well two harmonics explain it) changes with depth across the OMZ core.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal

FFT_DEPTHS_M = [150.0, 200.0, 250.0, 300.0]   # same OMZ-core depths used in Analyses 5 and 8

panel_order_6 = REGION_NAMES + ["Basin"]

for _depth_6 in FFT_DEPTHS_M:
    regional_6 = regional_mean(ds_ensemble[O2_VAR].sel(depth=_depth_6))  # dims: region, time
    _t_months_6 = np.arange(regional_6.sizes["time"], dtype=float)

    fig6, axes6 = plt.subplots(4, 3, figsize=(13.0, 13.0), constrained_layout=True)

    for row, region_name in enumerate(panel_order_6):
        s_raw = regional_6.sel(region=region_name).values
        s_detrended = scipy.signal.detrend(s_raw, type="linear")

        # --- periodogram (fs=12 -> frequency axis directly in cycles/year) ----
        freqs, power = scipy.signal.periodogram(s_detrended, fs=12.0, scaling="spectrum")
        _annual_idx = int(np.argmin(np.abs(freqs - 1.0)))
        _semiannual_idx = int(np.argmin(np.abs(freqs - 2.0)))

        ax_p = axes6[row, 0]
        ax_p.plot(freqs, power, color="black", linewidth=1.0)
        ax_p.axvline(1.0, color="firebrick", linestyle="--", linewidth=1.0)
        ax_p.axvline(2.0, color="steelblue", linestyle="--", linewidth=1.0)
        ax_p.text(1.0, power[_annual_idx], f"  annual\n  {power[_annual_idx]:.2f}",
                   color="firebrick", fontsize=8, va="bottom")
        ax_p.text(2.0, power[_semiannual_idx], f"  semi-annual\n  {power[_semiannual_idx]:.2f}",
                   color="steelblue", fontsize=8, va="bottom")
        ax_p.set_xlim(0, 6)
        ax_p.set_title(f"{region_name}: periodogram", loc="left", fontsize=10, fontweight="bold")
        if row == 3:
            ax_p.set_xlabel("Frequency (cycles/year)")
        ax_p.set_ylabel("Power")

        # --- explicit annual + semiannual harmonic least-squares fit ----------
        design_6 = np.column_stack([
            np.sin(2 * np.pi * 1 * _t_months_6 / 12.0), np.cos(2 * np.pi * 1 * _t_months_6 / 12.0),
            np.sin(2 * np.pi * 2 * _t_months_6 / 12.0), np.cos(2 * np.pi * 2 * _t_months_6 / 12.0),
        ])
        coeffs_6, *_ = np.linalg.lstsq(design_6, s_detrended, rcond=None)
        reconstructed_6 = design_6 @ coeffs_6
        residual_6 = s_detrended - reconstructed_6
        _r2_6 = 1.0 - np.var(residual_6) / np.var(s_detrended)

        ax_r = axes6[row, 1]
        ax_r.plot(regional_6["time"].values, s_detrended, color="grey", linewidth=0.8, alpha=0.6, label="observed (detrended)")
        ax_r.plot(regional_6["time"].values, reconstructed_6, color="firebrick", linewidth=1.4, label="annual+semiannual fit")
        ax_r.set_title(f"{region_name}: seasonal fit (R²={_r2_6:.2f})", loc="left", fontsize=10, fontweight="bold")
        if row == 3:
            ax_r.set_xlabel("Time")
        if row == 0:
            ax_r.legend(fontsize=8, loc="upper right")

        ax_res = axes6[row, 2]
        ax_res.plot(regional_6["time"].values, residual_6, color="black", linewidth=0.9)
        ax_res.axhline(0, color="grey", linewidth=0.5)
        ax_res.set_title(f"{region_name}: residual (interannual proxy)", loc="left", fontsize=10, fontweight="bold")
        if row == 3:
            ax_res.set_xlabel("Time")

    fig6.suptitle(f"Bay of Bengal — FFT / Harmonic Decomposition of the Seasonal Cycle @ "
                  f"{_depth_6:.0f} m", fontsize=13, fontweight="bold")

    add_outer_frame(fig6)

    savefig(fig6, f"06_fft_harmonic_decomposition_{_depth_6:.0f}m.png")
    plt.show()


## 7. Modified Mann-Kendall trend test + Sen's slope

**What this shows**: a formal, non-parametric trend test at every grid cell, for each of the 7 config depths, plus a summary for each region + basin. Rather than just eyeballing a line going up or down (as in Analysis 1), this gives every trend a p-value and a robust slope estimate (Sen's slope — the median of all pairwise slopes, far less sensitive to outliers than ordinary least-squares).

**Why "correlated seasonal" rather than the plain Hamed-Rao test**: this notebook uses a modified Mann-Kendall test that corrects for serial autocorrelation (Hamed & Rao 1998) rather than skipping that correction — plain Mann-Kendall on autocorrelated data understates the true variance of the test statistic and overstates significance. But this dataset is *monthly*, with a strong seasonal cycle (Analysis 6 shows a real annual/semi-annual signal). Applying the plain (non-seasonal) Hamed-Rao correction directly to raw monthly values would treat the repeating seasonal cycle itself as "autocorrelation," which both distorts the correction and conflates seasonality with the long-term trend it's supposed to isolate. `pymannkendall`'s **Correlated Seasonal Mann-Kendall test** (Hipel & McLeod, 1994) is used instead: it applies the Mann-Kendall/Sen's-slope framework *within* each calendar month across years (so the seasonal cycle itself is never treated as trend) while still accounting for cross-season correlation in the test statistic's variance — i.e. it is the seasonal generalization of that same autocorrelation-correction principle, adapted correctly for monthly data with a strong annual cycle. Sen's slope from this test is reported directly in **umol/kg per year** (confirmed against a synthetic series with a known trend before trusting it here).

**Reading the figure**: one map per depth (7 panels, `RdBu_r`, zero-centered — red = oxygenating, blue = deoxygenating), Sen's slope magnitude in umol/kg/year. Grid cells where the trend is significant at p < 0.05 are marked with a small stipple dot; cells without a stipple did not reach significance at that depth, even if the map color suggests a trend — the stippling is what actually establishes "is this real." A per-region + basin summary table (slope, p-value, significance) at all 7 depths is printed above the figure.


In [ ]:
import time as _time_mod
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
import pymannkendall as mk

MK_DEPTHS_M = ANALYSIS1_DEPTHS_M   # all 7 config depths, per the "ideally all 7" guidance

def _cell_trend_7(series):
    """Runs the Correlated Seasonal Mann-Kendall test on one grid cell's monthly
    series; returns (sen_slope_per_year, p_value), or (nan, nan) if too much
    data is missing to run the test at all."""
    if np.isnan(series).any():
        return np.nan, np.nan
    try:
        result = mk.correlated_seasonal_test(series, period=12)
        return float(result.slope), float(result.p)
    except Exception:
        return np.nan, np.nan

# --- 1. Per-grid-cell Sen's slope + p-value maps, one per depth ------------
_t0_7 = _time_mod.time()
slope_maps_7 = {}
pvalue_maps_7 = {}
for depth_m in MK_DEPTHS_M:
    da_d = ds_ensemble[O2_VAR].interp(depth=depth_m, method="linear")
    slope_grid = np.full((len(LAT_VALUES), len(LON_VALUES)), np.nan)
    pval_grid = np.full((len(LAT_VALUES), len(LON_VALUES)), np.nan)
    for i in range(len(LAT_VALUES)):
        for j in range(len(LON_VALUES)):
            series_ij = da_d.isel(lat=i, lon=j).values
            slope_grid[i, j], pval_grid[i, j] = _cell_trend_7(series_ij)
    slope_maps_7[depth_m] = slope_grid
    pvalue_maps_7[depth_m] = pval_grid
    print(f"  depth {depth_m:.0f} m done ({_time_mod.time() - _t0_7:.1f}s elapsed)")
print(f"Per-cell trend maps for all {len(MK_DEPTHS_M)} depths took {_time_mod.time() - _t0_7:.1f}s total.")

# --- 2. Per-region + basin trend summary, same 7 depths ---------------------
print("\nRegion/basin Sen's slope summary (umol/kg/year), Correlated Seasonal Mann-Kendall:")
region_trend_rows_7 = []
for depth_m in MK_DEPTHS_M:
    regional_series_7 = regional_mean(ds_ensemble[O2_VAR].interp(depth=depth_m, method="linear"))
    for region_name in REGION_NAMES + ["Basin"]:
        s = regional_series_7.sel(region=region_name).values
        slope, pval = _cell_trend_7(s)
        sig = "significant (p<0.05)" if (not np.isnan(pval) and pval < SIGNIFICANCE_ALPHA) else "not significant"
        region_trend_rows_7.append((depth_m, region_name, slope, pval, sig))
        print(f"  {depth_m:>4.0f} m  {region_name:9s}  slope={slope:+7.3f}  p={pval:.4f}  {sig}")

# --- 3. Composite figure: one Sen's-slope map per depth ---------------------
_all_slopes_7 = np.concatenate([s[~np.isnan(s)] for s in slope_maps_7.values()])
_vmax_7 = float(np.nanpercentile(np.abs(_all_slopes_7), 98)) or 1.0
_norm_7 = TwoSlopeNorm(vcenter=0.0, vmin=-_vmax_7, vmax=_vmax_7)

fig7, axes7 = plt.subplots(3, 3, figsize=(13.2, 12.0), constrained_layout=True)
axes7_flat = axes7.ravel()
_cs_7 = None
for k, depth_m in enumerate(MK_DEPTHS_M):
    ax = axes7_flat[k]
    _cs_7 = ax.pcolormesh(LON_VALUES, LAT_VALUES, slope_maps_7[depth_m], cmap=CMAP_DIVERGING,
                            norm=_norm_7, shading="nearest")
    _sig_mask = pvalue_maps_7[depth_m] < SIGNIFICANCE_ALPHA
    _lon_grid, _lat_grid = np.meshgrid(LON_VALUES, LAT_VALUES)
    ax.scatter(_lon_grid[_sig_mask], _lat_grid[_sig_mask], **STIPPLE_KW)
    ax.set_title(f"{depth_m:.0f} m", loc="left", fontsize=10, fontweight="bold")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
for k in range(len(MK_DEPTHS_M), len(axes7_flat)):
    axes7_flat[k].axis("off")

cbar7 = fig7.colorbar(_cs_7, ax=axes7_flat[:len(MK_DEPTHS_M)].tolist(), shrink=0.85, pad=0.02)
cbar7.set_label("Sen's slope (\u00b5mol/kg/year)")

fig7.suptitle("Bay of Bengal \u2014 Sen's Slope (Correlated Seasonal Mann-Kendall), 7 Depths "
              "(stippled = p<0.05)", fontsize=13, fontweight="bold")

add_outer_frame(fig7)

savefig(fig7, "07_mannkendall_sens_slope.png")
plt.show()


## 8. Formal change-point detection

**What this shows**: automated detection of "change points" — months where the statistical behavior of the deseasonalized oxygen series abruptly shifts — for each region + the basin, at **four depths spanning the OMZ core — 150 m, 200 m, 250 m, and 300 m** (the same depths used in Analyses 5 and 6), one composite figure per depth (4 PNGs total: `08_changepoint_detection_150m.png`, `_200m.png`, `_250m.png`, `_300m.png`). Splitting by depth into separate figures (rather than cramming all 16 region×depth panels into one image) keeps each 4-row panel figure legible. This is a genuinely useful check on the Mann-Kendall/Sen's-slope results in Analysis 7: a "trend" can either be a smooth, gradual decline, or it can actually be one or two abrupt jumps with flat periods in between — change-point detection tells you which, and comparing across depths shows whether any detected shift is confined to one part of the OMZ core or extends through it.

**Method and why**: `ruptures`' **PELT** (Pruned Exact Linear Time) algorithm with an **RBF (radial basis function) cost** is used, rather than Binary Segmentation. PELT finds the *exact* optimal segmentation for a given penalty in expected linear time (Binary Segmentation is a greedy approximation that can miss the true optimum), and the RBF cost detects a shift in the *general distribution* of the series (not just its mean), which is the more conservative, general-purpose choice when we don't know in advance whether an anomaly would show up as a mean shift, a variance shift, or both. The penalty (which controls how many change points are found — too low over-segments into noise, too high misses real shifts) is chosen automatically, independently at each depth: the smallest penalty (on a `log(n)`-scaled grid) that keeps the number of detected change points at a modest, interpretable count, printed explicitly below so the choice is not hidden.

**The honesty check this analysis exists to run**: this project's pipeline has a documented data-quality anomaly in the raw G4D-DOC product around **December 2019 - January 2020** (a spurious oxygenation "pulse" — see Analysis 13 for the direct product-vs-product QC check). The code below explicitly checks, at every one of the four depths, whether the change-point algorithm flags anything in or near that window in the **ensemble** series analyzed here (which blends G4D-DOC with GEOXYGEN, so the anomaly may be diluted, not eliminated) — and prints the honest answer either way for each depth: if the window is flagged, whether that's more likely the data artifact than a real oceanographic shift; if it is **not** flagged, that is reported explicitly as a miss rather than reinterpreted after the fact to look like a success.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ruptures as rpt

CHANGEPOINT_DEPTHS_M = [150.0, 200.0, 250.0, 300.0]   # same OMZ-core depths used in Analyses 5 and 6
CHANGEPOINT_KNOWN_ANOMALY_WINDOW = (pd.Timestamp("2019-11-01"), pd.Timestamp("2020-02-28"))
CHANGEPOINT_MAX_INTERPRETABLE = 6   # cap on how many change points we consider "interpretable"
CHANGEPOINT_PENALTY_GRID = [1.0, 2.0, 3.0, 5.0, 8.0, 12.0, 20.0, 30.0, 50.0]

panel_order_8 = REGION_NAMES + ["Basin"]

def _choose_penalty_and_fit_8(signal):
    """Tries penalties on a log(n)-scaled grid (smallest first) and returns the
    first one that yields a modest, interpretable number of change points."""
    for mult in CHANGEPOINT_PENALTY_GRID:
        pen = mult * np.log(len(signal))
        algo = rpt.Pelt(model="rbf").fit(signal.reshape(-1, 1))
        bkps = algo.predict(pen=pen)
        n_changepoints = len(bkps) - 1   # ruptures always includes len(signal) as the last "breakpoint"
        if n_changepoints <= CHANGEPOINT_MAX_INTERPRETABLE:
            return bkps[:-1], pen, mult
    return [], pen, mult   # fell through: even the largest penalty over-segmented; report none

for _depth_8 in CHANGEPOINT_DEPTHS_M:
    regional_8 = regional_mean(ds_ensemble[O2_VAR].sel(depth=_depth_8))
    _time_8 = pd.DatetimeIndex(regional_8["time"].values)
    _n_8 = regional_8.sizes["time"]

    print("\n" + "#" * 78)
    print(f"CHANGE-POINT DETECTION @ {_depth_8:.0f} m")
    print("#" * 78)

    results_8 = {}
    for region_name in panel_order_8:
        climatology = regional_8.sel(region=region_name).groupby("time.month").mean("time")
        s_deseasonalized = (regional_8.sel(region=region_name).groupby("time.month") - climatology).values
        bkps_idx, pen_used, mult_used = _choose_penalty_and_fit_8(s_deseasonalized)
        bkps_dates = [_time_8[i] for i in bkps_idx]
        results_8[region_name] = dict(series=s_deseasonalized, bkps_idx=bkps_idx, bkps_dates=bkps_dates,
                                        penalty=pen_used, penalty_mult=mult_used)
        print(f"{region_name}: penalty multiplier={mult_used:g} (pen={pen_used:.2f}) -> "
              f"{len(bkps_idx)} change point(s): {[d.strftime('%Y-%m') for d in bkps_dates]}")

    # --- Explicit honesty check against the known Dec2019-Jan2020 anomaly -------
    print("\n" + "=" * 78)
    print(f"CROSS-CHECK @ {_depth_8:.0f} m vs. the known Dec 2019 - Jan 2020 G4D-DOC data-quality anomaly:")
    _lo, _hi = CHANGEPOINT_KNOWN_ANOMALY_WINDOW
    for region_name in panel_order_8:
        flagged = [d for d in results_8[region_name]["bkps_dates"] if _lo <= d <= _hi]
        if flagged:
            print(f"  {region_name}: FLAGGED a change point at {[d.strftime('%Y-%m') for d in flagged]} "
                  "inside the known-anomaly window. Given this coincides exactly with the documented "
                  "G4D-DOC data-quality issue (and not with any independently corroborated oceanographic "
                  "event), this is most plausibly the algorithm correctly detecting the DATA ARTIFACT, "
                  "not a real oceanographic regime shift.")
        else:
            print(f"  {region_name}: did NOT flag any change point in the known-anomaly window "
                  f"({_lo:%Y-%m} to {_hi:%Y-%m}). This is reported as a genuine MISS -- the ensemble "
                  "blend with GEOXYGEN evidently dilutes the G4D-DOC-only pulse below this method's "
                  "detection threshold at this depth/region, rather than the method successfully "
                  "catching it. See Analysis 13 for a direct per-product check that is more likely to "
                  "catch it.")
    print("=" * 78)

    # --- Figure: one panel per region + basin, deseasonalized series + change points
    fig8, axes8 = plt.subplots(4, 1, figsize=(12, 12), sharex=True, constrained_layout=True)
    for ax, region_name in zip(axes8, panel_order_8):
        r = results_8[region_name]
        ax.plot(_time_8, r["series"], color="black", linewidth=0.9)
        ax.axhline(0, color="grey", linewidth=0.5)
        for d in r["bkps_dates"]:
            ax.axvline(d, color="firebrick", linestyle="--", linewidth=1.3)
        ax.axvspan(CHANGEPOINT_KNOWN_ANOMALY_WINDOW[0], CHANGEPOINT_KNOWN_ANOMALY_WINDOW[1],
                    color="orange", alpha=0.15)
        _n_bkps = len(r["bkps_idx"])
        ax.set_title(f"{region_name}  ({_n_bkps} change point(s))",
                     loc="left", fontsize=11, fontweight="bold")
        ax.set_ylabel("O$_2$ anomaly (µmol/kg)")
    axes8[-1].set_xlabel("Time")
    axes8[0].text(0.01, 0.90, "orange band = known Dec2019-Jan2020 G4D-DOC anomaly window; "
                  "red dashed = detected change point", transform=axes8[0].transAxes,
                  fontsize=8, va="top", color="dimgrey")

    fig8.suptitle(f"Bay of Bengal — Change-Point Detection (PELT, RBF cost) @ "
                  f"{_depth_8:.0f} m", fontsize=13, fontweight="bold")

    add_outer_frame(fig8)

    savefig(fig8, f"08_changepoint_detection_{_depth_8:.0f}m.png")
    plt.show()


## 9. Threshold-based OMZ volume, area, and thickness

**What this shows**: three complementary time series describing how big the oxygen minimum zone (OMZ) is, using the Hypoxic/Suboxic boundary (`OMZ_BOUNDARY_THRESHOLD_UMOL_KG` = 22 umol/kg, from the config cell) as the definition of "inside the OMZ": (a) total **OMZ volume** (the whole basin, all depths), (b) **OMZ area** at its core depth (the single depth level with the largest long-term-average OMZ footprint — determined from the data, not assumed), and (c) basin-mean **OMZ thickness** per water column (how many vertical meters of a typical column are inside the OMZ). Comparing their trends tells you *why* the OMZ is growing or shrinking — is it spreading horizontally (area), getting vertically thicker, or both?

**Getting the geometry right**: cell volumes use true depth-layer thicknesses (the vertical distance between the midpoints of adjacent native depth levels — not just "one level = one unit"), with the surface layer starting at 0 m and the bottom layer's lower edge extrapolated below the deepest level (1995 m) using the same half-spacing as the last two levels, exactly as in the companion `Bay_of_Bengal_Yearly_Maps.ipynb` notebook's zone-volume calculation. Horizontal cell areas use the exact spherical-cap-sector formula (`R\u00b2 \u00b7 \u0394lon(rad) \u00b7 (sin(lat_hi) - sin(lat_lo))`, R = 6371 km), not a flat-Earth approximation.

**Trend annotation**: each of the 3 series gets a Sen's slope (with p-value) from the same Correlated Seasonal Mann-Kendall method used in Analysis 7, printed on its panel. A markdown-free, data-driven summary at the end of the code cell compares the area and thickness trends' relative (%/year) magnitudes to state plainly whether the volume trend is more area-driven or more thickness-driven.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pymannkendall as mk

# --- 1. True depth-layer thicknesses (km), 0 m top edge, extrapolated bottom edge
_depth_m_9 = DEPTH_LEVELS_M
_edges_9 = np.zeros(len(_depth_m_9) + 1)
_edges_9[0] = 0.0
_edges_9[1:-1] = (_depth_m_9[:-1] + _depth_m_9[1:]) / 2.0
_last_spacing_9 = _depth_m_9[-1] - _depth_m_9[-2]
_edges_9[-1] = _depth_m_9[-1] + _last_spacing_9 / 2.0
thickness_km_9 = np.diff(_edges_9) / 1000.0   # per native depth level

# --- 2. Exact spherical-cap-sector cell areas (km^2), function of latitude only
_R_KM_9 = 6371.0
_dlat_9 = float(np.median(np.diff(LAT_VALUES)))
_dlon_9 = float(np.median(np.diff(LON_VALUES)))
_lat_lo_9 = np.deg2rad(LAT_VALUES - _dlat_9 / 2.0)
_lat_hi_9 = np.deg2rad(LAT_VALUES + _dlat_9 / 2.0)
_dlon_rad_9 = np.deg2rad(_dlon_9)
area_km2_9 = (_R_KM_9 ** 2) * _dlon_rad_9 * (np.sin(_lat_hi_9) - np.sin(_lat_lo_9))   # per lat row

print(f"Depth layer thicknesses: {thickness_km_9.min()*1000:.0f}-{thickness_km_9.max()*1000:.0f} m "
      f"(bottom edge extrapolated to {_edges_9[-1]:.0f} m). Cell area: "
      f"{area_km2_9.min():.0f}-{area_km2_9.max():.0f} km\u00b2 (varies only with latitude).")

# --- 3. OMZ mask (O2 < threshold, valid ocean cells only) -------------------
omz_mask_9 = ds_ensemble[O2_VAR] < OMZ_BOUNDARY_THRESHOLD_UMOL_KG   # dims: time, depth, lat, lon (NaN -> False)

# --- (a) OMZ VOLUME(t): sum of thickness*area over every OMZ cell -----------
_thickness_by_depth_9 = xr.DataArray(thickness_km_9, dims="depth", coords={"depth": ds_ensemble["depth"]})
_area_by_lat_9 = xr.DataArray(area_km2_9, dims="lat", coords={"lat": ds_ensemble["lat"]})
_cell_volume_9 = _thickness_by_depth_9 * _area_by_lat_9
omz_volume_km3_9 = (omz_mask_9 * _cell_volume_9).sum(dim=("depth", "lat", "lon"))
omz_volume_million_km3_9 = omz_volume_km3_9 / 1.0e6

# --- (b) OMZ AREA(t) at the core depth (largest long-term-mean OMZ footprint)
_area_da_9 = xr.DataArray(area_km2_9, dims="lat", coords={"lat": ds_ensemble["lat"]})
omz_area_per_depth_9 = (omz_mask_9 * _area_da_9).sum(dim=("lat", "lon"))   # dims: time, depth
_mean_area_by_depth_9 = omz_area_per_depth_9.mean(dim="time")
OMZ_CORE_DEPTH_M = float(_mean_area_by_depth_9.idxmax(dim="depth").values)
omz_area_km2_9 = omz_area_per_depth_9.sel(depth=OMZ_CORE_DEPTH_M)
print(f"OMZ core depth (largest long-term-mean OMZ footprint) = {OMZ_CORE_DEPTH_M:.0f} m "
      f"(mean area there = {float(_mean_area_by_depth_9.sel(depth=OMZ_CORE_DEPTH_M)):.0f} km\u00b2).")

# --- (c) OMZ THICKNESS(t): basin cosine-weighted mean of per-column OMZ thickness
_thickness_da_9 = xr.DataArray(thickness_km_9, dims="depth", coords={"depth": ds_ensemble["depth"]})
_per_column_thickness_km_9 = (omz_mask_9 * _thickness_da_9).sum(dim="depth")   # dims: time, lat, lon
_valid_column_9 = ds_ensemble[O2_VAR].notnull().any(dim="depth")               # a column exists if any depth is valid
_coslat_9 = np.cos(np.deg2rad(ds_ensemble["lat"]))
_w_9 = (_coslat_9 * _valid_column_9).fillna(0)
omz_thickness_m_9 = 1000.0 * (_per_column_thickness_km_9 * _coslat_9).where(_valid_column_9).weighted(_w_9).mean(dim=("lat", "lon"))

# --- 4. Trend annotation on all 3 series, reusing Analysis 7's method -------
def _series_trend_9(values):
    try:
        r = mk.correlated_seasonal_test(np.asarray(values, dtype=float), period=12)
        return float(r.slope), float(r.p)
    except Exception:
        return np.nan, np.nan

_series_9 = {
    "OMZ volume (10\u2076 km\u00b3)": omz_volume_million_km3_9.values,
    f"OMZ area @ {OMZ_CORE_DEPTH_M:.0f} m (km\u00b2)": omz_area_km2_9.values,
    "OMZ thickness, basin mean (m)": omz_thickness_m_9.values,
}
_trends_9 = {name: _series_trend_9(vals) for name, vals in _series_9.items()}
for name, (slope, pval) in _trends_9.items():
    sig = "p<0.05" if (not np.isnan(pval) and pval < SIGNIFICANCE_ALPHA) else "not significant"
    print(f"  {name}: Sen's slope = {slope:+.4g} per year ({sig}, p={pval:.4f})")

# --- 5. Data-driven verdict: is the volume trend more area- or thickness-driven?
_area_slope, _area_p = _trends_9[f"OMZ area @ {OMZ_CORE_DEPTH_M:.0f} m (km\u00b2)"]
_thick_slope, _thick_p = _trends_9["OMZ thickness, basin mean (m)"]
_area_pct_per_yr = 100 * _area_slope / float(np.nanmean(omz_area_km2_9.values))
_thick_pct_per_yr = 100 * _thick_slope / float(np.nanmean(omz_thickness_m_9.values))
print(f"\nRelative trend comparison: area changing {_area_pct_per_yr:+.2f}%/year vs. "
      f"thickness changing {_thick_pct_per_yr:+.2f}%/year (at the core depth / basin mean respectively).")
if abs(_area_pct_per_yr) > abs(_thick_pct_per_yr) * 1.2:
    print("VERDICT: the OMZ volume trend appears to be predominantly AREA-driven "
          "(horizontal expansion/contraction) rather than thickness-driven.")
elif abs(_thick_pct_per_yr) > abs(_area_pct_per_yr) * 1.2:
    print("VERDICT: the OMZ volume trend appears to be predominantly THICKNESS-driven "
          "(vertical deepening/shoaling) rather than area-driven.")
else:
    print("VERDICT: area and thickness are changing at comparable relative rates -- the "
          "volume trend does not have a single dominant driver.")

# --- 6. Figure: 3 stacked time series panels --------------------------------
_time_9 = pd.DatetimeIndex(ds_ensemble["time"].values)
fig9, axes9 = plt.subplots(3, 1, figsize=(12, 10), sharex=True, constrained_layout=True)
for ax, (name, vals) in zip(axes9, _series_9.items()):
    slope, pval = _trends_9[name]
    ax.plot(_time_9, vals, color="black", linewidth=1.1)
    sig_txt = "p<0.05" if (not np.isnan(pval) and pval < SIGNIFICANCE_ALPHA) else "n.s."
    ax.set_title(f"{name}   (Sen's slope = {slope:+.4g}/year, {sig_txt})",
                 loc="left", fontsize=10.5, fontweight="bold")
    ax.set_ylabel(name.split("(")[-1].rstrip(")") if "(" in name else "")
axes9[-1].set_xlabel("Time")

fig9.suptitle("Bay of Bengal \u2014 OMZ Volume, Area, and Thickness Over Time "
              f"(threshold = {OMZ_BOUNDARY_THRESHOLD_UMOL_KG:.0f} \u00b5mol/kg)",
              fontsize=13, fontweight="bold")

add_outer_frame(fig9)

savefig(fig9, "09_omz_volume_area_thickness.png")
plt.show()


## 10. Oxycline / OMZ-boundary depth mapping

**What this shows**: the depth of the **oxycline** — here defined as the shallowest depth at which oxygen first drops below the same Hypoxic/Suboxic boundary used to define "the OMZ" throughout this notebook (`OMZ_BOUNDARY_THRESHOLD_UMOL_KG` = 22 umol/kg), i.e. the upper boundary of the oxygen minimum zone. This is a genuinely different quantity from Analysis 9's OMZ thickness: thickness is basin-averaged and depth-integrated, while the oxycline depth is a single number **per water column per month** — exactly analogous to finding "the top of the thermocline" in physical oceanography, just for oxygen.

**Why interpolation, not nearest-snapping**: at each column/time, the code below scans the native depth levels from shallow to deep, finds the first pair of adjacent levels that bracket the threshold crossing (oxygen above the threshold at the shallower level, below it at the deeper level), and **linearly interpolates between those two levels' oxygen values** to estimate the exact crossing depth — rather than just reporting whichever of the two native levels happens to be closer. With levels 50-100 m apart in the upper ocean, nearest-snapping would introduce tens of meters of avoidable error into a quantity this notebook then trends over time. Two edge cases are handled explicitly and documented in the code's output: a column that is already below the threshold at the shallowest sampled level (10 m — the oxycline is reported at 10 m, since there is nothing shallower to interpolate from), and a column that never crosses the threshold at any sampled depth (no oxycline exists there that month — reported as missing/NaN, not a fabricated deep value).

**Reading the figure**: (a) the time-mean oxycline depth map — shallower (brighter) means a bigger, shallower OMZ footprint at that location on average; (b) region + basin monthly oxycline-depth time series with a Sen's slope trend (note the sign convention: a **negative** slope means the oxycline is **shoaling** — getting shallower, i.e. the OMZ is expanding upward toward the surface; a **positive** slope means it's **deepening** — the OMZ retreating, an improving-oxygen signal); (c) a latitude-time Hovmoller of the oxycline depth (longitude-averaged), showing whether shoaling/deepening is basin-wide or concentrated in particular latitude bands.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pymannkendall as mk

def _oxycline_depth_1col_10(o2_profile, depth_levels, threshold):
    """o2_profile, depth_levels: 1-D arrays, shallow-to-deep, same length.
    Returns the interpolated depth (m) of the first threshold crossing, the
    shallowest level (boundary case: already below threshold at the top), or
    NaN (boundary case: never crosses -- no oxycline this column/month)."""
    if np.isnan(o2_profile).all():
        return np.nan
    if o2_profile[0] < threshold:
        return float(depth_levels[0])   # already inside the OMZ at the shallowest sample
    for k in range(1, len(o2_profile)):
        o_prev, o_curr = o2_profile[k - 1], o2_profile[k]
        if np.isnan(o_prev) or np.isnan(o_curr):
            continue
        if o_prev >= threshold and o_curr < threshold:
            frac = (threshold - o_prev) / (o_curr - o_prev)
            return float(depth_levels[k - 1] + frac * (depth_levels[k] - depth_levels[k - 1]))
    return np.nan   # never crosses at any sampled depth this column/month

# --- 1. Compute the oxycline depth field (time, lat, lon) -------------------
_o2_vals_10 = ds_ensemble[O2_VAR].transpose("time", "depth", "lat", "lon").values
_nt_10, _nd_10, _nlat_10, _nlon_10 = _o2_vals_10.shape
oxycline_depth_vals_10 = np.full((_nt_10, _nlat_10, _nlon_10), np.nan)
_n_at_surface_10 = 0
_n_never_crosses_10 = 0
for _ti in range(_nt_10):
    for _i in range(_nlat_10):
        for _j in range(_nlon_10):
            profile = _o2_vals_10[_ti, :, _i, _j]
            d = _oxycline_depth_1col_10(profile, DEPTH_LEVELS_M, OMZ_BOUNDARY_THRESHOLD_UMOL_KG)
            oxycline_depth_vals_10[_ti, _i, _j] = d
            if not np.isnan(profile).all():
                if profile[0] < OMZ_BOUNDARY_THRESHOLD_UMOL_KG:
                    _n_at_surface_10 += 1
                elif np.isnan(d):
                    _n_never_crosses_10 += 1

oxycline_depth_10 = xr.DataArray(
    oxycline_depth_vals_10, dims=("time", "lat", "lon"),
    coords={"time": ds_ensemble["time"], "lat": ds_ensemble["lat"], "lon": ds_ensemble["lon"]},
    name="oxycline_depth_m",
)
_n_valid_column_months_10 = int((~np.isnan(_o2_vals_10).all(axis=1)).sum())
_n_valid_columns_avg_10 = _n_valid_column_months_10 / _nt_10
print(f"Oxycline computed for {_nt_10} months x ~{_n_valid_columns_avg_10:.0f} valid ocean columns "
      f"per month on average ({_n_valid_column_months_10} valid column-months total).")
print(f"  {_n_at_surface_10} column-months were already below the threshold at 10 m (shallowest sample).")
print(f"  {_n_never_crosses_10} column-months never crossed the threshold at any sampled depth (no oxycline -> NaN).")

# --- 2. (a) Time-mean oxycline depth map ------------------------------------
oxycline_time_mean_10 = oxycline_depth_10.mean(dim="time", skipna=True)

# --- 3. (b) Region + basin oxycline time series + Sen's slope --------------
oxycline_regional_10 = regional_mean(oxycline_depth_10)   # reuses the same helper as every other analysis
_trend_rows_10 = []
for region_name in REGION_NAMES + ["Basin"]:
    vals = oxycline_regional_10.sel(region=region_name).values
    try:
        r = mk.correlated_seasonal_test(vals, period=12)
        slope, pval = float(r.slope), float(r.p)
    except Exception:
        slope, pval = np.nan, np.nan
    direction = "SHOALING (OMZ expanding upward)" if slope < 0 else "DEEPENING (OMZ retreating)"
    sig = "p<0.05" if (not np.isnan(pval) and pval < SIGNIFICANCE_ALPHA) else "not significant"
    _trend_rows_10.append((region_name, slope, pval))
    print(f"  {region_name:9s}: oxycline Sen's slope = {slope:+.3f} m/year -> {direction} ({sig}, p={pval:.4f})")

# --- 4. (c) Latitude-time Hovmoller (longitude-averaged) -------------------
oxycline_hovmoller_10 = oxycline_depth_10.mean(dim="lon", skipna=True)   # dims: time, lat

# --- 5. Composite figure -----------------------------------------------------
fig10 = plt.figure(figsize=(13.2, 12.0), constrained_layout=True)
gs10 = fig10.add_gridspec(2, 2, height_ratios=[1, 1])

ax_map = fig10.add_subplot(gs10[0, 0])
cs_map = ax_map.pcolormesh(LON_VALUES, LAT_VALUES, oxycline_time_mean_10.values,
                              cmap=CMAP_SEQUENTIAL, shading="nearest")
ax_map.set_title("(a) Time-mean oxycline depth", loc="left", fontsize=10, fontweight="bold")
ax_map.set_xlabel("Longitude"); ax_map.set_ylabel("Latitude")
fig10.colorbar(cs_map, ax=ax_map, label="Depth (m)", shrink=0.85)

ax_ts = fig10.add_subplot(gs10[0, 1])
for region_name in REGION_NAMES + ["Basin"]:
    ax_ts.plot(oxycline_regional_10["time"].values, oxycline_regional_10.sel(region=region_name).values,
               linewidth=1.1, label=region_name)
ax_ts.invert_yaxis()   # shallower (more OMZ) drawn upward, visually consistent with "shoaling = worse"
ax_ts.set_title("(b) Region + basin oxycline depth", loc="left", fontsize=10, fontweight="bold")
ax_ts.set_xlabel("Time"); ax_ts.set_ylabel("Depth (m)")
ax_ts.legend(fontsize=8)

ax_hov = fig10.add_subplot(gs10[1, :])
cs_hov = ax_hov.contourf(oxycline_hovmoller_10["time"].values, oxycline_hovmoller_10["lat"].values,
                           oxycline_hovmoller_10.values.T, levels=21, cmap=CMAP_SEQUENTIAL, extend="both")
ax_hov.set_title("(c) Oxycline depth: time \u2013 latitude Hovm\u00f6ller (longitude-averaged)",
                 loc="left", fontsize=10, fontweight="bold")
ax_hov.set_xlabel("Time"); ax_hov.set_ylabel("Latitude")
fig10.colorbar(cs_hov, ax=ax_hov, label="Depth (m)", shrink=0.85)

fig10.suptitle(f"Bay of Bengal \u2014 Oxycline (OMZ Upper-Boundary) Depth, threshold = "
              f"{OMZ_BOUNDARY_THRESHOLD_UMOL_KG:.0f} \u00b5mol/kg", fontsize=13, fontweight="bold")

add_outer_frame(fig10)

savefig(fig10, "10_oxycline_depth.png")
plt.show()


## 11. Spatial autocorrelation: semivariogram + global Moran's I

**What this shows**: how spatially "clumpy" or smooth the oxygen field is at 150 m (the same OMZ-core depth used in Analysis 3, and the shallowest of the four OMZ-core depths used in Analyses 5, 6, and 8), and whether that changes between an El Nino year, a La Nina year, a neutral year, and the full-record average. Two complementary tools are used: an empirical **semivariogram** (via `skgstat`), fit with a spherical model to estimate the **range** (the distance beyond which two points stop being spatially correlated) and **sill** (the plateau semi-variance at that distance); and the **global Moran's I** statistic (via `esda`/`libpysal`), a single number from -1 to +1 summarizing overall spatial autocorrelation (near +1 = strongly clustered/smooth, near 0 = spatially random).

**How the 3 ENSO-state years are chosen**: the ONI (Oceanic Nino Index) CSV is read and converted to a monthly series (see Analysis 12 for the full method — the same conversion is repeated here so this analysis doesn't depend on cell-execution order). For each calendar year actually present in the ensemble record, the mean ONI across that year's months is computed, and the year with the **highest** annual-mean ONI is labeled the El Nino snapshot, the **lowest** the La Nina snapshot, and the one **closest to zero** the neutral snapshot. Each snapshot is the year-mean O2 field at 150 m for that calendar year (not a single month), to keep the spatial pattern reasonably stable and not dominated by one month's noise.

**A caveat the code checks for automatically**: this basin is only about 1400 km across. If the true spatial correlation length of the oxygen field is comparable to or larger than that, the empirical semivariogram will never reach a clear plateau within the domain, and the fitted "range" becomes an artifact (just the largest lag distance sampled) rather than a real measurement. The code checks for exactly this (a fitted range pinned at \u226590% of the maximum sampled lag) and, if it happens, says so explicitly rather than reporting a spurious "range changed by X%" comparison between ENSO states — global Moran's I (which doesn't depend on finding a plateau) is used as the more reliable comparison in that case.

**Reading the figure**: 4 rows (El Nino year, La Nina year, neutral year, full-record mean), 2 columns. Left: the snapshot's spatial O2 field (for context — what pattern is actually being analyzed). Right: the empirical semivariogram (dots) with the fitted spherical model (line); the fitted range and sill are annotated, along with the global Moran's I for that same snapshot. A short printed discussion after the code compares whether the range is noticeably larger or smaller in the anomalous (El Nino/La Nina) years vs. the neutral year — a shrinking range in an anomalous year would suggest the oxygen field becomes spatially more fragmented/patchy during ENSO extremes; a larger range would suggest it becomes more spatially coherent.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import skgstat as skg
import libpysal
from esda.moran import Moran

SEMIVARIOGRAM_REPRESENTATIVE_DEPTH_M = 150.0

# --- 1. Load + convert ONI to a monthly series (same method as Analysis 12) -
_oni_raw_11 = pd.read_csv(ONI_CSV_PATH)
_season_center_month_11 = {
    "DJF": 1, "JFM": 2, "FMA": 3, "MAM": 4, "AMJ": 5, "MJJ": 6,
    "JJA": 7, "JAS": 8, "ASO": 9, "SON": 10, "OND": 11, "NDJ": 12,
}
_oni_raw_11["center_month"] = _oni_raw_11["SEAS"].map(_season_center_month_11)
_oni_raw_11 = _oni_raw_11.dropna(subset=["center_month"])
_oni_raw_11["time"] = pd.to_datetime(dict(year=_oni_raw_11["YR"], month=_oni_raw_11["center_month"].astype(int), day=1))
oni_monthly_11 = _oni_raw_11.set_index("time")["ANOM"].sort_index()

# --- 2. Pick El Nino / La Nina / neutral YEARS from the ensemble's own record
_years_11 = sorted(pd.DatetimeIndex(ds_ensemble["time"].values).year.unique())
_annual_oni_11 = {y: oni_monthly_11[oni_monthly_11.index.year == y].mean() for y in _years_11}
_annual_oni_11 = {y: v for y, v in _annual_oni_11.items() if not np.isnan(v)}
_el_nino_year_11 = max(_annual_oni_11, key=_annual_oni_11.get)
_la_nina_year_11 = min(_annual_oni_11, key=_annual_oni_11.get)
_neutral_year_11 = min(_annual_oni_11, key=lambda y: abs(_annual_oni_11[y]))
print(f"El Nino snapshot year = {_el_nino_year_11} (mean ONI = {_annual_oni_11[_el_nino_year_11]:+.2f})")
print(f"La Nina snapshot year = {_la_nina_year_11} (mean ONI = {_annual_oni_11[_la_nina_year_11]:+.2f})")
print(f"Neutral snapshot year = {_neutral_year_11} (mean ONI = {_annual_oni_11[_neutral_year_11]:+.2f})")

# --- 3. Build the 4 snapshot fields (year-means + full-record mean) at 150 m
_da150_11 = ds_ensemble[O2_VAR].sel(depth=SEMIVARIOGRAM_REPRESENTATIVE_DEPTH_M)
snapshots_11 = {
    f"El Ni\u00f1o {_el_nino_year_11}": _da150_11.sel(time=_da150_11["time.year"] == _el_nino_year_11).mean("time"),
    f"La Ni\u00f1a {_la_nina_year_11}": _da150_11.sel(time=_da150_11["time.year"] == _la_nina_year_11).mean("time"),
    f"Neutral {_neutral_year_11}": _da150_11.sel(time=_da150_11["time.year"] == _neutral_year_11).mean("time"),
    "Full-record mean": _da150_11.mean("time"),
}

# --- 4. Local equirectangular projection to km, for physically-interpretable ranges
_R_KM_11 = 6371.0
_lat0_11 = np.deg2rad(float(np.mean(LAT_VALUES)))
_lon_grid_11, _lat_grid_11 = np.meshgrid(LON_VALUES, LAT_VALUES)
_x_km_11 = _R_KM_11 * np.cos(_lat0_11) * np.deg2rad(_lon_grid_11)
_y_km_11 = _R_KM_11 * np.deg2rad(_lat_grid_11)

variogram_results_11 = {}
for name, field in snapshots_11.items():
    vals = field.values
    valid = ~np.isnan(vals)
    coords = np.column_stack([_x_km_11[valid], _y_km_11[valid]])
    v = vals[valid]
    V = skg.Variogram(coords, v, model="spherical", n_lags=12, use_nugget=True)
    rng, sill, nugget = V.parameters
    _max_lag = float(V.bins.max())
    _range_saturated = rng >= 0.9 * _max_lag   # fitted range pinned at (near) the largest sampled lag

    w = libpysal.weights.KNN.from_array(coords, k=8)
    w.transform = "r"
    moran = Moran(v, w)

    variogram_results_11[name] = dict(field=field, V=V, range_km=rng, sill=sill, nugget=nugget,
                                        moran_I=moran.I, moran_p=moran.p_sim, range_saturated=_range_saturated,
                                        max_lag_km=_max_lag)
    _sat_note = "  ** SATURATED: semivariance never plateaus within the sampled domain -- 'range' " \
                "is NOT a reliable fitted value here, it is just the largest lag distance available " \
                "(the basin is smaller than the field's true correlation length) **" if _range_saturated else ""
    print(f"{name}: range={rng:.0f} km (max sampled lag={_max_lag:.0f} km), sill={sill:.2f}, "
          f"nugget={nugget:.2f}, Moran's I={moran.I:.3f} (p={moran.p_sim:.3f}){_sat_note}")

# --- 5. Discussion: does the range change between anomalous and neutral years?
_neutral_res_11 = variogram_results_11[f"Neutral {_neutral_year_11}"]
_any_saturated_11 = any(r["range_saturated"] for r in variogram_results_11.values())
if _any_saturated_11:
    print("\nCAVEAT: at least one snapshot's semivariogram is SATURATED (see ** notes above) -- "
          "the empirical semivariance keeps rising all the way to the largest lag distance sampled "
          "within this ~1400x1450 km domain, without reaching a clear plateau/sill. This means the "
          "field's true spatial correlation length is at least comparable to the basin's own size, "
          "and the fitted 'range' number is therefore NOT a reliable, resolvable estimate -- it is "
          "an artifact of the domain being too small, not a genuine measurement. Comparing 'range' "
          "percentage differences between El Nino/La Nina/neutral years would be comparing noise in "
          "an unresolved fit parameter, so that comparison is reported honestly as INCONCLUSIVE rather "
          "than asserting a spurious trend. The global Moran's I values above (which do not depend on "
          "fitting a plateau) remain valid and ARE comparable between snapshots -- all 4 snapshots show "
          "high (~0.9) Moran's I, meaning the O2 field is strongly spatially autocorrelated basin-wide "
          "in every ENSO state tested, consistent with the semivariogram's own saturation.")
else:
    for key in [f"El Ni\u00f1o {_el_nino_year_11}", f"La Ni\u00f1a {_la_nina_year_11}"]:
        r = variogram_results_11[key]["range_km"]
        pct = 100 * (r - _neutral_res_11["range_km"]) / _neutral_res_11["range_km"]
        direction = "LARGER (more spatially coherent)" if pct > 0 else "SMALLER (more spatially fragmented/patchy)"
        print(f"{key} range is {pct:+.0f}% vs. the neutral year -> {direction}.")

# --- 6. Composite figure -----------------------------------------------------
fig11, axes11 = plt.subplots(4, 2, figsize=(13.0, 13.0), constrained_layout=True)
for row, (name, res) in enumerate(variogram_results_11.items()):
    ax_map = axes11[row, 0]
    cs = ax_map.pcolormesh(LON_VALUES, LAT_VALUES, res["field"].values, cmap=CMAP_SEQUENTIAL, shading="nearest")
    ax_map.set_title(f"{name}: O$_2$ @ {SEMIVARIOGRAM_REPRESENTATIVE_DEPTH_M:.0f} m", loc="left",
                     fontsize=10, fontweight="bold")
    fig11.colorbar(cs, ax=ax_map, shrink=0.85, label="\u00b5mol/kg")

    ax_v = axes11[row, 1]
    V = res["V"]
    ax_v.scatter(V.bins, V.experimental, color="black", s=20, zorder=3)
    _xx = np.linspace(0, V.bins.max(), 100)
    ax_v.plot(_xx, V.fitted_model(_xx), color="firebrick", linewidth=1.5, zorder=2)
    ax_v.axhline(res["sill"], color="grey", linestyle=":", linewidth=1.0)
    ax_v.axvline(res["range_km"], color="grey", linestyle=":", linewidth=1.0)
    _rng_str = f"{res['range_km']:.0f}"
    _sill_str = f"{res['sill']:.1f}"
    _moran_str = f"{res['moran_I']:.2f}"
    ax_v.set_title(f"Semivariogram: range={_rng_str} km, sill={_sill_str}, Moran's I={_moran_str}",
                   loc="left", fontsize=9.5, fontweight="bold")
    ax_v.set_xlabel("Lag distance (km)")
    ax_v.set_ylabel("Semivariance")

fig11.suptitle("Bay of Bengal \u2014 Spatial Autocorrelation: Semivariogram + Global Moran's I",
               fontsize=13, fontweight="bold")

add_outer_frame(fig11)

savefig(fig11, "11_semivariogram_moran.png")
plt.show()


## 12. Composite / lag-correlation vs. climate indices (ONI + DMI only)

**Only two climate indices are used here: ONI (ENSO) and DMI (IOD). No Indian-monsoon rainfall index is included, and this is a deliberate, disclosed omission, not an oversight**: a machine-readable, operationally-current (through the ensemble's own 2005\u20132022 span) All-India monthly rainfall series could not be sourced from within this project — the IITM/IRIDL homogeneous series available only extends to Dec 2004, IMD's operational portal is an interactive, JavaScript-driven page rather than a downloadable file, and data.gov.in's catalog blocks automated fetching. See `external_climate_indices/README.md` for the full account of what was tried. If you obtain a suitable rainfall series yourself, it can be added to `ONI_CSV_PATH`/`DMI_CSV_PATH`-style handling in the config cell and folded into this analysis as a third index.

**What this shows**: (a) a monthly, deseasonalized O2 anomaly series at 150 m (basin + each region — the per-region numbers are reported in the printed output below the figure, to keep the figure itself readable); (b) lag cross-correlation (Pearson and Spearman) between the basin anomaly and each climate index at lags of -12 to +12 months, so you can see whether oxygen tends to lead or lag ENSO/IOD swings, and by how much; (c) composite anomaly maps for El Nino (ONI \u2265 +0.5), La Nina (ONI \u2264 -0.5), positive IOD (DMI \u2265 +0.4), and negative IOD (DMI \u2264 -0.4) months, each stippled where a one-sample t-test rejects "no anomaly" at p < 0.05.

**On the significance band in the cross-correlation plot**: monthly ocean anomaly series are themselves autocorrelated (an anomalous month tends to be followed by another similar one), which inflates the *apparent* significance of a correlation if you naively use the raw sample size N. The dashed significance band shown is instead computed from an **effective sample size** (Bretherton et al., 1999: `N_eff = N(1-\u03c1_x\u03c1_y)/(1+\u03c1_x\u03c1_y)`, using each series' own lag-1 autocorrelation \u03c1), which is smaller than N and gives an honest, appropriately-widened significance band rather than the naive, overconfident one.


In [ ]:
import numpy as np
import pandas as pd
import scipy.stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm

COMPOSITE_REPRESENTATIVE_DEPTH_M = 150.0
ONI_EL_NINO_THRESHOLD, ONI_LA_NINA_THRESHOLD = 0.5, -0.5
DMI_POS_IOD_THRESHOLD, DMI_NEG_IOD_THRESHOLD = 0.4, -0.4
LAG_RANGE_MONTHS = range(-12, 13)

# --- 1. Load + convert ONI to a monthly series (same convention as Analysis 11)
_oni_raw_12 = pd.read_csv(ONI_CSV_PATH)
_season_center_month_12 = {
    "DJF": 1, "JFM": 2, "FMA": 3, "MAM": 4, "AMJ": 5, "MJJ": 6,
    "JJA": 7, "JAS": 8, "ASO": 9, "SON": 10, "OND": 11, "NDJ": 12,
}
_oni_raw_12["center_month"] = _oni_raw_12["SEAS"].map(_season_center_month_12)
_oni_raw_12 = _oni_raw_12.dropna(subset=["center_month"])
_oni_raw_12["time"] = pd.to_datetime(dict(year=_oni_raw_12["YR"], month=_oni_raw_12["center_month"].astype(int), day=1))
oni_monthly_12 = _oni_raw_12.set_index("time")["ANOM"].sort_index()

# --- 2. Load + clean the DMI series (whitespace column names, -9999 sentinel)
_dmi_raw_12 = pd.read_csv(DMI_CSV_PATH)
_dmi_raw_12.columns = [c.strip() for c in _dmi_raw_12.columns]
_dmi_value_col_12 = [c for c in _dmi_raw_12.columns if c.lower() != "date"][0]
_dmi_raw_12[_dmi_value_col_12] = pd.to_numeric(_dmi_raw_12[_dmi_value_col_12], errors="coerce")
_dmi_raw_12.loc[_dmi_raw_12[_dmi_value_col_12] <= -999, _dmi_value_col_12] = np.nan
_dmi_raw_12["time"] = pd.to_datetime(_dmi_raw_12["Date"].astype(str).str.strip())
dmi_monthly_12 = _dmi_raw_12.set_index("time")[_dmi_value_col_12].sort_index()
print(f"ONI series: {oni_monthly_12.index.min():%Y-%m} to {oni_monthly_12.index.max():%Y-%m}, "
      f"{oni_monthly_12.isna().sum()} missing values.")
print(f"DMI series: {dmi_monthly_12.index.min():%Y-%m} to {dmi_monthly_12.index.max():%Y-%m}, "
      f"{dmi_monthly_12.isna().sum()} missing values.")

# --- 3. Deseasonalized O2 anomaly: full field (time,lat,lon) + region/basin series
_da150_12 = ds_ensemble[O2_VAR].sel(depth=COMPOSITE_REPRESENTATIVE_DEPTH_M)
_clim_12 = _da150_12.groupby("time.month").mean("time")
anomaly_field_12 = (_da150_12.groupby("time.month") - _clim_12)   # dims: time, lat, lon
anomaly_regional_12 = regional_mean(anomaly_field_12)              # dims: region, time
_time_12 = pd.DatetimeIndex(anomaly_regional_12["time"].values)

# Align climate indices to the ensemble's own monthly index (reindex, forward info only where present)
oni_aligned_12 = oni_monthly_12.reindex(_time_12)
dmi_aligned_12 = dmi_monthly_12.reindex(_time_12)
print(f"After aligning to the ensemble's {len(_time_12)}-month record: "
      f"{oni_aligned_12.isna().sum()} ONI month(s) and {dmi_aligned_12.isna().sum()} DMI month(s) missing.")

# --- 4. Lag cross-correlation (Pearson + Spearman), with effective-N significance
def _effective_n_12(x, y):
    def _lag1(s):
        return np.corrcoef(s[:-1], s[1:])[0, 1]
    rx, ry = _lag1(x), _lag1(y)
    n = len(x)
    n_eff = n * (1 - rx * ry) / (1 + rx * ry)
    return max(n_eff, 3.0)

def _critical_r_12(n_eff, alpha=SIGNIFICANCE_ALPHA):
    dof = max(n_eff - 2, 1.0)
    t_crit = scipy.stats.t.ppf(1 - alpha / 2, dof)
    return t_crit / np.sqrt(dof + t_crit ** 2)

def _lagged_corr_12(o2_series, index_series, lags):
    """Positive lag = O2 lags the index (index leads); index_series shifted
    forward by `lag` months and correlated against o2_series at each lag."""
    pear, spear = [], []
    valid_pair = pd.DataFrame({"o2": o2_series, "idx": index_series}).dropna()
    _n_eff = _effective_n_12(valid_pair["o2"].values, valid_pair["idx"].values)
    _r_crit = _critical_r_12(_n_eff)
    for lag in lags:
        shifted_idx = index_series.shift(lag)
        df = pd.DataFrame({"o2": o2_series, "idx": shifted_idx}).dropna()
        if len(df) < 10:
            pear.append(np.nan)
            spear.append(np.nan)
            continue
        pear.append(df["o2"].corr(df["idx"], method="pearson"))
        spear.append(df["o2"].corr(df["idx"], method="spearman"))
    return np.array(pear), np.array(spear), _r_crit, _n_eff

basin_anom_12 = pd.Series(anomaly_regional_12.sel(region="Basin").values, index=_time_12)
_lags_list_12 = list(LAG_RANGE_MONTHS)
pear_oni_12, spear_oni_12, rcrit_oni_12, neff_oni_12 = _lagged_corr_12(basin_anom_12, oni_aligned_12, _lags_list_12)
pear_dmi_12, spear_dmi_12, rcrit_dmi_12, neff_dmi_12 = _lagged_corr_12(basin_anom_12, dmi_aligned_12, _lags_list_12)
print(f"\nBasin vs ONI: effective N = {neff_oni_12:.0f} (raw N={len(_time_12)}), "
      f"critical |r| at p<{SIGNIFICANCE_ALPHA} = {rcrit_oni_12:.3f}")
print(f"Basin vs DMI: effective N = {neff_dmi_12:.0f} (raw N={len(_time_12)}), "
      f"critical |r| at p<{SIGNIFICANCE_ALPHA} = {rcrit_dmi_12:.3f}")

print("\nPer-region peak (|Pearson r| max over -12..+12 month lags) vs each index:")
for region_name in REGION_NAMES + ["Basin"]:
    series = pd.Series(anomaly_regional_12.sel(region=region_name).values, index=_time_12)
    for idx_name, idx_series in [("ONI", oni_aligned_12), ("DMI", dmi_aligned_12)]:
        p, s, rcrit, neff = _lagged_corr_12(series, idx_series, _lags_list_12)
        best_i = int(np.nanargmax(np.abs(p)))
        sig = "significant" if abs(p[best_i]) > rcrit else "not significant"
        print(f"  {region_name:9s} vs {idx_name}: peak r={p[best_i]:+.2f} at lag={_lags_list_12[best_i]:+d} "
              f"months ({sig}, crit={rcrit:.2f})")

# --- 5. Composite anomaly maps + one-sample t-test stippling ----------------
def _composite_12(mask_months):
    sel = anomaly_field_12.sel(time=anomaly_field_12["time"].isin(mask_months))
    comp_mean = sel.mean(dim="time")
    tstat, pval = scipy.stats.ttest_1samp(sel.values, popmean=0.0, axis=0, nan_policy="omit")
    return comp_mean.values, pval, sel.sizes["time"]

_composites_12 = {}
_composite_defs_12 = {
    "El Ni\u00f1o (ONI\u2265+0.5)": oni_aligned_12[oni_aligned_12 >= ONI_EL_NINO_THRESHOLD].index,
    "La Ni\u00f1a (ONI\u2264-0.5)": oni_aligned_12[oni_aligned_12 <= ONI_LA_NINA_THRESHOLD].index,
    "+IOD (DMI\u2265+0.4)": dmi_aligned_12[dmi_aligned_12 >= DMI_POS_IOD_THRESHOLD].index,
    "-IOD (DMI\u2264-0.4)": dmi_aligned_12[dmi_aligned_12 <= DMI_NEG_IOD_THRESHOLD].index,
}
for name, months in _composite_defs_12.items():
    comp_mean, pval, n_months = _composite_12(months)
    _composites_12[name] = dict(mean=comp_mean, pval=pval, n_months=n_months)
    print(f"{name}: {n_months} months composited.")

# --- 6. Composite figure: 2 cross-corr panels (top) + 2x2 composite maps (bottom)
fig12 = plt.figure(figsize=(13.0, 13.0), constrained_layout=True)
gs12 = fig12.add_gridspec(3, 2, height_ratios=[1, 1, 1])

ax_oni = fig12.add_subplot(gs12[0, 0])
ax_oni.plot(_lags_list_12, pear_oni_12, color="black", linewidth=1.4, label="Pearson")
ax_oni.plot(_lags_list_12, spear_oni_12, color="firebrick", linestyle="--", linewidth=1.2, label="Spearman")
ax_oni.axhspan(-rcrit_oni_12, rcrit_oni_12, color="grey", alpha=0.2, label=f"n.s. band (N$_{{eff}}$={neff_oni_12:.0f})")
ax_oni.axvline(0, color="grey", linewidth=0.6)
ax_oni.set_title("Basin O$_2$ anomaly vs. ONI: lag cross-correlation", loc="left", fontsize=10, fontweight="bold")
ax_oni.set_xlabel("Lag (months); positive = O$_2$ lags ONI")
ax_oni.set_ylabel("Correlation")

ax_dmi = fig12.add_subplot(gs12[0, 1])
ax_dmi.plot(_lags_list_12, pear_dmi_12, color="black", linewidth=1.4, label="Pearson")
ax_dmi.plot(_lags_list_12, spear_dmi_12, color="firebrick", linestyle="--", linewidth=1.2, label="Spearman")
ax_dmi.axhspan(-rcrit_dmi_12, rcrit_dmi_12, color="grey", alpha=0.2, label=f"n.s. band (N$_{{eff}}$={neff_dmi_12:.0f})")
ax_dmi.axvline(0, color="grey", linewidth=0.6)
ax_dmi.set_title("Basin O$_2$ anomaly vs. DMI: lag cross-correlation", loc="left", fontsize=10, fontweight="bold")
ax_dmi.set_xlabel("Lag (months); positive = O$_2$ lags DMI")
ax_dmi.set_ylabel("Correlation")
ax_dmi.legend(fontsize=8, loc="upper right")
ax_oni.legend(fontsize=8, loc="upper right")

_all_comp_vals_12 = np.concatenate([np.abs(c["mean"][~np.isnan(c["mean"])]) for c in _composites_12.values()])
_vmax_12 = float(np.nanpercentile(_all_comp_vals_12, 98)) or 1.0
_norm_12 = TwoSlopeNorm(vcenter=0.0, vmin=-_vmax_12, vmax=_vmax_12)
_lon_grid_12, _lat_grid_12 = np.meshgrid(LON_VALUES, LAT_VALUES)
_cs_12 = None
for k, (name, comp) in enumerate(_composites_12.items()):
    ax = fig12.add_subplot(gs12[1 + k // 2, k % 2])
    _cs_12 = ax.pcolormesh(LON_VALUES, LAT_VALUES, comp["mean"], cmap=CMAP_DIVERGING, norm=_norm_12, shading="nearest")
    _sig = comp["pval"] < SIGNIFICANCE_ALPHA
    ax.scatter(_lon_grid_12[_sig], _lat_grid_12[_sig], **STIPPLE_KW)
    _n_months_str = comp["n_months"]
    ax.set_title(f"{name}  (n={_n_months_str} months)", loc="left", fontsize=9.5, fontweight="bold")
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

fig12.colorbar(_cs_12, ax=[fig12.axes[2], fig12.axes[3], fig12.axes[4], fig12.axes[5]],
                shrink=0.85, pad=0.02, label="O$_2$ anomaly (\u00b5mol/kg)")

fig12.suptitle(f"Bay of Bengal \u2014 Lag Correlation \u0026 Composite Anomalies vs. ONI/DMI @ "
              f"{COMPOSITE_REPRESENTATIVE_DEPTH_M:.0f} m", fontsize=13, fontweight="bold")

add_outer_frame(fig12)

savefig(fig12, "12_composite_lag_correlation.png")
plt.show()


## 13. ML-based cross-product QC (G4D-DOC vs. GEOXYGEN)

**What this shows**: an automated, unsupervised check for data-quality anomalies by comparing the two raw source products (G4D-DOC and GEOXYGEN) that feed the ensemble, rather than relying on eyeballing plots. If the two independent products disagree unusually strongly at a particular place and time compared to their normal level of disagreement, that is a signal worth flagging — it might be a genuine short-lived event, but it might also be a processing artifact in one of the source products (this project has one documented example: a spurious G4D-DOC oxygenation "pulse" around **December 2019 - January 2020**).

**Method**: at every grid cell (depth, lat, lon), the code computes `G4D-DOC minus GEOXYGEN` for every month, then flags outliers in that cell's own difference time series two independent ways: (1) a **robust modified z-score** using the median and MAD (median absolute deviation) rather than the mean/standard deviation — robust statistics are used deliberately because the mean/std themselves get pulled around by the very outliers we're trying to detect; flagged at `|z| > 3.5` (the standard Iglewicz & Hoaglin threshold); and (2) an **Isolation Forest** (`sklearn.ensemble.IsolationForest`, random seed from the config cell) fit on all valid difference values, which detects outliers by how easily they can be "isolated" by random splits rather than by any distributional assumption. Using two independent methods lets the two cross-check each other rather than trusting a single method's threshold choice.

**The honesty check this analysis exists to run**: the code explicitly counts how many of each method's flagged month x location combinations fall inside the Dec 2019 - Jan 2020 window versus outside it, and reports the plain result — including reporting a miss explicitly, without adjusting thresholds afterward to manufacture a hit. This is a real test of whether these generic anomaly-detection methods, run without being told in advance where to look, actually rediscover a known problem.


In [ ]:
import numpy as np
import pandas as pd
import re
import warnings
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.ensemble import IsolationForest

KNOWN_ANOMALY_WINDOW_13 = (pd.Timestamp("2019-12-01"), pd.Timestamp("2020-01-31"))
ROBUST_Z_THRESHOLD = 3.5   # Iglewicz & Hoaglin (1993) modified z-score convention

def _load_gridded_product_13(directory, prefix):
    _fname_re = re.compile(prefix + r"_gridded_(\d{4})_(\d{2})\.nc$")
    def _preprocess(ds):
        m = _fname_re.search(ds.encoding.get("source", ""))
        if not m:
            raise ValueError(f"Could not parse year/month from {ds.encoding.get('source','')!r}")
        t = pd.Timestamp(year=int(m.group(1)), month=int(m.group(2)), day=1)
        return ds.expand_dims(time=[t])
    files = sorted(directory.glob(f"{prefix}_gridded_*.nc"))
    if not files:
        raise FileNotFoundError(f"No {prefix} gridded files found in {directory}")
    ds = xr.open_mfdataset(files, preprocess=_preprocess, combine="by_coords", engine="netcdf4", chunks=None)
    return ds.sortby("time").load()

# --- 1. Load both raw source products, restricted to their time overlap -----
if not GRIDDED_G4D_DOC_DIR.exists() or not GRIDDED_GEOXYGEN_DIR.exists():
    print("GRIDDED_G4D_DOC_DIR / GRIDDED_GEOXYGEN_DIR not found -- skipping Analysis 13. "
          "Check the config cell paths.")
else:
    ds_g4d_13 = _load_gridded_product_13(GRIDDED_G4D_DOC_DIR, "G4D_DOC")
    ds_geo_13 = _load_gridded_product_13(GRIDDED_GEOXYGEN_DIR, "GEOXYGEN")
    _common_time_13 = sorted(set(ds_g4d_13["time"].values) & set(ds_geo_13["time"].values))
    print(f"G4D-DOC: {ds_g4d_13.sizes['time']} months on disk. GEOXYGEN: {ds_geo_13.sizes['time']} months on disk. "
          f"Common months usable for the difference field: {len(_common_time_13)}.")
    ds_g4d_13 = ds_g4d_13.sel(time=_common_time_13)
    ds_geo_13 = ds_geo_13.sel(time=_common_time_13)

    # --- 2. G4D-DOC minus GEOXYGEN difference field (time, depth, lat, lon) -
    diff_field_13 = ds_g4d_13["o2_umol_kg"] - ds_geo_13["o2_umol_kg"]
    _diff_vals_13 = diff_field_13.transpose("time", "depth", "lat", "lon").values
    _nt13, _nd13, _nlat13, _nlon13 = _diff_vals_13.shape

    # --- 3. Robust modified z-score, per grid cell across time --------------
    # (always-land cells are all-NaN in every month -- nanmedian correctly returns
    # NaN for those and is expected/harmless; suppressed here since it is already
    # documented rather than left as an alarming-looking runtime warning)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        _median_13 = np.nanmedian(_diff_vals_13, axis=0, keepdims=True)
        _mad_13 = np.nanmedian(np.abs(_diff_vals_13 - _median_13), axis=0, keepdims=True)
    _mad_13_safe = np.where(_mad_13 < 1e-9, np.nan, _mad_13)   # avoid /0 at cells with a perfectly constant diff
    modified_z_13 = 0.6745 * (_diff_vals_13 - _median_13) / _mad_13_safe
    flagged_zscore_13 = np.abs(modified_z_13) > ROBUST_Z_THRESHOLD

    # --- 4. Isolation Forest, fit on all valid difference values -----------
    _valid_mask_13 = ~np.isnan(_diff_vals_13)
    _valid_vals_13 = _diff_vals_13[_valid_mask_13].reshape(-1, 1)
    iso_13 = IsolationForest(n_estimators=200, contamination=0.01, random_state=RANDOM_SEED)
    iso_pred_13 = iso_13.fit_predict(_valid_vals_13)   # -1 = anomaly, 1 = normal
    flagged_isoforest_13 = np.zeros(_diff_vals_13.shape, dtype=bool)
    flagged_isoforest_13[_valid_mask_13] = (iso_pred_13 == -1)

    _n_flag_z_13 = int(np.nansum(flagged_zscore_13))
    _n_flag_iso_13 = int(flagged_isoforest_13.sum())
    print(f"Robust z-score flagged {_n_flag_z_13} month\u00d7location\u00d7depth combinations "
          f"(|z|>{ROBUST_Z_THRESHOLD}).")
    print(f"Isolation Forest flagged {_n_flag_iso_13} month\u00d7location\u00d7depth combinations "
          "(contamination=0.01).")

    # --- 5. Explicit honesty check vs. the known Dec2019-Jan2020 anomaly ----
    _time_index_13 = pd.DatetimeIndex(diff_field_13["time"].values)
    _lo13, _hi13 = KNOWN_ANOMALY_WINDOW_13
    _in_window_13 = (_time_index_13 >= _lo13) & (_time_index_13 <= _hi13)
    print("\n" + "=" * 78)
    print(f"CROSS-CHECK vs. known G4D-DOC anomaly window ({_lo13:%Y-%m} to {_hi13:%Y-%m}):")
    for method_name, flagged in [("Robust z-score", flagged_zscore_13), ("Isolation Forest", flagged_isoforest_13)]:
        _n_in_window = int(flagged[_in_window_13].sum())
        _n_total = int(flagged.sum())
        _n_months_total = flagged.shape[0]
        _expected_share = _in_window_13.sum() / _n_months_total
        if _n_in_window > 0:
            _share = _n_in_window / max(_n_total, 1)
            verdict = ("FLAGGED: this window accounts for a disproportionate share of all flags "
                       f"({_share*100:.1f}% of flags, vs. {_expected_share*100:.1f}% of months) -- consistent "
                       "with correctly detecting a real, concentrated data artifact." if _share > _expected_share * 2
                       else "flagged some cells in this window, but not at a rate clearly above its share of "
                       "the record -- weak/ambiguous detection, not a clean catch.")
            print(f"  {method_name}: {_n_in_window} of {_n_total} total flags fall in the known-anomaly "
                  f"window. {verdict}")
        else:
            print(f"  {method_name}: 0 flags in the known-anomaly window. This is reported as a genuine "
                  "MISS -- this method, run generically without being told where to look, did not "
                  "rediscover the documented anomaly.")
    print("=" * 78)

    # --- 6. Figure ------------------------------------------------------------
    _basin_diff_ts_13 = np.nanmean(_diff_vals_13, axis=(1, 2, 3))
    _n_flagged_per_month_z_13 = np.nansum(flagged_zscore_13, axis=(1, 2, 3))
    _n_flagged_per_month_iso_13 = flagged_isoforest_13.sum(axis=(1, 2, 3))

    fig13, axes13 = plt.subplots(2, 2, figsize=(13.0, 9.0), constrained_layout=True)

    ax_a = axes13[0, 0]
    ax_a.plot(_time_index_13, _basin_diff_ts_13, color="black", linewidth=1.1)
    ax_a.axhline(0, color="grey", linewidth=0.5)
    ax_a.axvspan(_lo13, _hi13, color="orange", alpha=0.2)
    ax_a.set_title("(a) Basin-mean G4D-DOC \u2212 GEOXYGEN", loc="left", fontsize=10, fontweight="bold")
    ax_a.set_ylabel("\u00b5mol/kg")

    ax_b = axes13[0, 1]
    ax_b.plot(_time_index_13, _n_flagged_per_month_z_13, color="steelblue", linewidth=1.1, label="robust z-score")
    ax_b.plot(_time_index_13, _n_flagged_per_month_iso_13, color="firebrick", linewidth=1.1, label="Isolation Forest")
    ax_b.axvspan(_lo13, _hi13, color="orange", alpha=0.2)
    ax_b.set_title("(b) Flagged cells per month, both methods", loc="left", fontsize=10, fontweight="bold")
    ax_b.set_ylabel("# flagged cells")
    ax_b.legend(fontsize=8)

    ax_c = axes13[1, 0]
    _window_flag_density_z_13 = flagged_zscore_13[_in_window_13].sum(axis=(0, 1))   # sum over window-months and depth
    cs_c = ax_c.pcolormesh(LON_VALUES, LAT_VALUES, _window_flag_density_z_13, cmap=CMAP_SEQUENTIAL, shading="nearest")
    ax_c.set_title("(c) Robust z-score flag count within the known-anomaly window\n(summed over depth+months)",
                   loc="left", fontsize=9.5, fontweight="bold")
    ax_c.set_xlabel("Longitude"); ax_c.set_ylabel("Latitude")
    fig13.colorbar(cs_c, ax=ax_c, shrink=0.85)

    ax_d = axes13[1, 1]
    _window_flag_density_iso_13 = flagged_isoforest_13[_in_window_13].sum(axis=(0, 1))
    cs_d = ax_d.pcolormesh(LON_VALUES, LAT_VALUES, _window_flag_density_iso_13, cmap=CMAP_SEQUENTIAL, shading="nearest")
    ax_d.set_title("(d) Isolation Forest flag count within the known-anomaly window\n(summed over depth+months)",
                   loc="left", fontsize=9.5, fontweight="bold")
    ax_d.set_xlabel("Longitude"); ax_d.set_ylabel("Latitude")
    fig13.colorbar(cs_d, ax=ax_d, shrink=0.85)

    fig13.suptitle("Bay of Bengal \u2014 ML-Based Cross-Product QC (G4D-DOC vs. GEOXYGEN)",
                   fontsize=13, fontweight="bold")

    add_outer_frame(fig13)

    savefig(fig13, "13_ml_cross_product_qc.png")
    plt.show()


## 14. AOU-based attribution of the oxygen trend

**IMPORTANT -- read before trusting any numbers from this section**: this project's pipeline does not produce temperature or salinity, and a recursive search of the entire `C:\\All_data` tree (done before this notebook was written) found **no** temperature/salinity product of any kind. Apparent Oxygen Utilization (AOU) analysis fundamentally *requires* T/S (to compute the oxygen solubility a water parcel would have at saturation). Per the brief this notebook was built from, this is handled with a clearly-labeled **fallback**, not a silent guess:

- `TS_CLIMATOLOGY_PATH` (config cell) is a single, clearly-named variable pointing at a monthly climatological T/S NetCDF file (e.g. World Ocean Atlas 2023, regridded to roughly this project's domain) that **you** supply -- it does **not** ship with this notebook and is not auto-downloaded, because a live download could not be verified from within the environment this notebook was developed in.
- The file is expected to have a 12-step monthly/seasonal dimension (named `month` or `time`) plus `depth`, `lat`, `lon` coordinates (any reasonable grid -- this code interpolates onto the project's own grid) and two variables recognizable as in-situ temperature (degC) and practical salinity (PSU) -- common name variants are checked automatically and the ones actually found are printed.
- **If the file is not found, this section prints a clear explanation and stops -- it does NOT fabricate T/S values or skip silently.**

**Even when a climatological T/S file IS supplied, there is a fundamental, disclosed limitation**: a *climatology* repeats the same 12 monthly values every year, by construction. That means the oxygen-solubility term computed from it has **no long-term trend of its own** (its year-to-year variation is exactly zero for a given calendar month) -- so this fallback **cannot** detect a genuine warming-driven solubility decline; the code below computes and reports this explicitly (rather than hiding it), and the entire observed long-term O2 trend will mechanically show up as "AOU/utilization-driven" as a direct consequence of the climatology having no trend, not because utilization is actually 100% of the real-world story. A genuine warming-vs-utilization attribution would need time-varying (in-situ or reanalysis) T/S, not a climatology -- this section says so plainly rather than presenting a false-precision result.

**Method (only runs if the T/S file is found)**: `gsw.O2sol(SA, CT, p, lon, lat)` (TEOS-10, Garcia & Gordon 1992/1993 solubility, preferred over any hand-rolled formula) gives the saturation oxygen concentration from Absolute Salinity, Conservative Temperature, and pressure; `AOU = O2_sol - O2_observed`. Sen's slopes (reusing Analysis 7's method) are computed and compared for `O2_observed`, `O2_sol`, and `AOU`, at the basin and regional level.


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import gsw
import pymannkendall as mk

_TS_TEMP_NAME_CANDIDATES = ["temperature", "t_an", "THETA", "temp", "TEMP"]
_TS_SALT_NAME_CANDIDATES = ["salinity", "s_an", "SALT", "salt", "SALINITY"]

def _find_var_14(ds, candidates):
    for name in candidates:
        if name in ds.data_vars:
            return name
    return None

print("=" * 78)
print(f"Checking for a T/S climatology at: {TS_CLIMATOLOGY_PATH}")
if not TS_CLIMATOLOGY_PATH.exists():
    print("NOT FOUND. Analysis 14 (AOU-based attribution) is SKIPPED -- no fabricated T/S values")
    print("are used. To run this analysis: obtain a monthly climatological T/S product (e.g. World")
    print("Ocean Atlas 2023), and point TS_CLIMATOLOGY_PATH (in the configuration cell) at it. See")
    print("this section's markdown cell above for the expected file format, and the companion manual")
    print("for more detail. This is a disclosed data gap, not a computation error.")
    print("=" * 78)
    _analysis_14_ran = False
else:
    _ts_ds_14 = xr.open_dataset(TS_CLIMATOLOGY_PATH)
    _temp_var_14 = _find_var_14(_ts_ds_14, _TS_TEMP_NAME_CANDIDATES)
    _salt_var_14 = _find_var_14(_ts_ds_14, _TS_SALT_NAME_CANDIDATES)
    if _temp_var_14 is None or _salt_var_14 is None:
        print(f"FOUND the file, but could not recognize a temperature/salinity variable in it "
              f"(looked for {_TS_TEMP_NAME_CANDIDATES} / {_TS_SALT_NAME_CANDIDATES}, "
              f"found variables: {list(_ts_ds_14.data_vars)}). Analysis 14 is SKIPPED.")
        print("=" * 78)
        _analysis_14_ran = False
    else:
        print(f"FOUND. Using '{_temp_var_14}' as temperature (degC) and '{_salt_var_14}' as salinity (PSU).")
        _seasonal_dim_14 = "month" if "month" in _ts_ds_14.dims else ("time" if "time" in _ts_ds_14.dims else None)
        if _seasonal_dim_14 is None:
            print("Could not find a 'month' or 'time' (12-step seasonal) dimension in the T/S file. "
                  "Analysis 14 is SKIPPED.")
            print("=" * 78)
            _analysis_14_ran = False
        else:
            _analysis_14_ran = True

if _analysis_14_ran:
    AOU_REPRESENTATIVE_DEPTH_M = 150.0

    # --- 1. Interpolate the climatology onto this project's own grid --------
    _ts_regridded_14 = _ts_ds_14[[_temp_var_14, _salt_var_14]].rename(
        {_seasonal_dim_14: "month", _temp_var_14: "temperature", _salt_var_14: "salinity"}
    )
    if np.issubdtype(_ts_regridded_14["month"].dtype, np.datetime64):
        _ts_regridded_14 = _ts_regridded_14.assign_coords(month=_ts_regridded_14["month"].dt.month)
    _ts_regridded_14 = _ts_regridded_14.interp(depth=DEPTH_LEVELS_M, lat=LAT_VALUES, lon=LON_VALUES)

    # --- 2. gsw/TEOS-10 solubility (Garcia & Gordon 1992/1993) per calendar month
    _lon3d_14, _lat3d_14 = np.meshgrid(LON_VALUES, LAT_VALUES)
    _depth3d_14 = DEPTH_LEVELS_M[:, None, None] * np.ones((1, len(LAT_VALUES), len(LON_VALUES)))
    _p_14 = gsw.p_from_z(-_depth3d_14, _lat3d_14[None, :, :])   # sea pressure, dbar

    _o2sol_by_month_14 = {}
    for m in range(1, 13):
        t_m = _ts_regridded_14["temperature"].sel(month=m).values
        s_m = _ts_regridded_14["salinity"].sel(month=m).values
        sa_m = gsw.SA_from_SP(s_m, _p_14, _lon3d_14[None, :, :], _lat3d_14[None, :, :])
        ct_m = gsw.CT_from_t(sa_m, t_m, _p_14)
        _o2sol_by_month_14[m] = gsw.O2sol(sa_m, ct_m, _p_14, _lon3d_14[None, :, :], _lat3d_14[None, :, :])

    # --- 3. Broadcast the (fixed, non-trending) monthly solubility field across
    # every year of the record, and compute AOU = O2_sol - O2_observed --------
    _o2sol_stack_14 = np.stack([_o2sol_by_month_14[m] for m in range(1, 13)], axis=0)   # (12, depth, lat, lon)
    o2sol_da_14 = xr.DataArray(_o2sol_stack_14, dims=("month", "depth", "lat", "lon"),
                                 coords={"month": np.arange(1, 13), "depth": ds_ensemble["depth"],
                                         "lat": ds_ensemble["lat"], "lon": ds_ensemble["lon"]})
    # Vectorized .sel() with a "time"-dimensioned indexer replaces the "month"
    # dimension with "time" automatically (each month of the fixed climatology
    # is looked up once per calendar month across all 18 years) -- this is the
    # step that makes the solubility field have NO trend of its own (see markdown).
    o2sol_timeseries_14 = o2sol_da_14.sel(month=ds_ensemble["time"].dt.month).drop_vars("month")

    # --- 4. Trend comparison: observed O2, solubility O2_sol, and AOU, at the
    # representative depth, basin + regions, reusing Analysis 7's MK method ---
    def _trend_14(vals):
        # A fixed climatology (e.g. the solubility series) can repeat EXACTLY
        # every year, giving zero inter-annual variance and triggering a
        # harmless divide-by-zero warning inside the trend test -- expected
        # and handled (p becomes NaN), so it is suppressed here rather than
        # printed as if it were a real problem.
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                r = mk.correlated_seasonal_test(np.asarray(vals, dtype=float), period=12)
            return float(r.slope), float(r.p)
        except Exception:
            return np.nan, np.nan

    o2_obs_regional_14 = regional_mean(ds_ensemble[O2_VAR].sel(depth=AOU_REPRESENTATIVE_DEPTH_M))
    o2sol_regional_14 = regional_mean(o2sol_timeseries_14.sel(depth=AOU_REPRESENTATIVE_DEPTH_M))
    aou_field_14 = o2sol_timeseries_14.sel(depth=AOU_REPRESENTATIVE_DEPTH_M) - ds_ensemble[O2_VAR].sel(depth=AOU_REPRESENTATIVE_DEPTH_M)
    aou_regional_14 = regional_mean(aou_field_14)

    print(f"\nTrend comparison @ {AOU_REPRESENTATIVE_DEPTH_M:.0f} m (umol/kg/year), basin + regions:")
    _trend_rows_14 = []
    for region_name in REGION_NAMES + ["Basin"]:
        s_obs, p_obs = _trend_14(o2_obs_regional_14.sel(region=region_name).values)
        s_sol, p_sol = _trend_14(o2sol_regional_14.sel(region=region_name).values)
        s_aou, p_aou = _trend_14(aou_regional_14.sel(region=region_name).values)
        _trend_rows_14.append((region_name, s_obs, s_sol, s_aou))
        print(f"  {region_name:9s}: O2_observed={s_obs:+.3f} (p={p_obs:.3f})   "
              f"O2_solubility={s_sol:+.4f} (p={p_sol:.3f}, expect ~0 by construction)   "
              f"AOU={s_aou:+.3f} (p={p_aou:.3f})")
    print("\nAs expected from using a fixed CLIMATOLOGY, the O2_solubility trend above is ~0 for "
          "every region -- confirming empirically (not just asserting) that this fallback mechanically "
          "assigns essentially the entire observed trend to the AOU/utilization term. This is a limitation "
          "of the T/S data available, not a finding about the real ocean's warming-vs-utilization balance.")

    # --- 5. Figure: basin-mean O2_observed vs. O2_solubility vs. AOU time series
    _time_14 = pd.DatetimeIndex(o2_obs_regional_14["time"].values)
    fig14, axes14 = plt.subplots(3, 1, figsize=(12, 10), sharex=True, constrained_layout=True)
    _panels_14 = [
        ("Observed O$_2$", o2_obs_regional_14, "black"),
        ("Solubility O$_2$ (climatological T/S)", o2sol_regional_14, "steelblue"),
        ("AOU (solubility \u2212 observed)", aou_regional_14, "firebrick"),
    ]
    for ax, (label, da, color) in zip(axes14, _panels_14):
        for region_name in REGION_NAMES + ["Basin"]:
            ax.plot(_time_14, da.sel(region=region_name).values, linewidth=1.0, alpha=0.85, label=region_name)
        ax.set_title(label, loc="left", fontsize=10.5, fontweight="bold")
        ax.set_ylabel("\u00b5mol/kg")
    axes14[0].legend(fontsize=8, ncol=4)
    axes14[-1].set_xlabel("Time")

    fig14.suptitle(f"Bay of Bengal \u2014 AOU-Based Attribution @ {AOU_REPRESENTATIVE_DEPTH_M:.0f} m "
                  "(climatological T/S fallback \u2014 see markdown caveat)", fontsize=12.5, fontweight="bold")

    add_outer_frame(fig14)

    savefig(fig14, "14_aou_attribution.png")
    plt.show()


## Closing summary

This notebook ran 14 analyses on the same shared, once-loaded ensemble dataset (`ds_ensemble`) and the shared region/config machinery from the Setup section. A few notes on how to read the results together, now that everything has run:

**Multiple independent lines of evidence for deoxygenation.** Analyses 1, 2, 7, 9, and 10 each look at long-term change from a different angle -- moving averages, full-depth Hovmoller structure, formal trend significance testing, threshold-based OMZ geometry, and oxycline depth. If these agree with each other (e.g. Analysis 7's Sen's slope maps show significant negative trends at the same depths/regions where Analysis 9 shows OMZ volume/thickness increasing and Analysis 10 shows the oxycline shoaling), that agreement across independent methods is much stronger evidence than any single result in isolation. Where they *don't* agree, that disagreement is worth investigating rather than picking whichever result tells the preferred story.

**Two analyses exist specifically to test honesty, not to produce a headline number.** Analysis 8 (change-point detection) and Analysis 13 (ML cross-product QC) were both run against the documented Dec 2019 - Jan 2020 G4D-DOC data-quality anomaly as an explicit pass/fail check -- re-read their printed output above for the actual verdict on your run (a real oceanographic finding should not depend on results that turn out to trace back to that anomaly, so knowing whether it was caught matters for trusting everything else). Do not treat "the algorithm didn't flag it" as a failure of the notebook -- an honest miss, reported plainly, is the entire point of running the check.

**Analysis 14's numbers carry a structural caveat.** Unless you have supplied your own time-varying (in-situ or reanalysis) temperature/salinity product at `TS_CLIMATOLOGY_PATH`, the solubility-vs-utilization attribution mechanically assigns essentially all of the observed trend to "utilization" -- not because that is necessarily true of the real ocean, but because a fixed climatology cannot show a warming-driven solubility trend by construction. Treat Analysis 14 as a demonstration of the *method* until real T/S data is available, not as a final attribution result.

**Comparing this notebook to your earlier `oxygen_zone_volumes` output.** The OMZ thresholds used throughout this notebook (`OMZ_ZONE_THRESHOLDS` in the config cell) are literature defaults, not the same numbers used in the companion `Bay_of_Bengal_Yearly_Maps.ipynb` notebook's zone-volume table -- the two are not directly comparable cell-for-cell. If you want a consistent basin-wide picture across both notebooks, pick one threshold convention and apply it in both places.

**What to change and re-run.** Everything tunable lives in the single configuration cell near the top of this notebook (Section 0.0) -- depths, region banding/weighting method, the moving-average window, OMZ thresholds, the random seed, color conventions, and every file path. Changing a value there and re-running from that cell down will propagate consistently through all 14 analyses, since every analysis reads from those same named variables rather than hardcoding its own copies.


---

## Part D — Driver Attribution (CatBoost / LightGBM / XGBoost + TreeSHAP, Random Forest + Permutation Importance, GAM)

Identifies the dominant drivers of dissolved oxygen using the Argo-derived driver dataset
(`BoB_Argo_Driver_Dataset_v2.parquet`, 2012-2022, 9 drivers, 324,408 rows). This parquet file is
not included in this repository (see `docs/driver_dataset_notes.md` for how it was built and
`data/README.md` for the data-availability note) — the derived importance tables and figures it
produced are saved under `results/`.

**Drivers (9):** depth_m, temp_degC, psal_psu, chla_mg_m3, bbp700_m-1, mld_m, n2_per_s2, oni, dmi.
**Target:** o2_target_umol_kg (ensemble G4D-DOC+GEOXYGEN blend, matched to each Argo point's grid
cell/depth/month). DOXY, NITRATE, and AOU are deliberately excluded as drivers — see
`docs/driver_dataset_notes.md`.

Train/test split is by Argo float (platform), not by row, to avoid leakage from
spatially/temporally autocorrelated profiles: 11 platforms / 255,912 rows train, 4 platforms /
64,049 rows (20%) held out for test.

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr

from catboost import CatBoostRegressor
import lightgbm as lgb
import xgboost as xgb
import shap
from pygam import LinearGAM, s

pd.set_option('display.width', 120)
RANDOM_SEED = 42


## 1. Load data and define train/test split (by Argo float)

In [ ]:
DRIVERS = ['depth_m','temp_degC','psal_psu','chla_mg_m3','bbp700_m-1','mld_m','n2_per_s2','oni','dmi']
TARGET = 'o2_target_umol_kg'

df = pd.read_parquet('BoB_Argo_Driver_Dataset_v2.parquet')
print(f"Loaded: {df.shape}")

clean = df.dropna(subset=DRIVERS + [TARGET]).reset_index(drop=True)
print(f"Clean modeling table (no NaN in any of 9 drivers or target): {len(clean)} / {len(df)} rows")
print(f"Platforms: {clean['platform_number'].nunique()}")


In [ ]:
# Platform-based (spatial) holdout: ~20% of rows, held out by whole float, so no
# spatially/temporally autocorrelated profile leaks between train and test.
TEST_PLATFORMS = [2902086, 2902113, 2902161, 2902189]

clean['split'] = np.where(clean['platform_number'].isin(TEST_PLATFORMS), 'test', 'train')
train = clean[clean['split']=='train']
test = clean[clean['split']=='test']
print(f"Train: {len(train)} rows, {train['platform_number'].nunique()} platforms")
print(f"Test:  {len(test)} rows ({100*len(test)/len(clean):.1f}%), {test['platform_number'].nunique()} platforms")

Xtr, ytr = train[DRIVERS], train[TARGET]
Xte, yte = test[DRIVERS], test[TARGET]


## 2. Model 1 — CatBoost (+ LightGBM, XGBoost cross-check) with TreeSHAP

CatBoost is the primary gradient-booster (native missing-value handling, ordered boosting reduces
overfitting on gappy real-world BGC-Argo data). LightGBM and XGBoost are fit alongside as a
within-family robustness cross-check, all using **gain-based** feature importance for a fair
comparison (LightGBM's default 'split'-count importance is a weaker, biased metric and is not used here).


In [ ]:
cb = CatBoostRegressor(iterations=800, depth=8, learning_rate=0.05, loss_function='RMSE',
                        random_seed=RANDOM_SEED, verbose=False)
cb.fit(Xtr, ytr)
pred_cb = cb.predict(Xte)
r2_cb, mae_cb = r2_score(yte, pred_cb), mean_absolute_error(yte, pred_cb)
print(f"CatBoost  Test R2={r2_cb:.4f}  MAE={mae_cb:.3f} umol/kg")

cb_native = pd.Series(cb.get_feature_importance(), index=DRIVERS).sort_values(ascending=False)
print("\nCatBoost native (PredictionValuesChange) importance:"); print(cb_native)


In [ ]:
# TreeSHAP - computed on a random sample of the test set for speed
explainer = shap.TreeExplainer(cb)
rng = np.random.RandomState(RANDOM_SEED)
sample_idx = rng.choice(len(Xte), size=min(5000, len(Xte)), replace=False)
Xte_sample = Xte.iloc[sample_idx]
shap_values = explainer.shap_values(Xte_sample)
cb_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=DRIVERS).sort_values(ascending=False)
print("CatBoost TreeSHAP mean(|SHAP value|) importance:"); print(cb_shap)

shap.summary_plot(shap_values, Xte_sample, feature_names=DRIVERS, show=False)
plt.tight_layout(); plt.savefig('shap_summary_catboost.png', dpi=200, bbox_inches='tight'); plt.show()


In [ ]:
lgbm = lgb.LGBMRegressor(n_estimators=800, max_depth=8, learning_rate=0.05,
                          random_state=RANDOM_SEED, verbosity=-1, importance_type='gain')
lgbm.fit(Xtr, ytr)
pred_lgb = lgbm.predict(Xte)
r2_lgb, mae_lgb = r2_score(yte, pred_lgb), mean_absolute_error(yte, pred_lgb)
print(f"LightGBM  Test R2={r2_lgb:.4f}  MAE={mae_lgb:.3f}")
lgb_gain = pd.Series(lgbm.feature_importances_, index=DRIVERS).sort_values(ascending=False)
print(lgb_gain)


In [ ]:
xgbm = xgb.XGBRegressor(n_estimators=800, max_depth=8, learning_rate=0.05,
                         random_state=RANDOM_SEED, verbosity=0)
xgbm.fit(Xtr, ytr)
pred_xgb = xgbm.predict(Xte)
r2_xgb, mae_xgb = r2_score(yte, pred_xgb), mean_absolute_error(yte, pred_xgb)
print(f"XGBoost  Test R2={r2_xgb:.4f}  MAE={mae_xgb:.3f}")
xgb_gain = pd.Series(xgbm.feature_importances_, index=DRIVERS).sort_values(ascending=False)
print(xgb_gain)


## 3. Model 2 — Random Forest + Permutation Importance

Permutation importance is computed on the **held-out test set** (already blocked by float, so no
extra leakage concern from correlated nearby points).


In [ ]:
rf = RandomForestRegressor(n_estimators=300, max_depth=16, min_samples_leaf=3,
                            n_jobs=-1, random_state=RANDOM_SEED)
rf.fit(Xtr, ytr)
pred_rf = rf.predict(Xte)
r2_rf, mae_rf = r2_score(yte, pred_rf), mean_absolute_error(yte, pred_rf)
print(f"Random Forest  Test R2={r2_rf:.4f}  MAE={mae_rf:.3f}")

rf_native = pd.Series(rf.feature_importances_, index=DRIVERS).sort_values(ascending=False)
print("\nRF native (impurity) importance:"); print(rf_native)


In [ ]:
rng = np.random.RandomState(RANDOM_SEED)
sub_idx = rng.choice(len(Xte), size=min(20000, len(Xte)), replace=False)
Xte_sub, yte_sub = Xte.iloc[sub_idx], yte.iloc[sub_idx]

perm = permutation_importance(rf, Xte_sub, yte_sub, n_repeats=10, random_state=RANDOM_SEED, n_jobs=-1)
rf_perm = pd.Series(perm.importances_mean, index=DRIVERS).sort_values(ascending=False)
print("RF permutation importance (mean decrease in R2 when shuffled):"); print(rf_perm)


## 4. Model 3 — GAM

A `LinearGAM` with one smooth term per driver. Fit on a 40,000-row subsample of the training set
(pyGAM scales poorly to 250k+ rows with 9 smooth terms). Importance is measured as the drop in
explained deviance when each term is removed and the model refit (leave-one-term-out), since GAM
has no single native "importance" concept the way tree models do.


In [ ]:
N_TRAIN_SUB = 40000
train_sub = train.sample(n=min(N_TRAIN_SUB, len(train)), random_state=RANDOM_SEED)
Xtr_g, ytr_g = train_sub[DRIVERS].values, train_sub[TARGET].values
Xte_g, yte_g = test[DRIVERS].values, test[TARGET].values

terms = s(0)
for i in range(1, len(DRIVERS)):
    terms += s(i)

gam = LinearGAM(terms, n_splines=15)
gam.gridsearch(Xtr_g, ytr_g, lam=np.logspace(-2, 2, 5), progress=False)

pred_gam = gam.predict(Xte_g)
r2_gam, mae_gam = r2_score(yte_g, pred_gam), mean_absolute_error(yte_g, pred_gam)
full_deviance = gam.statistics_['pseudo_r2']['explained_deviance']
print(f"GAM  Test R2={r2_gam:.4f}  MAE={mae_gam:.3f}  Full explained deviance={full_deviance:.4f}")


In [ ]:
gam_importance = {}
n_reduced = len(DRIVERS) - 1
for i, feat in enumerate(DRIVERS):
    tt = s(0)
    for k in range(1, n_reduced):
        tt += s(k)
    cols = [j for j in range(len(DRIVERS)) if j != i]
    gam_r = LinearGAM(tt, n_splines=15)
    gam_r.gridsearch(Xtr_g[:, cols], ytr_g, lam=np.logspace(-2, 2, 5), progress=False)
    reduced_dev = gam_r.statistics_['pseudo_r2']['explained_deviance']
    gam_importance[feat] = full_deviance - reduced_dev

gam_imp = pd.Series(gam_importance).sort_values(ascending=False)
print("GAM importance (explained-deviance drop when term removed):"); print(gam_imp)


## 5. Cross-model comparison and consensus ranking

Each method's importances are normalized to sum to 1, then averaged into a consensus rank across
all 7 importance measures (CatBoost TreeSHAP, CatBoost native, LightGBM gain, XGBoost gain,
RF impurity, RF permutation, GAM deviance-drop). A second ranking excludes `depth_m`, since depth
structurally dominates via the OMZ's vertical oxygen profile and can mask the relative ordering of
the remaining physical/biogeochemical/climate drivers.


In [ ]:
methods = {
    'CatBoost_TreeSHAP': cb_shap, 'CatBoost_native': cb_native,
    'LightGBM_gain': lgb_gain, 'XGBoost_gain': xgb_gain,
    'RF_impurity': rf_native, 'RF_permutation': rf_perm,
    'GAM_deviance_drop': gam_imp,
}
table = pd.DataFrame({m: v.reindex(DRIVERS) for m, v in methods.items()})
table_norm = table.clip(lower=0)
table_norm = table_norm / table_norm.sum(axis=0)
print("Normalized importance by method:"); print(table_norm.round(4))


In [ ]:
corr = pd.DataFrame(index=methods.keys(), columns=methods.keys(), dtype=float)
for m1 in methods:
    for m2 in methods:
        corr.loc[m1, m2] = spearmanr(table_norm[m1], table_norm[m2])[0]
print("Spearman rank correlation across the 7 importance measures:"); print(corr.round(3))


In [ ]:
mean_norm_importance = table_norm.mean(axis=1).sort_values(ascending=False)
print("=== CONSENSUS MEAN NORMALIZED IMPORTANCE (all 9 drivers) ===")
print(mean_norm_importance)

drivers_no_depth = [d for d in DRIVERS if d != 'depth_m']
table_nodepth = table.loc[drivers_no_depth].clip(lower=0)
table_nodepth_norm = table_nodepth / table_nodepth.sum(axis=0)
mean_nodepth = table_nodepth_norm.mean(axis=1).sort_values(ascending=False)
print("\n=== CONSENSUS RANKING EXCLUDING depth_m ===")
print(mean_nodepth)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
table_norm.loc[DRIVERS].plot(kind='bar', ax=axes[0], width=0.85)
axes[0].set_title('Normalized feature importance by method (all 9 drivers)')
axes[0].set_ylabel('Normalized importance (sums to 1 per method)')
axes[0].legend(fontsize=8, ncol=2)
axes[0].tick_params(axis='x', rotation=45)

table_nodepth_norm.loc[drivers_no_depth].plot(kind='bar', ax=axes[1], width=0.85)
axes[1].set_title('Normalized importance excluding depth_m')
axes[1].set_ylabel('Normalized importance (sums to 1 per method)')
axes[1].legend(fontsize=8, ncol=2)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('driver_importance_comparison.png', dpi=200, bbox_inches='tight')
plt.show()


## 6. Results summary

| Rank | Driver | Consensus normalized importance |
|---|---|---|
| 1 | depth_m | ~0.79 |
| 2 | temp_degC | ~0.12 |
| 3 | psal_psu | ~0.03 |
| 4 | oni | ~0.02 |
| 5 | dmi | ~0.02 |
| 6 | bbp700_m-1 | ~0.01 |
| 7 | chla_mg_m3 | ~0.01 |
| 8 | mld_m / n2_per_s2 | ~0.003 each |

All three model families agree closely (Spearman rank correlation 0.78-0.98 across every pair of
importance measures): **depth is overwhelmingly the dominant structural driver** (expected - it sets
the OMZ's vertical position), **temperature is a clear, consistent second** (tracks the thermocline,
itself tied to depth), and **salinity, ENSO (ONI), and IOD (DMI)** form a modest third tier. The
BGC-optical variables (chlorophyll, backscatter) and the stratification diagnostics (MLD, N2) rank
lowest and contribute little independent predictive power in this dataset - consistent with their
sparse Argo sensor coverage limiting statistical power, not necessarily their true oceanographic
irrelevance. All models achieve high test R2 (0.955-0.987) on floats held out entirely from training,
so the relationships are genuinely predictive, not overfit to the training floats.


---

## Provenance note

Two pieces of derived work referenced in `results/` were produced by scripts that predate this
repository and were not preserved as standalone files, so their code isn't reproduced above —
only their outputs and documentation are carried here:

- **The Argo driver-dataset build** (raw BGC-Argo profiles → `BoB_Argo_Driver_Dataset_v2.parquet`),
  documented in `docs/driver_dataset_notes.md`.
- **The Gaussian Process OMZ-core projection** to 2030/2040/2050, documented in
  `docs/OMZ_Core_O2_Prediction_Report.md`, including the exact kernel specification used.

Everything else in this repository — the grid/ensemble pipeline, the 14-analysis study, the
yearly-map and zone-volume outputs, and the driver-attribution modelling above — is reproduced in
full in this notebook.